In [ ]:
#processed.csv generation
#SNR calculation
"""
PPG Signal Quality Assessment & Enhancement Pipeline (統合版)
既存の優れた前処理パイプラインにSNR計算機能を統合

【統合された手法の文献根拠】
前処理:
- Liang et al. (2018): Chebyshev Type II 4次フィルタが最適
  https://doi.org/10.1038/sdata.2018.76
- Allen & Murray (2004): 0.5-12 Hz推奨
  Physiological Measurement, 25(1), R1
- Park et al. (2022): PPG前処理の包括的レビュー
- Wang et al. (2007): Cubic spline baseline correction

品質評価:
- Reddy et al. (2022): 周波数ドメインSNR計算
  IEEE Access, 10, 20111-20127
- Mohagheghian et al. (2022): 信号品質指標の最適化
  IEEE Transactions on Biomedical Engineering, 69(10), 3982-3991
- Welch (1967): パワースペクトル推定の標準手法
"""

import pandas as pd
import numpy as np
from scipy import signal
from scipy.stats import skew, kurtosis
from scipy.fft import fft, fftfreq
from scipy.interpolate import UnivariateSpline
from scipy.ndimage import median_filter, minimum_filter
import logging
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ログ設定
log_filename = f"ppg_quality_assessment_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename, encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


class PPGQualityAssessor:
    """
    PPG信号品質評価クラス
    
    文献根拠:
    - Welch法によるパワースペクトル推定 (Welch, 1967)
    - 周波数ドメインSNR (Reddy et al., 2022)
    - Perfusion Index計算 (標準的な臨床指標)
    """
    
    def __init__(self, fs=2000.0):
        self.fs = fs
        
    def calculate_snr_welch(self, signal_data, signal_band=(0.5, 4.0), noise_band=(4.0, 15.0)):
        """
        Welch法によるSNR計算
        
        文献根拠:
        - Welch (1967): 安定したパワースペクトル推定
        - Reddy et al. (2022): 周波数ドメインSNR評価
        
        Parameters:
        -----------
        signal_data : array-like
            入力信号
        signal_band : tuple
            信号帯域 (Hz) - 心拍成分
        noise_band : tuple
            ノイズ帯域 (Hz)
            
        Returns:
        --------
        snr_db : float
            SNR [dB]
        signal_power : float
            信号パワー
        noise_power : float
            ノイズパワー
        """
        try:
            # Welch法でパワースペクトル密度を推定
            nperseg = min(len(signal_data), 2048)
            freqs, psd = signal.welch(
                signal_data, 
                fs=self.fs, 
                nperseg=nperseg,
                noverlap=nperseg//2,
                scaling='density'
            )
            
            # 信号帯域のパワー
            signal_mask = (freqs >= signal_band[0]) & (freqs <= signal_band[1])
            signal_power = np.trapz(psd[signal_mask], freqs[signal_mask])
            
            # ノイズ帯域のパワー
            noise_mask = (freqs >= noise_band[0]) & (freqs <= noise_band[1])
            noise_power = np.trapz(psd[noise_mask], freqs[noise_mask])
            
            # SNR計算
            if noise_power > 1e-10:
                snr_db = 10 * np.log10(signal_power / noise_power)
            else:
                snr_db = np.inf
                logger.warning("Noise power near zero, SNR set to infinity")
            
            return snr_db, signal_power, noise_power
            
        except Exception as e:
            logger.error(f"Error in calculate_snr_welch: {str(e)}")
            return np.nan, np.nan, np.nan
    
    def calculate_snr_fft(self, signal_data, signal_band=(0.5, 4.0), noise_band=(4.0, 15.0)):
        """
        FFTベースのSNR計算（高速版）
        
        Parameters:
        -----------
        signal_data : array-like
            入力信号
        signal_band : tuple
            信号帯域 (Hz)
        noise_band : tuple
            ノイズ帯域 (Hz)
            
        Returns:
        --------
        snr_db : float
            SNR [dB]
        """
        try:
            n = len(signal_data)
            sig_detrend = signal_data - np.mean(signal_data)
            
            # FFT
            yf = fft(sig_detrend)
            xf = fftfreq(n, 1/self.fs)
            
            # パワースペクトル
            psd = np.abs(yf)**2 / n
            
            # 信号帯域
            sig_mask = (xf >= signal_band[0]) & (xf <= signal_band[1])
            p_signal = np.sum(psd[sig_mask])
            
            # ノイズ帯域
            noise_mask = (xf >= noise_band[0]) & (xf <= noise_band[1])
            p_noise = np.sum(psd[noise_mask])
            
            if p_noise > 0:
                snr_db = 10 * np.log10(p_signal / p_noise)
            else:
                snr_db = np.inf
            
            return snr_db
            
        except Exception as e:
            logger.error(f"Error in calculate_snr_fft: {str(e)}")
            return np.nan
    
    def calculate_perfusion_index(self, signal_data):
        """
        Perfusion Index (PI) 計算
        
        PI = (AC / DC) × 100 [%]
        
        文献根拠: 臨床的PPG品質評価の標準指標
        
        Parameters:
        -----------
        signal_data : array-like
            入力信号
            
        Returns:
        --------
        pi_percent : float
            Perfusion Index [%]
        dc_component : float
            DC成分
        ac_component : float
            AC成分 (RMS)
        """
        try:
            # DC成分（平均値）
            dc_component = np.mean(signal_data)
            
            # AC成分（標準偏差 or RMS）
            ac_component = np.std(signal_data)
            
            # PI計算
            if abs(dc_component) > 1e-6:
                pi_percent = (ac_component / abs(dc_component)) * 100
            else:
                pi_percent = np.nan
            
            return pi_percent, dc_component, ac_component
            
        except Exception as e:
            logger.error(f"Error in calculate_perfusion_index: {str(e)}")
            return np.nan, np.nan, np.nan
    
    def calculate_statistical_features(self, signal_data):
        """
        統計的特徴量の計算
        
        文献根拠:
        - Mohagheghian et al. (2022): 特徴量ベースの信号品質評価
        
        Returns:
        --------
        features : dict
            統計的特徴量
        """
        try:
            features = {
                'mean': float(np.mean(signal_data)),
                'std': float(np.std(signal_data)),
                'skewness': float(skew(signal_data)),
                'kurtosis': float(kurtosis(signal_data)),
                'peak_to_peak': float(np.max(signal_data) - np.min(signal_data)),
                'cv': float(np.std(signal_data) / (abs(np.mean(signal_data)) + 1e-10)),
                'rms': float(np.sqrt(np.mean(signal_data**2)))
            }
            return features
            
        except Exception as e:
            logger.error(f"Error in calculate_statistical_features: {str(e)}")
            return {}
    
    def comprehensive_quality_assessment(self, signal_data):
        """
        包括的品質評価
        
        Returns:
        --------
        quality_metrics : dict
            全品質指標
        """
        try:
            quality_metrics = {}
            
            # SNR計算（Welch法 - より安定）
            snr_welch, sig_power, noise_power = self.calculate_snr_welch(signal_data)
            quality_metrics['SNR_Welch_dB'] = snr_welch
            quality_metrics['Signal_Power'] = sig_power
            quality_metrics['Noise_Power'] = noise_power
            
            # SNR計算（FFT法 - 比較用）
            snr_fft = self.calculate_snr_fft(signal_data)
            quality_metrics['SNR_FFT_dB'] = snr_fft
            
            # Perfusion Index
            pi, dc, ac = self.calculate_perfusion_index(signal_data)
            quality_metrics['PI_Percent'] = pi
            quality_metrics['DC_Component'] = dc
            quality_metrics['AC_Component'] = ac
            
            # 統計的特徴
            stat_features = self.calculate_statistical_features(signal_data)
            quality_metrics.update(stat_features)
            
            return quality_metrics
            
        except Exception as e:
            logger.error(f"Error in comprehensive_quality_assessment: {str(e)}")
            return {}


class PPGPreprocessor:
    """
    PPG信号前処理クラス（既存コードベース）
    文献ベースの最適な手法を実装
    """
    
    def __init__(self, fs=2000.0):
        self.fs = fs
        logger.info(f"PPGPreprocessor initialized with fs={fs} Hz")
        
    def bandpass_filter(self, signal_data, low_cut=0.5, high_cut=10.0, order=4):
        """
        Chebyshev Type II バンドパスフィルタ
        
        文献根拠:
        - Liang et al. (2018): 4th-order Chebyshev Type-II filter が最適
        """
        try:
            nyq = 0.5 * self.fs
            low = low_cut / nyq
            high = high_cut / nyq
            
            sos = signal.cheby2(order, rs=40, Wn=[low, high], btype='band', output='sos')
            filtered_signal = signal.sosfiltfilt(sos, signal_data)
            
            return filtered_signal
            
        except Exception as e:
            logger.error(f"Error in bandpass_filter: {str(e)}")
            return signal_data
    
    def detect_artifacts_mad(self, signal_data, threshold=3.5):
        """
        MAD (Median Absolute Deviation) ベースのアーティファクト検出
        
        文献根拠:
        - Leys et al. (2013): MADは外れ値検出に対してロバスト
        """
        try:
            median = np.median(signal_data)
            mad = np.median(np.abs(signal_data - median))
            
            if mad == 0:
                std = np.std(signal_data)
                artifact_mask = np.abs(signal_data - np.mean(signal_data)) > threshold * std
            else:
                modified_z_scores = 0.6745 * (signal_data - median) / mad
                artifact_mask = np.abs(modified_z_scores) > threshold
            
            return artifact_mask
            
        except Exception as e:
            logger.error(f"Error in detect_artifacts_mad: {str(e)}")
            return np.zeros(len(signal_data), dtype=bool)
    
    def remove_artifacts(self, signal_data, artifact_mask, method='interpolation'):
        """
        アーティファクト除去
        """
        try:
            cleaned_signal = signal_data.copy()
            
            if method == 'interpolation':
                valid_indices = np.where(~artifact_mask)[0]
                artifact_indices = np.where(artifact_mask)[0]
                
                if len(valid_indices) > 1 and len(artifact_indices) > 0:
                    cleaned_signal[artifact_indices] = np.interp(
                        artifact_indices,
                        valid_indices,
                        signal_data[valid_indices]
                    )
            
            elif method == 'median':
                kernel_size = int(self.fs * 0.1)
                if kernel_size % 2 == 0:
                    kernel_size += 1
                cleaned_signal[artifact_mask] = median_filter(signal_data, size=kernel_size)[artifact_mask]
            
            return cleaned_signal
            
        except Exception as e:
            logger.error(f"Error in remove_artifacts: {str(e)}")
            return signal_data
    
    def baseline_correction_spline(self, signal_data, smoothness=0.01):
        """
        Cubic Spline補間によるベースライン補正
        
        文献根拠:
        - Wang et al. (2007): Cubic spline interpolationが効果的
        """
        try:
            x = np.arange(len(signal_data))
            
            window_size = int(self.fs * 0.8)
            if window_size % 2 == 0:
                window_size += 1
            
            baseline_estimate = minimum_filter(signal_data, size=window_size, mode='nearest')
            
            s_param = smoothness * len(signal_data)
            spline = UnivariateSpline(x, baseline_estimate, s=s_param, k=3)
            baseline = spline(x)
            
            corrected_signal = signal_data - baseline
            
            return corrected_signal
            
        except Exception as e:
            logger.error(f"Error in baseline_correction_spline: {str(e)}")
            return signal_data
    
    def full_preprocessing_pipeline(self, signal_data, 
                                   bandpass_params={'low_cut': 0.5, 'high_cut': 10.0, 'order': 4},
                                   artifact_params={'threshold': 3.5, 'method': 'interpolation'},
                                   baseline_params={'method': 'spline', 'smoothness': 0.01}):
        """
        完全な前処理パイプライン
        
        処理順序:
        1. Bandpass filtering
        2. Artifact detection & removal
        3. Baseline correction
        """
        try:
            logger.debug("Starting full preprocessing pipeline")
            processing_info = {}
            
            # 入力統計
            processing_info['input_stats'] = {
                'length': len(signal_data),
                'mean': float(np.mean(signal_data)),
                'std': float(np.std(signal_data))
            }
            
            # Step 1: Bandpass filtering
            filtered_signal = self.bandpass_filter(signal_data, **bandpass_params)
            
            # Step 2: Artifact detection & removal
            artifact_mask = self.detect_artifacts_mad(filtered_signal, threshold=artifact_params['threshold'])
            artifact_removed_signal = self.remove_artifacts(filtered_signal, artifact_mask, method=artifact_params['method'])
            
            processing_info['artifacts'] = {
                'count': int(np.sum(artifact_mask)),
                'percentage': float((np.sum(artifact_mask) / len(signal_data)) * 100)
            }
            
            # Step 3: Baseline correction
            if baseline_params['method'] == 'spline':
                processed_signal = self.baseline_correction_spline(
                    artifact_removed_signal, 
                    smoothness=baseline_params['smoothness']
                )
            else:
                processed_signal = artifact_removed_signal
            
            # 出力統計
            processing_info['output_stats'] = {
                'mean': float(np.mean(processed_signal)),
                'std': float(np.std(processed_signal))
            }
            
            return processed_signal, processing_info
            
        except Exception as e:
            logger.error(f"Error in full_preprocessing_pipeline: {str(e)}")
            return signal_data, {}


def process_all_files_with_quality_assessment():
    """
    全ファイルの処理 + 品質評価（統合版メイン関数）
    """
    logger.info("="*80)
    logger.info("PPG Signal Quality Assessment & Enhancement Pipeline")
    logger.info("前処理 + 品質評価の統合処理")
    logger.info("="*80)
    
    # パラメータ設定
    depths = [3, 9, 15, 21]
    state_map = {1: "Normal", 2: "Ischaemia", 3: "Congestion"}
    channels = {"Red": "ppgA_Red_raw", "IR": "ppgA_IR_raw"}
    
    # 初期化
    preprocessor = PPGPreprocessor(fs=2000.0)
    quality_assessor = PPGQualityAssessor(fs=2000.0)
    
    # 品質評価結果を保存
    quality_records = []
    
    # 統計
    total_files = 0
    successful_files = 0
    
    # 全ファイルを処理
    for depth in depths:
        for i_state, state_name in state_map.items():
            total_files += 1
            
            input_filename = f"{depth}mm_adjusted_data{i_state}.csv"
            output_filename = f"{depth}mm_processed_data{i_state}.csv"
            
            logger.info("-" * 80)
            logger.info(f"Processing: {input_filename} (Depth: {depth}mm, State: {state_name})")
            logger.info("-" * 80)
            
            try:
                if not os.path.exists(input_filename):
                    logger.error(f"File not found: {input_filename}")
                    continue
                
                df = pd.read_csv(input_filename)
                logger.info(f"Loaded: {input_filename} ({len(df)} rows)")
                
                # 各チャンネルを処理
                for channel_name, column_name in channels.items():
                    logger.info(f"  Channel: {channel_name}")
                    
                    if column_name not in df.columns:
                        logger.error(f"Column '{column_name}' not found")
                        continue
                    
                    # 生信号
                    raw_signal = df[column_name].values
                    
                    # NaN/Inf処理
                    if np.any(np.isnan(raw_signal)) or np.any(np.isinf(raw_signal)):
                        raw_signal = np.nan_to_num(raw_signal, nan=np.nanmedian(raw_signal))
                    
                    # === 前処理前の品質評価 ===
                    raw_quality = quality_assessor.comprehensive_quality_assessment(raw_signal)
                    logger.info(f"    Raw SNR (Welch): {raw_quality['SNR_Welch_dB']:.2f} dB")
                    logger.info(f"    Raw PI: {raw_quality['PI_Percent']:.4f} %")
                    
                    # 品質記録
                    record_raw = {
                        'Depth_mm': depth,
                        'State': state_name,
                        'Channel': channel_name,
                        'Processing': 'Raw'
                    }
                    record_raw.update(raw_quality)
                    quality_records.append(record_raw)
                    
                    # === 前処理実行 ===
                    processed_signal, processing_info = preprocessor.full_preprocessing_pipeline(
                        raw_signal,
                        bandpass_params={'low_cut': 0.5, 'high_cut': 10.0, 'order': 4},
                        artifact_params={'threshold': 3.5, 'method': 'interpolation'},
                        baseline_params={'method': 'spline', 'smoothness': 0.01}
                    )
                    
                    # === 前処理後の品質評価 ===
                    processed_quality = quality_assessor.comprehensive_quality_assessment(processed_signal)
                    snr_improvement = processed_quality['SNR_Welch_dB'] - raw_quality['SNR_Welch_dB']
                    
                    logger.info(f"    Processed SNR (Welch): {processed_quality['SNR_Welch_dB']:.2f} dB")
                    logger.info(f"    Processed PI: {processed_quality['PI_Percent']:.4f} %")
                    logger.info(f"    ✓ SNR Improvement: {snr_improvement:+.2f} dB")
                    logger.info(f"    Artifacts removed: {processing_info['artifacts']['count']} ({processing_info['artifacts']['percentage']:.2f}%)")
                    
                    # 品質記録
                    record_processed = {
                        'Depth_mm': depth,
                        'State': state_name,
                        'Channel': channel_name,
                        'Processing': 'Processed'
                    }
                    record_processed.update(processed_quality)
                    record_processed['SNR_Improvement_dB'] = snr_improvement
                    record_processed['Artifacts_Count'] = processing_info['artifacts']['count']
                    record_processed['Artifacts_Percent'] = processing_info['artifacts']['percentage']
                    quality_records.append(record_processed)
                    
                    # 前処理済み信号を保存
                    new_column_name = f"{channel_name}_processed"
                    df[new_column_name] = processed_signal
                
                # CSV保存
                df.to_csv(output_filename, index=False)
                logger.info(f"✓ Saved: {output_filename}")
                successful_files += 1
                
            except Exception as e:
                logger.error(f"✗ Failed: {input_filename}")
                logger.exception(f"Error: {str(e)}")
    
    # === 品質評価サマリーの保存 ===
    if quality_records:
        df_quality = pd.DataFrame(quality_records)
        quality_summary_file = "PPG_Quality_Assessment_Summary.csv"
        df_quality.to_csv(quality_summary_file, index=False)
        logger.info(f"\n✅ Quality assessment saved: {quality_summary_file}")
        
        # 統計サマリー
        logger.info("\n" + "="*80)
        logger.info("QUALITY ASSESSMENT SUMMARY")
        logger.info("="*80)
        
        for proc_type in ['Raw', 'Processed']:
            subset = df_quality[df_quality['Processing'] == proc_type]
            if len(subset) > 0:
                snr_vals = subset['SNR_Welch_dB'].replace([np.inf, -np.inf], np.nan).dropna()
                pi_vals = subset['PI_Percent'].replace([np.inf, -np.inf], np.nan).dropna()
                
                logger.info(f"\n{proc_type} Signal Quality:")
                if len(snr_vals) > 0:
                    logger.info(f"  SNR (Welch): {snr_vals.mean():.2f} ± {snr_vals.std():.2f} dB")
                    logger.info(f"  SNR Range: [{snr_vals.min():.2f}, {snr_vals.max():.2f}] dB")
                if len(pi_vals) > 0:
                    logger.info(f"  PI: {pi_vals.mean():.4f} ± {pi_vals.std():.4f} %")
        
        # SNR改善度の分析
        processed_subset = df_quality[df_quality['Processing'] == 'Processed']
        if 'SNR_Improvement_dB' in processed_subset.columns:
            improvements = processed_subset['SNR_Improvement_dB'].replace([np.inf, -np.inf], np.nan).dropna()
            if len(improvements) > 0:
                logger.info(f"\nSNR Improvement Statistics:")
                logger.info(f"  Mean: {improvements.mean():.2f} dB")
                logger.info(f"  Median: {improvements.median():.2f} dB")
                logger.info(f"  Range: [{improvements.min():.2f}, {improvements.max():.2f}] dB")
    
    # 処理サマリー
    logger.info("\n" + "="*80)
    logger.info("PROCESSING SUMMARY")
    logger.info("="*80)
    logger.info(f"Total files: {total_files}")
    logger.info(f"Successful: {successful_files}")
    logger.info(f"Success rate: {(successful_files/total_files)*100:.1f}%")
    logger.info(f"Log file: {log_filename}")
    logger.info("="*80)


if __name__ == "__main__":
    process_all_files_with_quality_assessment()

2026-01-29 11:38:19,848 - INFO - ================================================================================
2026-01-29 11:38:19,849 - INFO - PPG Signal Quality (SNR) Table Generator
2026-01-29 11:38:19,849 - INFO - ================================================================================
2026-01-29 11:38:19,849 - INFO - --------------------------------------------------------------------------------
2026-01-29 11:38:19,850 - INFO - Processing: 3mm_adjusted_data1.csv (Depth: 3mm, State: Normal)
2026-01-29 11:38:19,961 - INFO - Loaded: 3mm_adjusted_data1.csv (61200 rows)
2026-01-29 11:38:19,983 - INFO -   Red: SNR = 15.33 dB
2026-01-29 11:38:19,987 - INFO -   IR: SNR = 18.78 dB
2026-01-29 11:38:19,987 - INFO - --------------------------------------------------------------------------------
2026-01-29 11:38:19,987 - INFO - Processing: 3mm_adjusted_data2.csv (Depth: 3mm, State: Ischaemia)
2026-01-29 11:38:20,035 - INFO - Loaded: 3mm_adjusted_data2.csv (61200 rows)
2026-01-29 1


 Depth      State Channel   SNR
     3     Normal     Red 15.33
     3     Normal      IR 18.78
     3  Ischaemia     Red 11.26
     3  Ischaemia      IR 17.36
     3 Congestion     Red 15.15
     3 Congestion      IR 19.59
     9     Normal     Red  6.70
     9     Normal      IR 17.93
     9  Ischaemia     Red  3.71
     9  Ischaemia      IR 14.92
     9 Congestion     Red  7.91
     9 Congestion      IR 17.96
    15     Normal     Red  4.36
    15     Normal      IR 15.44
    15  Ischaemia     Red  1.33
    15  Ischaemia      IR 11.00
    15 Congestion     Red  1.51
    15 Congestion      IR 14.71
    21     Normal     Red  3.42
    21     Normal      IR 11.17


In [ ]:
#waveform visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, savgol_filter
import matplotlib.ticker as ticker
import math

# ========= パラメータ設定 =========
fs = 2000  # サンプリングレート [Hz]
start_time = 15.0    # 基本の表示開始時間 [秒]
display_time = 2.0   # 表示時間 [秒]
OFFSET_SAMPLES = 500 # MA安定化のためのオフセット点数 (0.25s)
OFFSET_TIME = OFFSET_SAMPLES / fs  

depths = [3, 9, 15, 21]
states = ["Normal", "Ischaemia", "Congestion"]
legend_labels = ['Normal', 'Ischaemia', 'Congestion']
channels = {"Red": "ppgA_Red_raw", "IR": "ppgA_IR_raw"} 

# --- フィルタリング＆スムージング設定 ---
BSF_LOW_CUT = 20.0  
BSF_HIGH_CUT = 50.0 
BSF_ORDER = 4       
MA_WINDOW = 200     
SG_WINDOW = 51 
SG_POLY = 3

# --- 手動ピーク位置合わせ設定（★修正: IR 9mm, 15mm の全状態に+0.75s, +0.4s適用） ---
manual_start_offsets_s = {
    3: {"Red": {"Normal": 0.0, "Ischaemia": 0.35, "Congestion": 0.7}, 
        "IR": {"Normal": 0.0, "Ischaemia": -0.7, "Congestion": -0.3}},
    9: {"Red": {"Normal": 0.2, "Ischaemia": 0.0, "Congestion": 0.0}, 
        "IR": {"Normal": 0.75, "Ischaemia": 0.75, "Congestion": 0.75 + 0.2}},  # ★全状態に+0.75s
    15: {"Red": {"Normal": 0.0, "Ischaemia": 0.5, "Congestion": 0.1}, 
         "IR": {"Normal": 0.4, "Ischaemia": 0.4 - 0.5, "Congestion": 0.4 + 0.03}},  # ★全状態に+0.4s
    21: {"Red": {"Normal": 0.0, "Ischaemia": 0.35, "Congestion": 0.0}, 
         "IR": {"Normal": 0.0, "Ischaemia": 0.2, "Congestion": 0.4}},
}

# ========= Sci Rep準拠の設定 =========
# フォント設定（Arial推奨、8-12 pt）
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 9  # 基本フォントサイズ
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9

# 線幅設定（最低1 pt以上）
plt.rcParams['lines.linewidth'] = 1.2
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['ytick.major.width'] = 1.0

# PDF出力設定
plt.rcParams['pdf.fonttype'] = 42  # TrueType埋め込み
plt.rcParams['ps.fonttype'] = 42

def calculate_amplitude(signal):
    """信号のPeak-to-Peak振幅を計算 (Max - Min)"""
    return np.max(signal) - np.min(signal)

def bandstop_filter_noise_reduction(sig, fs, low_cutoff, high_cutoff, order=BSF_ORDER):
    nyq = fs / 2
    b, a = butter(order, [low_cutoff / nyq, high_cutoff / nyq], btype='stop')
    return filtfilt(b, a, sig)

def moving_average_smoothing(data, window):
    if len(data) < window: return data
    ret = np.cumsum(data, dtype=float)
    ret[window:] = ret[window:] - ret[:-window]
    smoothed = ret[window - 1:] / window
    padding_val = smoothed[0]
    padded_start = np.full(window - 1, padding_val)
    return np.concatenate((padded_start, smoothed))

def process_signal_and_center(df, ch_col, load_start_index, load_end_index):
    """BSFとMAを適用し、DCゼロセンター化した信号を返す"""
    signal = df[ch_col].values[load_start_index:load_end_index]
    signal_filtered = bandstop_filter_noise_reduction(signal, fs, BSF_LOW_CUT, BSF_HIGH_CUT)
    smoothed = moving_average_smoothing(signal_filtered, window=MA_WINDOW)
    return smoothed - np.mean(smoothed)

# ========= Figure設定（ダブルカラム183mm ≈ 7.2インチ） =========
fig_width_inch = 7.2  # ダブルカラム幅 (183mm)
fig_height_inch = 9.0  # 縦横比を考慮
fig, axes = plt.subplots(len(depths), len(channels), 
                         figsize=(fig_width_inch, fig_height_inch), 
                         sharex=False, sharey=False) 
plot_data_length = int(display_time * fs)

# 1. 最大オフセットの決定
max_abs_offset = 0.0
for d in depths:
    for ch in channels:
        for state in states:
            offset = manual_start_offsets_s[d][ch][state]
            max_abs_offset = max(max_abs_offset, abs(offset))

# 2. 動的な読み込み範囲の決定
load_start_time_final = start_time - OFFSET_TIME - max_abs_offset
load_start_index_final = int(load_start_time_final * fs)
load_end_time_final = start_time + display_time + max_abs_offset
load_end_index_final = int(load_end_time_final * fs)

# 3. 基準となるゼロオフセット
zero_offset_time = OFFSET_TIME + max_abs_offset
zero_offset_samples = int(zero_offset_time * fs)

for row, depth in enumerate(depths):
    for col, (ch_name, ch_col) in enumerate(channels.items()):
        ax = axes[row, col]
        
        # 背景色の設定
        if (ch_name == "Red" and depth in [9, 15, 21]) or (ch_name == "IR" and depth == 21):
            ax.set_facecolor('#F0F0F0')
        
        processed_data = {}
        max_amplitude = 0.0
        
        # 1. 全てのStateのデータを処理
        for i, state in enumerate(states, start=1):
            file = f"{depth}mm_adjusted_data{i}.csv" 
            try:
                df = pd.read_csv(file)
                
                signal_centered = process_signal_and_center(df, ch_col, load_start_index_final, load_end_index_final)
                amplitude = calculate_amplitude(signal_centered)
                
                if amplitude > max_amplitude:
                    max_amplitude = amplitude
                
                processed_data[state] = {
                    'signal': signal_centered,
                    'amplitude': amplitude,
                    'manual_offset_s': manual_start_offsets_s[depth][ch_name][state]
                }
                
            except FileNotFoundError:
                processed_data[state] = None
        
        # 2. 相対振幅スケーリングとプロット
        reference_amplitude = max_amplitude 
        
        for i, state in enumerate(states, start=1):
            data_info = processed_data.get(state)
            if data_info is None:
                continue
                
            sig = data_info['signal']
            amp = data_info['amplitude']
            manual_offset_s = data_info['manual_offset_s']
            
            # 相対振幅正規化
            if reference_amplitude > 1e-12:
                 signal_scaled = sig * (1.0 / reference_amplitude) 
            else:
                 signal_scaled = np.zeros_like(sig)
            
            # プロットデータの切り出し
            manual_offset_samples = int(round(manual_offset_s * fs))
            plot_start_index = zero_offset_samples + manual_offset_samples
            plot_end_index = plot_start_index + plot_data_length
            
            if plot_end_index > len(signal_scaled) or plot_start_index < 0:
                 continue
                 
            signal_norm = signal_scaled[plot_start_index:plot_end_index]
            time = np.arange(0, display_time * fs) / fs 

            # プロット（線幅1.2 pt以上）
            ax.plot(time, signal_norm, label=legend_labels[i-1], linewidth=1.2)

        # 軸ラベルとタイトル
        ax.set_title(f"{ch_name} {depth} mm", fontsize=10)
        ax.tick_params(axis='both', which='major', labelsize=9, width=1.0) 
        
        ax.set_xlim(0, display_time)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5)) 
        ax.set_ylim(-0.6, 0.6)
        ax.grid(True, linewidth=0.5)

        if col == 0:
            ax.set_ylabel('Amplitude', fontsize=10)
        if row == len(depths)-1:
            ax.set_xlabel('Time (s)', fontsize=10)

# 凡例
handles, labels = axes[0,0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, fontsize=9, 
           bbox_to_anchor=(0.5, 1.00), frameon=False)

plt.tight_layout(rect=[0, 0, 1, 0.98])

# ========= 保存（Sci Rep準拠） =========
# 1. TIFF形式（600 dpi推奨）
plt.savefig('PPG_waveform_comparison.tiff', dpi=600, format='tiff', 
            bbox_inches='tight', pil_kwargs={'compression': 'tiff_lzw'})

# 2. PDF形式（ベクター、フォント埋め込み）
plt.savefig('PPG_waveform_comparison.pdf', dpi=600, format='pdf', 
            bbox_inches='tight')

# 3. EPS形式（オプション）
plt.savefig('PPG_waveform_comparison.eps', dpi=600, format='eps', 
            bbox_inches='tight')

print("✅ 図を保存しました:")
print("  - PPG_waveform_comparison.tiff (600 dpi)")
print("  - PPG_waveform_comparison.pdf (ベクター)")
print("  - PPG_waveform_comparison.eps (ベクター)")

plt.show()

In [ ]:
"""
Master Feature Extraction: 3mm Red Channel Only (CORRECTED VERSION)
====================================================================
Extracts comprehensive pulse-level features from 3mm depth Red channel
FIXED: Delta_T calculation using robust peak-to-trough method
Following the preprocessing pipeline from the reference code
Output format designed for easy integration with other wavelengths/depths
"""

import pandas as pd
import numpy as np
from scipy import signal, stats
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ========================================
# CONFIGURATION
# ========================================
FS = 2000
TARGET_DEPTH = 3
TARGET_CHANNEL = 'Red'
RAW_COLUMN = 'ppgA_Red_raw'
PROCESSED_COLUMN = 'Red_processed'

STATE_MAP = {1: 'Normal', 2: 'Ischaemia', 3: 'Congestion'}

OUTPUT_DIR = 'master_features_3mm_red'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, 'processing_log.txt')

def log_message(message):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f"[{timestamp}] {message}"
    print(entry)
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(entry + '\n')

# ========================================
# HELPER FUNCTIONS
# ========================================

def calculate_delta_t_robust(vpg, t, fs):
    """
    Robust Delta_T calculation using multiple methods
    
    Delta_T = Time interval between:
    1. Maximum acceleration point (VPG peak)
    2. Maximum deceleration point (first significant VPG trough after peak)
    
    Parameters:
    -----------
    vpg : array
        First derivative of normalized PPG signal (velocity)
    t : array
        Time array
    fs : int
        Sampling frequency
    
    Returns:
    --------
    float : Delta_T in seconds
    """
    # Method 1: Peak-to-trough approach (most reliable)
    # Find VPG maximum (maximum acceleration during systole)
    vpg_max_idx = np.argmax(vpg)
    
    # Define search region after VPG peak
    search_start = vpg_max_idx + int(0.02 * fs)  # Start 20ms after peak
    search_end = min(vpg_max_idx + int(0.4 * fs), len(vpg))  # Search within 400ms
    
    if search_end > search_start and search_start < len(vpg):
        vpg_search_region = vpg[search_start:search_end]
        
        # Find troughs (negative peaks) in search region
        troughs, properties = signal.find_peaks(
            -vpg_search_region, 
            prominence=0.03 * np.max(np.abs(vpg)),
            distance=int(0.02 * fs)  # Minimum 20ms between troughs
        )
        
        if len(troughs) > 0:
            # First significant trough after VPG peak
            first_trough_idx = search_start + troughs[0]
            delta_t = (first_trough_idx - vpg_max_idx) / fs
            
            # Sanity check: Delta_T should be between 50ms and 400ms
            if 0.05 <= delta_t <= 0.4:
                return delta_t
    
    # Method 2: Zero-crossing approach (fallback)
    # Find zero-crossings in VPG
    zero_crossings = np.where(np.diff(np.sign(vpg)))[0]
    
    if len(zero_crossings) >= 2:
        # Find zero-crossings bracketing the VPG peak
        zc_before_peak = zero_crossings[zero_crossings < vpg_max_idx]
        zc_after_peak = zero_crossings[zero_crossings > vpg_max_idx]
        
        if len(zc_before_peak) > 0 and len(zc_after_peak) > 0:
            # Take closest zero-crossings before and after peak
            zc1 = zc_before_peak[-1]
            zc2 = zc_after_peak[0]
            delta_t = (zc2 - zc1) / fs
            
            # Sanity check
            if 0.05 <= delta_t <= 0.4:
                return delta_t
    
    # Method 3: Threshold-based approach (second fallback)
    # Find where VPG drops below a threshold after peak
    threshold = 0.1 * vpg[vpg_max_idx]  # 10% of peak value
    after_peak = vpg[vpg_max_idx:]
    
    if len(after_peak) > 10:
        below_threshold = np.where(after_peak < threshold)[0]
        if len(below_threshold) > 0:
            delta_t = below_threshold[0] / fs
            
            # Sanity check
            if 0.05 <= delta_t <= 0.4:
                return delta_t
    
    # Method 4: Default estimate based on pulse width (last resort)
    # Typically Delta_T is 15-25% of pulse duration
    pulse_duration = len(vpg) / fs
    delta_t = 0.2 * pulse_duration
    
    # Clamp to reasonable range
    delta_t = np.clip(delta_t, 0.05, 0.4)
    
    return delta_t

# ========================================
# FEATURE EXTRACTION
# ========================================

def extract_comprehensive_features(seg_raw, seg_processed, fs=2000):
    """
    Extract all pulse-level features from single pulse
    
    Parameters:
    -----------
    seg_raw : array
        Raw signal segment (one pulse, before preprocessing)
    seg_processed : array
        Processed signal segment (one pulse, after preprocessing)
    fs : int
        Sampling frequency
    
    Returns:
    --------
    dict : Dictionary containing all features
    """
    f = {}
    
    # ===== PREPROCESSING COMPONENTS =====
    # DC component: mean of RAW signal (before preprocessing)
    dc_component = np.mean(seg_raw)
    f['DC_Component'] = dc_component
    
    # AC component: peak-to-peak of PROCESSED signal (after preprocessing)
    ac_component = np.max(seg_processed) - np.min(seg_processed)
    f['AC_Component'] = ac_component
    
    # Perfusion Index (PI): AC/DC ratio as percentage
    f['PI'] = (ac_component / (dc_component + 1e-10)) * 100
    
    # ===== NORMALIZATION FOR SHAPE ANALYSIS =====
    # 0-1 normalization of processed signal for shape features
    seg_min = np.min(seg_processed)
    seg_max = np.max(seg_processed)
    seg_norm = (seg_processed - seg_min) / (seg_max - seg_min + 1e-10)
    
    # Time array
    t = np.arange(len(seg_norm)) / fs
    total_len = len(seg_norm)
    
    # Systolic peak index
    s_peak_idx = np.argmax(seg_norm)
    
    # ===== INTENSITY / AMPLITUDE FEATURES =====
    f['Amplitude'] = ac_component
    f['Max_Start_Datum_Diff'] = np.max(seg_processed[:total_len//4]) if total_len >= 4 else 0
    
    # ===== AREA FEATURES =====
    # Linear baseline correction for area calculation
    baseline = np.linspace(seg_processed[0], seg_processed[-1], total_len)
    sig_bc = seg_processed - baseline
    
    f['AUC'] = np.trapz(sig_bc)
    f['S_AUC'] = np.trapz(sig_bc[:s_peak_idx+1])
    f['D_AUC'] = np.trapz(sig_bc[s_peak_idx:])
    f['AUC_Ratio'] = f['S_AUC'] / (f['D_AUC'] + 1e-10)
    
    # Datum areas (quarter segments)
    q = total_len // 4
    if q > 0:
        f['Start_Datum_Area'] = np.trapz(sig_bc[:q])
        f['End_Datum_Area'] = np.trapz(sig_bc[-q:])
        f['Datum_Area_Ratio'] = f['Start_Datum_Area'] / (f['End_Datum_Area'] + 1e-10)
    else:
        f['Start_Datum_Area'] = 0
        f['End_Datum_Area'] = 0
        f['Datum_Area_Ratio'] = 0
    
    # ===== TIME / WIDTH FEATURES =====
    f['Rise_Time'] = s_peak_idx / fs
    f['Fall_Time'] = (total_len - s_peak_idx) / fs
    f['Pulse_Width'] = total_len / fs
    f['Rise_Decay_Time_Ratio'] = f['Rise_Time'] / (f['Fall_Time'] + 1e-10)
    
    # PW50: width at 50% amplitude
    above_50 = np.where(seg_norm >= 0.5)[0]
    f['PW50'] = (above_50[-1] - above_50[0]) / fs if len(above_50) > 0 else 0
    
    # Systolic/Diastolic width (estimate dicrotic notch at 60-70% of pulse)
    notch_idx = int(s_peak_idx + 0.3 * (total_len - s_peak_idx))
    f['Systolic_Width'] = notch_idx / fs
    f['Diastolic_Width'] = (total_len - notch_idx) / fs
    f['Width_Ratio'] = f['Systolic_Width'] / (f['Diastolic_Width'] + 1e-10)
    
    # ===== SLOPE FEATURES =====
    # 1st derivative (VPG - Velocity Photoplethysmogram)
    vpg = np.gradient(seg_norm, t)
    
    f['Upslope'] = np.max(vpg)
    f['Downslope'] = np.min(vpg)
    f['Onset_End_Slope'] = (seg_norm[-1] - seg_norm[0]) / (t[-1] - t[0] + 1e-10)
    f['Slope_Ratio'] = f['Upslope'] / (abs(f['Downslope']) + 1e-10)
    
    # ===== DELTA_T (CORRECTED) =====
    # Use robust calculation method
    f['Delta_T'] = calculate_delta_t_robust(vpg, t, fs)
    
    # Arc length features
    dx = np.diff(t)
    dy = np.diff(seg_norm)
    arc_increments = np.sqrt(dx**2 + dy**2)
    total_arc = np.sum(arc_increments)
    upslope_arc = np.sum(arc_increments[:s_peak_idx]) if s_peak_idx > 0 else 0
    downslope_arc = np.sum(arc_increments[s_peak_idx:])
    
    f['Upslope_Length'] = upslope_arc
    f['Downslope_Length'] = downslope_arc
    f['Slope_Length_Ratio'] = upslope_arc / (downslope_arc + 1e-10)
    f['Upslope_Length_Ratio'] = upslope_arc / (total_arc + 1e-10)
    f['Downslope_Length_Ratio'] = downslope_arc / (total_arc + 1e-10)
    f['Length_Height_Ratio'] = total_arc / (seg_norm[s_peak_idx] + 1e-10)
    
    # ===== SDPPG (2nd derivative / APG - Acceleration Photoplethysmogram) =====
    apg = np.gradient(vpg, t)
    
    # SDPPG a-wave (early systolic acceleration peak)
    a_wave_idx_max = min(int(0.2 * fs), len(apg))
    a_wave = np.max(apg[:a_wave_idx_max]) if a_wave_idx_max > 0 else 1e-10
    
    # SDPPG indices (with safe bounds checking)
    def safe_apg_ratio(start_frac, end_frac, mode='max'):
        start_idx = int(start_frac * fs)
        end_idx = int(end_frac * fs)
        if start_idx >= len(apg) or end_idx > len(apg) or start_idx >= end_idx:
            return 0
        segment = apg[start_idx:end_idx]
        if len(segment) == 0:
            return 0
        val = np.max(segment) if mode == 'max' else np.min(segment)
        return val / (a_wave + 1e-10)
    
    f['SDPPG_b_a'] = safe_apg_ratio(0.05, 0.15, 'min')
    f['SDPPG_c_a'] = safe_apg_ratio(0.15, 0.25, 'max')
    f['SDPPG_d_a'] = safe_apg_ratio(0.25, 0.35, 'min')
    f['SDPPG_e_a'] = safe_apg_ratio(0.35, 0.50, 'max')
    
    # DNP (Dicrotic Notch Prominence)
    search_start = s_peak_idx + int(0.1 * fs)
    if search_start < len(seg_norm) - 10:
        f['DNP'] = seg_norm[notch_idx] if notch_idx < len(seg_norm) else 0
    else:
        f['DNP'] = 0
    
    # ===== SHAPE & STATS =====
    f['Skewness'] = stats.skew(seg_norm)
    f['Kurtosis'] = stats.kurtosis(seg_norm)
    f['Variance'] = np.var(seg_norm)
    
    # RWI (Reflection Wave Index)
    if search_start < len(seg_norm) - 10:
        diastolic_seg = seg_norm[search_start:]
        d_peak = np.max(diastolic_seg)
        f['RWI'] = d_peak / (seg_norm[s_peak_idx] + 1e-10)
    else:
        f['RWI'] = 0
    
    # IPAR (Inflection Point Area Ratio)
    inflection_idx = int(0.3 * total_len)
    if inflection_idx > 0 and inflection_idx < len(sig_bc):
        area_before = np.trapz(sig_bc[:inflection_idx])
        area_after = np.trapz(sig_bc[inflection_idx:])
        f['IPAR'] = area_before / (area_after + 1e-10)
    else:
        f['IPAR'] = 0
    
    return f

# ========================================
# MAIN PROCESSING
# ========================================

def main():
    log_message("="*80)
    log_message(f"MASTER FEATURE EXTRACTION: {TARGET_DEPTH}mm {TARGET_CHANNEL} Channel")
    log_message(f"CORRECTED VERSION: Delta_T calculation fixed")
    log_message("="*80)
    
    all_features = []
    delta_t_stats = {'zero_count': 0, 'valid_count': 0, 'values': []}
    
    for state_code, state_name in STATE_MAP.items():
        # Load processed data file
        filename = f'{TARGET_DEPTH}mm_processed_data{state_code}.csv'
        
        if not os.path.exists(filename):
            log_message(f"⚠️  File not found: {filename}")
            continue
        
        try:
            df = pd.read_csv(filename)
            log_message(f"📂 Processing: {filename} ({state_name})")
            
            # Check for required columns
            if RAW_COLUMN not in df.columns:
                log_message(f"  ⚠️  Raw column '{RAW_COLUMN}' not found")
                continue
            if PROCESSED_COLUMN not in df.columns:
                log_message(f"  ⚠️  Processed column '{PROCESSED_COLUMN}' not found")
                continue
            
            # Extract 30 seconds of data
            max_samples = int(30 * FS)
            raw_signal = df[RAW_COLUMN].values[:max_samples]
            processed_signal = df[PROCESSED_COLUMN].values[:max_samples]
            
            # Detect pulse boundaries using processed signal
            # Find troughs (valleys) in inverted signal
            troughs, _ = signal.find_peaks(-processed_signal, distance=int(FS * 0.6))
            
            pulse_count = 0
            for i in range(len(troughs) - 1):
                idx1, idx2 = troughs[i], troughs[i+1]
                
                # Quality filter: pulse duration between 0.4 and 1.5 seconds
                pulse_duration = (idx2 - idx1) / FS
                if pulse_duration < 0.4 or pulse_duration > 1.5:
                    continue
                
                # Extract pulse segments
                seg_raw = raw_signal[idx1:idx2]
                seg_processed = processed_signal[idx1:idx2]
                
                # Extract features
                features = extract_comprehensive_features(seg_raw, seg_processed, FS)
                
                # Track Delta_T statistics
                if features['Delta_T'] == 0:
                    delta_t_stats['zero_count'] += 1
                else:
                    delta_t_stats['valid_count'] += 1
                    delta_t_stats['values'].append(features['Delta_T'])
                
                # Add metadata
                features['Depth'] = TARGET_DEPTH
                features['Channel'] = TARGET_CHANNEL
                features['State'] = state_name
                features['Pulse_Index'] = i
                
                all_features.append(features)
                pulse_count += 1
            
            log_message(f"  ✓ Extracted {pulse_count} pulses from {state_name}")
        
        except Exception as e:
            log_message(f"  ✗ Error processing {filename}: {str(e)}")
            import traceback
            log_message(traceback.format_exc())
            continue
    
    # Save master table
    if all_features:
        df_master = pd.DataFrame(all_features)
        
        # Reorder columns: metadata first, then features
        metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
        feature_cols = [col for col in df_master.columns if col not in metadata_cols]
        df_master = df_master[metadata_cols + sorted(feature_cols)]
        
        output_csv = os.path.join(OUTPUT_DIR, 'master_features_3mm_Red.csv')
        df_master.to_csv(output_csv, index=False)
        
        log_message("\n" + "="*80)
        log_message("SUMMARY")
        log_message("="*80)
        log_message(f"✓ Output file: {output_csv}")
        log_message(f"✓ Total pulses: {len(df_master)}")
        log_message(f"✓ Total features: {len(feature_cols)}")
        log_message(f"✓ Columns: {len(df_master.columns)}")
        
        # Data distribution
        log_message("\nData Distribution:")
        for state in STATE_MAP.values():
            count = len(df_master[df_master['State'] == state])
            log_message(f"  {state:12s}: {count} pulses")
        
        # Delta_T quality check
        log_message("\nDelta_T Quality Check:")
        log_message(f"  Valid values: {delta_t_stats['valid_count']}")
        log_message(f"  Zero values: {delta_t_stats['zero_count']}")
        if delta_t_stats['values']:
            log_message(f"  Mean: {np.mean(delta_t_stats['values']):.4f} s")
            log_message(f"  Std: {np.std(delta_t_stats['values']):.4f} s")
            log_message(f"  Min: {np.min(delta_t_stats['values']):.4f} s")
            log_message(f"  Max: {np.max(delta_t_stats['values']):.4f} s")
            log_message(f"  Median: {np.median(delta_t_stats['values']):.4f} s")
        
        # Feature list
        log_message(f"\nExtracted Features ({len(feature_cols)}):")
        for feat in sorted(feature_cols):
            log_message(f"  - {feat}")
        
        log_message("\n" + "="*80)
        log_message("✓ FEATURE EXTRACTION COMPLETE")
        log_message("="*80)
    else:
        log_message("\n⚠️  ERROR: No pulses extracted. Check input files and columns.")

if __name__ == "__main__":
    main()


In [ ]:
"""
Diagnostic Performance Analysis: 3mm Red Channel Only (FULLY CORRECTED)
========================================================================
Calculate comprehensive diagnostic performance metrics for all features
FIXED: Proper handling of inverse relationships (AUC < 0.5)
Comparisons: Normal vs Ischaemia, Normal vs Congestion
Output: Detailed performance tables with AUC, sensitivity, specificity, etc.
"""

import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from scipy import stats
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ========================================
# CONFIGURATION
# ========================================
INPUT_FILE = 'master_features_3mm_red/master_features_3mm_Red.csv'
OUTPUT_DIR = 'diagnostic_performance_3mm_red'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, 'performance_analysis_log.txt')
ERROR_LOG = os.path.join(OUTPUT_DIR, 'error_details.txt')

def log_message(message, error=False):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f"[{timestamp}] {message}"
    print(entry)
    
    log_file = ERROR_LOG if error else LOG_FILE
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write(entry + '\n')

# ========================================
# PERFORMANCE METRICS CALCULATION
# ========================================

def calculate_auc_ci(y_true, y_pred, confidence=0.95, n_bootstraps=1000):
    """
    Calculate AUC confidence interval using bootstrap method
    
    FIXED: Convert pandas Series to numpy array to avoid index issues
    """
    if len(np.unique(y_true)) < 2:
        return np.nan, np.nan
    
    # ===== FIX: Convert to numpy arrays =====
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    try:
        auc_base = roc_auc_score(y_true, y_pred)
    except:
        return np.nan, np.nan
    
    # Bootstrap
    np.random.seed(42)
    auc_scores = []
    n_samples = len(y_true)
    
    for i in range(n_bootstraps):
        indices = np.random.choice(n_samples, n_samples, replace=True)
        y_true_boot = y_true[indices]  # ← Now works with numpy array
        y_pred_boot = y_pred[indices]
        
        if len(np.unique(y_true_boot)) < 2:
            continue
        
        try:
            auc_boot = roc_auc_score(y_true_boot, y_pred_boot)
            auc_scores.append(auc_boot)
        except:
            continue
    
    if len(auc_scores) < 10:
        return np.nan, np.nan
    
    alpha = (1 - confidence) / 2
    ci_lower = np.percentile(auc_scores, alpha * 100)
    ci_upper = np.percentile(auc_scores, (1 - alpha) * 100)
    
    return ci_lower, ci_upper

def calculate_optimal_threshold(y_true, y_pred):
    """
    Calculate optimal threshold using Youden's J statistic
    J = Sensitivity + Specificity - 1
    
    Parameters:
    -----------
    y_true : array
        True binary labels
    y_pred : array
        Predicted values (already corrected for direction)
    
    Returns:
    --------
    float : Optimal threshold value
    """
    try:
        fpr, tpr, thresholds = roc_curve(y_true, y_pred)
        j_scores = tpr - fpr
        optimal_idx = np.argmax(j_scores)
        return thresholds[optimal_idx]
    except:
        return np.median(y_pred)

def calculate_diagnostic_performance(df, feature, comparison='Ischaemia'):
    """
    Calculate comprehensive diagnostic performance metrics
    FULLY CORRECTED: Handles inverse relationships properly
    
    Parameters:
    -----------
    df : DataFrame
        Master features dataframe
    feature : str
        Feature name to analyze
    comparison : str
        'Ischaemia' or 'Congestion'
    
    Returns:
    --------
    dict : Dictionary with performance metrics, or None if calculation fails
    """
    try:
        # Filter data: Normal vs comparison state
        df_filtered = df[df['State'].isin(['Normal', comparison])].copy()
        
        # Check if feature exists and has valid data
        if feature not in df_filtered.columns:
            log_message(f"Feature {feature} not found", error=True)
            return None
        
        # Remove NaN and infinite values
        df_clean = df_filtered[[feature, 'State']].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(df_clean) < 10:
            log_message(f"Feature {feature}: insufficient data (N={len(df_clean)})", error=True)
            return None
        
        if len(df_clean['State'].unique()) < 2:
            log_message(f"Feature {feature}: only one class present", error=True)
            return None
        
        # Binary labels
        y_true = (df_clean['State'] == comparison).astype(int)
        y_pred_original = df_clean[feature].values
        
        # Check for variance
        if np.std(y_pred_original) < 1e-10:
            log_message(f"Feature {feature}: zero variance", error=True)
            return None
        
        # Sample sizes
        n_normal = int((y_true == 0).sum())
        n_disease = int((y_true == 1).sum())
        
        # ===== CRITICAL FIX: Handle inverse relationships =====
        # Calculate raw AUC
        auc_raw = roc_auc_score(y_true, y_pred_original)
        
        # Determine if relationship is inverse (AUC < 0.5)
        if auc_raw < 0.5:
            # Inverse relationship: lower values indicate disease
            auc_corrected = 1 - auc_raw
            y_pred_corrected = -y_pred_original  # Invert predictions
            is_inverse = True
        else:
            # Normal relationship: higher values indicate disease
            auc_corrected = auc_raw
            y_pred_corrected = y_pred_original
            is_inverse = False
        
        # Use corrected predictions for all downstream calculations
        ci_lower, ci_upper = calculate_auc_ci(y_true, y_pred_corrected)
        threshold = calculate_optimal_threshold(y_true, y_pred_corrected)
        
        # Binary predictions based on corrected values
        y_pred_binary = (y_pred_corrected >= threshold).astype(int)
        
        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary).ravel()
        
        # Calculate metrics
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        f1_score = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
        
        # Likelihood ratios
        if specificity < 1.0 and specificity > 0:
            lr_positive = sensitivity / (1 - specificity)
        elif specificity == 1.0 and sensitivity > 0:
            lr_positive = np.inf
        else:
            lr_positive = np.nan
        
        if specificity > 0 and sensitivity < 1.0:
            lr_negative = (1 - sensitivity) / specificity
        elif sensitivity == 1.0:
            lr_negative = 0.0
        else:
            lr_negative = np.nan
        
        # Youden's J statistic
        youden_j = sensitivity + specificity - 1
        
        # Convert threshold back to original scale if inverse
        if is_inverse:
            threshold_original = -threshold
        else:
            threshold_original = threshold
        
        return {
            'Feature': feature,
            'Comparison': comparison,
            'N_Normal': n_normal,
            'N_Disease': n_disease,
            'AUC': float(auc_corrected),
            'CI_Lower': float(ci_lower) if not np.isnan(ci_lower) else np.nan,
            'CI_Upper': float(ci_upper) if not np.isnan(ci_upper) else np.nan,
            'Optimal_Threshold': float(threshold_original),
            'Sensitivity': float(sensitivity),
            'Specificity': float(specificity),
            'PPV': float(ppv),
            'NPV': float(npv),
            'Accuracy': float(accuracy),
            'F1_Score': float(f1_score),
            'Youden_J': float(youden_j),
            'LR_Positive': float(lr_positive) if np.isfinite(lr_positive) else np.nan,
            'LR_Negative': float(lr_negative) if np.isfinite(lr_negative) else np.nan,
            'TP': int(tp),
            'TN': int(tn),
            'FP': int(fp),
            'FN': int(fn),
            'Is_Inverse': is_inverse  # Track if relationship was inverted
        }
    
    except Exception as e:
        import traceback
        error_msg = f"Feature {feature} for {comparison}: {str(e)}\n{traceback.format_exc()}"
        log_message(error_msg, error=True)
        return None

# ========================================
# PULSE COUNT SUMMARY (for manuscript reporting)
# ========================================

def summarize_pulse_counts(df, output_dir):
    """
    Report actual number of valid cardiac cycles per state.
    This is the base N before any feature-level NaN removal.
    Used for manuscript Methods section reporting.
    """
    log_message("\n" + "="*80)
    log_message("VALID CARDIAC CYCLE COUNTS (Base N per State)")
    log_message("="*80)
    
    # Total per state
    summary = (
        df.groupby('State')
        .size()
        .reset_index(name='N_Pulses')
        .sort_values('State')
    )
    
    total = summary['N_Pulses'].sum()
    
    log_message(f"\n{'State':<15} {'N Pulses':>10}")
    log_message("-" * 26)
    for _, row in summary.iterrows():
        log_message(f"  {row['State']:<13} {row['N_Pulses']:>10}")
    log_message("-" * 26)
    log_message(f"  {'Total':<13} {total:>10}")
    
    # Save CSV
    summary_file = os.path.join(output_dir, 'pulse_count_summary.csv')
    summary.to_csv(summary_file, index=False)
    log_message(f"\n✓ Saved pulse count summary: {summary_file}")
    
    # Also report per-feature valid N range (min/max after NaN removal)
    # Useful to confirm data quality across features
    metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
    feature_cols = [col for col in df.columns if col not in metadata_cols]
    
    valid_n_per_feature = (
        df[feature_cols + ['State']]
        .replace([np.inf, -np.inf], np.nan)
        .groupby('State')
        .apply(lambda g: g.drop(columns='State').notna().sum())
    )
    
    log_message("\nPer-feature valid N range (after NaN/Inf removal):")
    for state in valid_n_per_feature.index:
        row = valid_n_per_feature.loc[state]
        log_message(
            f"  {state:<13}: "
            f"min={row.min()}, max={row.max()}, "
            f"mean={row.mean():.1f}"
        )
    
    return summary

# ========================================
# MAIN ANALYSIS
# ========================================

def main():
    log_message("="*80)
    log_message("DIAGNOSTIC PERFORMANCE ANALYSIS: 3mm Red Channel (FULLY CORRECTED)")
    log_message("="*80)
    
    # Load master table
    if not os.path.exists(INPUT_FILE):
        log_message(f"ERROR: Input file not found: {INPUT_FILE}")
        return
    
    df = pd.read_csv(INPUT_FILE)
    log_message(f"✓ Loaded: {INPUT_FILE}")
    log_message(f"  Total rows: {len(df)}")
    log_message(f"  Total columns: {len(df.columns)}")
    pulse_counts = summarize_pulse_counts(df, OUTPUT_DIR)
    
    # Data distribution
    log_message("\nData Distribution:")
    for state in sorted(df['State'].unique()):
        count = len(df[df['State'] == state])
        log_message(f"  {state:12s}: {count} pulses")
    
    # Get feature columns (exclude metadata)
    metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
    feature_cols = [col for col in df.columns if col not in metadata_cols]
    
    log_message(f"\n✓ Total features to analyze: {len(feature_cols)}")
    
    # Analysis configurations
    comparisons = ['Ischaemia', 'Congestion']
    
    all_results = []
    success_count = 0
    error_count = 0
    inverse_count = 0
    
    for comparison in comparisons:
        log_message(f"\n{'='*80}")
        log_message(f"Analyzing: Normal vs {comparison}")
        log_message(f"{'='*80}")
        
        for i, feature in enumerate(feature_cols, 1):
            if i % 10 == 0:
                log_message(f"  Progress: {i}/{len(feature_cols)}")
            
            result = calculate_diagnostic_performance(df, feature, comparison)
            
            if result:
                all_results.append(result)
                success_count += 1
                if result['Is_Inverse']:
                    inverse_count += 1
            else:
                error_count += 1
    
    # Save results
    if all_results:
        df_results = pd.DataFrame(all_results)
        
        # Sort by AUC descending within each comparison
        df_results = df_results.sort_values(['Comparison', 'AUC'], 
                                            ascending=[True, False])
        
        # Save comprehensive results
        output_csv = os.path.join(OUTPUT_DIR, 'diagnostic_performance_3mm_Red_all.csv')
        df_results.to_csv(output_csv, index=False, float_format='%.6f')
        
        log_message("\n" + "="*80)
        log_message("RESULTS SUMMARY")
        log_message("="*80)
        log_message(f"✓ Output file: {output_csv}")
        log_message(f"✓ Total successful analyses: {success_count}")
        log_message(f"✓ Total failed analyses: {error_count}")
        log_message(f"✓ Inverse relationships corrected: {inverse_count}")
        log_message(f"✓ Success rate: {100*success_count/(success_count+error_count):.1f}%")
        
        # Top 10 features by comparison
        log_message("\n" + "="*80)
        log_message("TOP 10 FEATURES BY AUC")
        log_message("="*80)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            top10 = df_comp.nlargest(10, 'AUC')
            
            log_message(f"\n{comparison} (Normal vs {comparison}):")
            log_message("-"*80)
            
            for idx, row in top10.iterrows():
                inverse_marker = " [INV]" if row['Is_Inverse'] else ""
                log_message(
                    f"  {row['Feature']:30s}{inverse_marker:6s} | "
                    f"AUC: {row['AUC']:.4f} ({row['CI_Lower']:.4f}-{row['CI_Upper']:.4f}) | "
                    f"Sens: {row['Sensitivity']:.3f} | Spec: {row['Specificity']:.3f} | "
                    f"Acc: {row['Accuracy']:.3f}"
                )
        
        # Create separate files for each comparison
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            comp_file = os.path.join(OUTPUT_DIR, f'diagnostic_performance_3mm_Red_{comparison}.csv')
            df_comp.to_csv(comp_file, index=False, float_format='%.6f')
            log_message(f"\n✓ Saved {comparison} results: {comp_file} ({len(df_comp)} features)")
        
        # Create ranking tables (top 20)
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            df_rank = df_comp.nlargest(20, 'AUC')
            
            rank_file = os.path.join(OUTPUT_DIR, f'ranking_table_3mm_Red_{comparison}.csv')
            df_rank.to_csv(rank_file, index=False, float_format='%.6f')
            log_message(f"✓ Saved ranking: {rank_file}")
        
        # Statistical summary by feature category
        log_message("\n" + "="*80)
        log_message("FEATURE CATEGORY ANALYSIS")
        log_message("="*80)
        
        def categorize_feature(name):
            """Categorize features by type"""
            name_lower = name.lower()
            if any(x in name_lower for x in ['pi', 'amplitude', 'ac_', 'dc_']):
                return 'Intensity'
            elif any(x in name_lower for x in ['auc', 'area', 'datum']):
                return 'Area'
            elif any(x in name_lower for x in ['time', 'width', 'pw50', 'delta_t']):
                return 'Time'
            elif any(x in name_lower for x in ['slope', 'upslope', 'downslope', 'length']):
                return 'Slope'
            elif any(x in name_lower for x in ['sdppg', '_a', '_b', '_c', '_d', '_e', 'dnp']):
                return 'SDPPG'
            elif any(x in name_lower for x in ['skewness', 'kurtosis', 'variance', 'rwi', 'ipar']):
                return 'Shape & Stats'
            else:
                return 'Other'
        
        df_results['Category'] = df_results['Feature'].apply(categorize_feature)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            
            log_message(f"\n{comparison}:")
            log_message("-"*80)
            
            for category in sorted(df_comp['Category'].unique()):
                df_cat = df_comp[df_comp['Category'] == category]
                mean_auc = df_cat['AUC'].mean()
                median_auc = df_cat['AUC'].median()
                max_auc = df_cat['AUC'].max()
                n_inverse = df_cat['Is_Inverse'].sum()
                
                log_message(
                    f"  {category:15s}: "
                    f"N={len(df_cat):2d} | "
                    f"Mean AUC={mean_auc:.4f} | "
                    f"Median AUC={median_auc:.4f} | "
                    f"Max AUC={max_auc:.4f} | "
                    f"Inverse={n_inverse}"
                )
        
        # Inverse relationship summary
        log_message("\n" + "="*80)
        log_message("INVERSE RELATIONSHIP SUMMARY")
        log_message("="*80)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            df_inv = df_comp[df_comp['Is_Inverse'] == True]
            
            if len(df_inv) > 0:
                log_message(f"\n{comparison} - Features with inverse relationships ({len(df_inv)}):")
                for idx, row in df_inv.iterrows():
                    log_message(f"  {row['Feature']:30s} | AUC: {row['AUC']:.4f}")
        
        log_message("\n" + "="*80)
        log_message("✓ ANALYSIS COMPLETE")
        log_message(f"✓ Check {ERROR_LOG} for detailed error messages (if any)")
        log_message("="*80)
    
    else:
        log_message("\n⚠️  ERROR: No valid results generated. Check error log for details.")

if __name__ == "__main__":
    main()


In [ ]:
"""
Master Feature Extraction: 3mm IR Channel Only
===============================================
Extracts comprehensive pulse-level features from 3mm depth IR channel
Following the same pipeline as 3mm Red for consistency
Output format designed for easy integration with other wavelengths/depths
"""

import pandas as pd
import numpy as np
from scipy import signal, stats
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ========================================
# CONFIGURATION
# ========================================
FS = 2000
TARGET_DEPTH = 3
TARGET_CHANNEL = 'IR'
RAW_COLUMN = 'ppgA_IR_raw'
PROCESSED_COLUMN = 'IR_processed'

STATE_MAP = {1: 'Normal', 2: 'Ischaemia', 3: 'Congestion'}

OUTPUT_DIR = 'master_features_3mm_ir'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, 'processing_log.txt')

def log_message(message):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f"[{timestamp}] {message}"
    print(entry)
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(entry + '\n')

# ========================================
# HELPER FUNCTIONS
# ========================================

def calculate_delta_t_robust(vpg, t, fs):
    """
    Robust Delta_T calculation using multiple methods
    
    Delta_T = Time interval between:
    1. Maximum acceleration point (VPG peak)
    2. Maximum deceleration point (first significant VPG trough after peak)
    """
    # Method 1: Peak-to-trough approach
    vpg_max_idx = np.argmax(vpg)
    
    search_start = vpg_max_idx + int(0.02 * fs)
    search_end = min(vpg_max_idx + int(0.4 * fs), len(vpg))
    
    if search_end > search_start and search_start < len(vpg):
        vpg_search_region = vpg[search_start:search_end]
        
        troughs, properties = signal.find_peaks(
            -vpg_search_region, 
            prominence=0.03 * np.max(np.abs(vpg)),
            distance=int(0.02 * fs)
        )
        
        if len(troughs) > 0:
            first_trough_idx = search_start + troughs[0]
            delta_t = (first_trough_idx - vpg_max_idx) / fs
            
            if 0.05 <= delta_t <= 0.4:
                return delta_t
    
    # Method 2: Zero-crossing approach
    zero_crossings = np.where(np.diff(np.sign(vpg)))[0]
    
    if len(zero_crossings) >= 2:
        zc_before_peak = zero_crossings[zero_crossings < vpg_max_idx]
        zc_after_peak = zero_crossings[zero_crossings > vpg_max_idx]
        
        if len(zc_before_peak) > 0 and len(zc_after_peak) > 0:
            zc1 = zc_before_peak[-1]
            zc2 = zc_after_peak[0]
            delta_t = (zc2 - zc1) / fs
            
            if 0.05 <= delta_t <= 0.4:
                return delta_t
    
    # Method 3: Threshold-based approach
    threshold = 0.1 * vpg[vpg_max_idx]
    after_peak = vpg[vpg_max_idx:]
    
    if len(after_peak) > 10:
        below_threshold = np.where(after_peak < threshold)[0]
        if len(below_threshold) > 0:
            delta_t = below_threshold[0] / fs
            
            if 0.05 <= delta_t <= 0.4:
                return delta_t
    
    # Method 4: Default estimate
    pulse_duration = len(vpg) / fs
    delta_t = 0.2 * pulse_duration
    delta_t = np.clip(delta_t, 0.05, 0.4)
    
    return delta_t

# ========================================
# FEATURE EXTRACTION
# ========================================

def extract_comprehensive_features(seg_raw, seg_processed, fs=2000):
    """
    Extract all pulse-level features from single pulse
    """
    f = {}
    
    # ===== PREPROCESSING COMPONENTS =====
    dc_component = np.mean(seg_raw)
    f['DC_Component'] = dc_component
    
    ac_component = np.max(seg_processed) - np.min(seg_processed)
    f['AC_Component'] = ac_component
    
    f['PI'] = (ac_component / (dc_component + 1e-10)) * 100
    
    # ===== NORMALIZATION =====
    seg_min = np.min(seg_processed)
    seg_max = np.max(seg_processed)
    seg_norm = (seg_processed - seg_min) / (seg_max - seg_min + 1e-10)
    
    t = np.arange(len(seg_norm)) / fs
    total_len = len(seg_norm)
    s_peak_idx = np.argmax(seg_norm)
    
    # ===== INTENSITY / AMPLITUDE =====
    f['Amplitude'] = ac_component
    f['Max_Start_Datum_Diff'] = np.max(seg_processed[:total_len//4]) if total_len >= 4 else 0
    
    # ===== AREA FEATURES =====
    baseline = np.linspace(seg_processed[0], seg_processed[-1], total_len)
    sig_bc = seg_processed - baseline
    
    f['AUC'] = np.trapz(sig_bc)
    f['S_AUC'] = np.trapz(sig_bc[:s_peak_idx+1])
    f['D_AUC'] = np.trapz(sig_bc[s_peak_idx:])
    f['AUC_Ratio'] = f['S_AUC'] / (f['D_AUC'] + 1e-10)
    
    q = total_len // 4
    if q > 0:
        f['Start_Datum_Area'] = np.trapz(sig_bc[:q])
        f['End_Datum_Area'] = np.trapz(sig_bc[-q:])
        f['Datum_Area_Ratio'] = f['Start_Datum_Area'] / (f['End_Datum_Area'] + 1e-10)
    else:
        f['Start_Datum_Area'] = 0
        f['End_Datum_Area'] = 0
        f['Datum_Area_Ratio'] = 0
    
    # ===== TIME / WIDTH =====
    f['Rise_Time'] = s_peak_idx / fs
    f['Fall_Time'] = (total_len - s_peak_idx) / fs
    f['Pulse_Width'] = total_len / fs
    f['Rise_Decay_Time_Ratio'] = f['Rise_Time'] / (f['Fall_Time'] + 1e-10)
    
    above_50 = np.where(seg_norm >= 0.5)[0]
    f['PW50'] = (above_50[-1] - above_50[0]) / fs if len(above_50) > 0 else 0
    
    notch_idx = int(s_peak_idx + 0.3 * (total_len - s_peak_idx))
    f['Systolic_Width'] = notch_idx / fs
    f['Diastolic_Width'] = (total_len - notch_idx) / fs
    f['Width_Ratio'] = f['Systolic_Width'] / (f['Diastolic_Width'] + 1e-10)
    
    # ===== SLOPE =====
    vpg = np.gradient(seg_norm, t)
    
    f['Upslope'] = np.max(vpg)
    f['Downslope'] = np.min(vpg)
    f['Onset_End_Slope'] = (seg_norm[-1] - seg_norm[0]) / (t[-1] - t[0] + 1e-10)
    f['Slope_Ratio'] = f['Upslope'] / (abs(f['Downslope']) + 1e-10)
    
    f['Delta_T'] = calculate_delta_t_robust(vpg, t, fs)
    
    dx = np.diff(t)
    dy = np.diff(seg_norm)
    arc_increments = np.sqrt(dx**2 + dy**2)
    total_arc = np.sum(arc_increments)
    upslope_arc = np.sum(arc_increments[:s_peak_idx]) if s_peak_idx > 0 else 0
    downslope_arc = np.sum(arc_increments[s_peak_idx:])
    
    f['Upslope_Length'] = upslope_arc
    f['Downslope_Length'] = downslope_arc
    f['Slope_Length_Ratio'] = upslope_arc / (downslope_arc + 1e-10)
    f['Upslope_Length_Ratio'] = upslope_arc / (total_arc + 1e-10)
    f['Downslope_Length_Ratio'] = downslope_arc / (total_arc + 1e-10)
    f['Length_Height_Ratio'] = total_arc / (seg_norm[s_peak_idx] + 1e-10)
    
    # ===== SDPPG =====
    apg = np.gradient(vpg, t)
    
    a_wave_idx_max = min(int(0.2 * fs), len(apg))
    a_wave = np.max(apg[:a_wave_idx_max]) if a_wave_idx_max > 0 else 1e-10
    
    def safe_apg_ratio(start_frac, end_frac, mode='max'):
        start_idx = int(start_frac * fs)
        end_idx = int(end_frac * fs)
        if start_idx >= len(apg) or end_idx > len(apg) or start_idx >= end_idx:
            return 0
        segment = apg[start_idx:end_idx]
        if len(segment) == 0:
            return 0
        val = np.max(segment) if mode == 'max' else np.min(segment)
        return val / (a_wave + 1e-10)
    
    f['SDPPG_b_a'] = safe_apg_ratio(0.05, 0.15, 'min')
    f['SDPPG_c_a'] = safe_apg_ratio(0.15, 0.25, 'max')
    f['SDPPG_d_a'] = safe_apg_ratio(0.25, 0.35, 'min')
    f['SDPPG_e_a'] = safe_apg_ratio(0.35, 0.50, 'max')
    
    search_start = s_peak_idx + int(0.1 * fs)
    if search_start < len(seg_norm) - 10:
        f['DNP'] = seg_norm[notch_idx] if notch_idx < len(seg_norm) else 0
    else:
        f['DNP'] = 0
    
    # ===== SHAPE & STATS =====
    f['Skewness'] = stats.skew(seg_norm)
    f['Kurtosis'] = stats.kurtosis(seg_norm)
    f['Variance'] = np.var(seg_norm)
    
    if search_start < len(seg_norm) - 10:
        diastolic_seg = seg_norm[search_start:]
        d_peak = np.max(diastolic_seg)
        f['RWI'] = d_peak / (seg_norm[s_peak_idx] + 1e-10)
    else:
        f['RWI'] = 0
    
    inflection_idx = int(0.3 * total_len)
    if inflection_idx > 0 and inflection_idx < len(sig_bc):
        area_before = np.trapz(sig_bc[:inflection_idx])
        area_after = np.trapz(sig_bc[inflection_idx:])
        f['IPAR'] = area_before / (area_after + 1e-10)
    else:
        f['IPAR'] = 0
    
    return f

# ========================================
# MAIN PROCESSING
# ========================================

def main():
    log_message("="*80)
    log_message(f"MASTER FEATURE EXTRACTION: {TARGET_DEPTH}mm {TARGET_CHANNEL} Channel")
    log_message("="*80)
    
    all_features = []
    delta_t_stats = {'zero_count': 0, 'valid_count': 0, 'values': []}
    
    for state_code, state_name in STATE_MAP.items():
        filename = f'{TARGET_DEPTH}mm_processed_data{state_code}.csv'
        
        if not os.path.exists(filename):
            log_message(f"⚠️  File not found: {filename}")
            continue
        
        try:
            df = pd.read_csv(filename)
            log_message(f"📂 Processing: {filename} ({state_name})")
            
            if RAW_COLUMN not in df.columns:
                log_message(f"  ⚠️  Raw column '{RAW_COLUMN}' not found")
                continue
            if PROCESSED_COLUMN not in df.columns:
                log_message(f"  ⚠️  Processed column '{PROCESSED_COLUMN}' not found")
                continue
            
            max_samples = int(30 * FS)
            raw_signal = df[RAW_COLUMN].values[:max_samples]
            processed_signal = df[PROCESSED_COLUMN].values[:max_samples]
            
            troughs, _ = signal.find_peaks(-processed_signal, distance=int(FS * 0.6))
            
            pulse_count = 0
            for i in range(len(troughs) - 1):
                idx1, idx2 = troughs[i], troughs[i+1]
                
                pulse_duration = (idx2 - idx1) / FS
                if pulse_duration < 0.4 or pulse_duration > 1.5:
                    continue
                
                seg_raw = raw_signal[idx1:idx2]
                seg_processed = processed_signal[idx1:idx2]
                
                features = extract_comprehensive_features(seg_raw, seg_processed, FS)
                
                if features['Delta_T'] == 0:
                    delta_t_stats['zero_count'] += 1
                else:
                    delta_t_stats['valid_count'] += 1
                    delta_t_stats['values'].append(features['Delta_T'])
                
                features['Depth'] = TARGET_DEPTH
                features['Channel'] = TARGET_CHANNEL
                features['State'] = state_name
                features['Pulse_Index'] = i
                
                all_features.append(features)
                pulse_count += 1
            
            log_message(f"  ✓ Extracted {pulse_count} pulses from {state_name}")
        
        except Exception as e:
            log_message(f"  ✗ Error processing {filename}: {str(e)}")
            import traceback
            log_message(traceback.format_exc())
            continue
    
    if all_features:
        df_master = pd.DataFrame(all_features)
        
        metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
        feature_cols = [col for col in df_master.columns if col not in metadata_cols]
        df_master = df_master[metadata_cols + sorted(feature_cols)]
        
        output_csv = os.path.join(OUTPUT_DIR, 'master_features_3mm_IR.csv')
        df_master.to_csv(output_csv, index=False)
        
        log_message("\n" + "="*80)
        log_message("SUMMARY")
        log_message("="*80)
        log_message(f"✓ Output file: {output_csv}")
        log_message(f"✓ Total pulses: {len(df_master)}")
        log_message(f"✓ Total features: {len(feature_cols)}")
        
        log_message("\nData Distribution:")
        for state in STATE_MAP.values():
            count = len(df_master[df_master['State'] == state])
            log_message(f"  {state:12s}: {count} pulses")
        
        log_message("\nDelta_T Quality Check:")
        log_message(f"  Valid values: {delta_t_stats['valid_count']}")
        log_message(f"  Zero values: {delta_t_stats['zero_count']}")
        if delta_t_stats['values']:
            log_message(f"  Mean: {np.mean(delta_t_stats['values']):.4f} s")
            log_message(f"  Std: {np.std(delta_t_stats['values']):.4f} s")
            log_message(f"  Min: {np.min(delta_t_stats['values']):.4f} s")
            log_message(f"  Max: {np.max(delta_t_stats['values']):.4f} s")
            log_message(f"  Median: {np.median(delta_t_stats['values']):.4f} s")
        
        log_message("\n" + "="*80)
        log_message("✓ FEATURE EXTRACTION COMPLETE")
        log_message("="*80)
    else:
        log_message("\n⚠️  ERROR: No pulses extracted.")

if __name__ == "__main__":
    main()



In [ ]:
"""
Diagnostic Performance Analysis: 3mm IR Channel Only (FULLY CORRECTED)
========================================================================
Calculate comprehensive diagnostic performance metrics for all features
FIXED: Proper handling of inverse relationships (AUC < 0.5)
Comparisons: Normal vs Ischaemia, Normal vs Congestion
Output: Detailed performance tables with AUC, sensitivity, specificity, etc.
"""

import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from scipy import stats
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ========================================
# CONFIGURATION
# ========================================
INPUT_FILE = 'master_features_3mm_ir/master_features_3mm_IR.csv'
OUTPUT_DIR = 'diagnostic_performance_3mm_ir'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, 'performance_analysis_log.txt')
ERROR_LOG = os.path.join(OUTPUT_DIR, 'error_details.txt')

def log_message(message, error=False):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f"[{timestamp}] {message}"
    print(entry)
    
    log_file = ERROR_LOG if error else LOG_FILE
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write(entry + '\n')

# ========================================
# PERFORMANCE METRICS CALCULATION
# ========================================

def calculate_auc_ci(y_true, y_pred, confidence=0.95, n_bootstraps=1000):
    """
    Calculate AUC confidence interval using bootstrap method
    
    FIXED: Convert pandas Series to numpy array to avoid index issues
    """
    if len(np.unique(y_true)) < 2:
        return np.nan, np.nan
    
    # ===== FIX: Convert to numpy arrays =====
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    try:
        auc_base = roc_auc_score(y_true, y_pred)
    except:
        return np.nan, np.nan
    
    # Bootstrap
    np.random.seed(42)
    auc_scores = []
    n_samples = len(y_true)
    
    for i in range(n_bootstraps):
        indices = np.random.choice(n_samples, n_samples, replace=True)
        y_true_boot = y_true[indices]  # ← Now works with numpy array
        y_pred_boot = y_pred[indices]
        
        if len(np.unique(y_true_boot)) < 2:
            continue
        
        try:
            auc_boot = roc_auc_score(y_true_boot, y_pred_boot)
            auc_scores.append(auc_boot)
        except:
            continue
    
    if len(auc_scores) < 10:
        return np.nan, np.nan
    
    alpha = (1 - confidence) / 2
    ci_lower = np.percentile(auc_scores, alpha * 100)
    ci_upper = np.percentile(auc_scores, (1 - alpha) * 100)
    
    return ci_lower, ci_upper

def calculate_optimal_threshold(y_true, y_pred):
    """
    Calculate optimal threshold using Youden's J statistic
    J = Sensitivity + Specificity - 1
    
    Parameters:
    -----------
    y_true : array
        True binary labels
    y_pred : array
        Predicted values (already corrected for direction)
    
    Returns:
    --------
    float : Optimal threshold value
    """
    try:
        fpr, tpr, thresholds = roc_curve(y_true, y_pred)
        j_scores = tpr - fpr
        optimal_idx = np.argmax(j_scores)
        return thresholds[optimal_idx]
    except:
        return np.median(y_pred)

def calculate_diagnostic_performance(df, feature, comparison='Ischaemia'):
    """
    Calculate comprehensive diagnostic performance metrics
    FULLY CORRECTED: Handles inverse relationships properly
    
    Parameters:
    -----------
    df : DataFrame
        Master features dataframe
    feature : str
        Feature name to analyze
    comparison : str
        'Ischaemia' or 'Congestion'
    
    Returns:
    --------
    dict : Dictionary with performance metrics, or None if calculation fails
    """
    try:
        # Filter data: Normal vs comparison state
        df_filtered = df[df['State'].isin(['Normal', comparison])].copy()
        
        # Check if feature exists and has valid data
        if feature not in df_filtered.columns:
            log_message(f"Feature {feature} not found", error=True)
            return None
        
        # Remove NaN and infinite values
        df_clean = df_filtered[[feature, 'State']].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(df_clean) < 10:
            log_message(f"Feature {feature}: insufficient data (N={len(df_clean)})", error=True)
            return None
        
        if len(df_clean['State'].unique()) < 2:
            log_message(f"Feature {feature}: only one class present", error=True)
            return None
        
        # Binary labels
        y_true = (df_clean['State'] == comparison).astype(int)
        y_pred_original = df_clean[feature].values
        
        # Check for variance
        if np.std(y_pred_original) < 1e-10:
            log_message(f"Feature {feature}: zero variance", error=True)
            return None
        
        # Sample sizes
        n_normal = int((y_true == 0).sum())
        n_disease = int((y_true == 1).sum())
        
        # ===== CRITICAL FIX: Handle inverse relationships =====
        # Calculate raw AUC
        auc_raw = roc_auc_score(y_true, y_pred_original)
        
        # Determine if relationship is inverse (AUC < 0.5)
        if auc_raw < 0.5:
            # Inverse relationship: lower values indicate disease
            auc_corrected = 1 - auc_raw
            y_pred_corrected = -y_pred_original  # Invert predictions
            is_inverse = True
        else:
            # Normal relationship: higher values indicate disease
            auc_corrected = auc_raw
            y_pred_corrected = y_pred_original
            is_inverse = False
        
        # Use corrected predictions for all downstream calculations
        ci_lower, ci_upper = calculate_auc_ci(y_true, y_pred_corrected)
        threshold = calculate_optimal_threshold(y_true, y_pred_corrected)
        
        # Binary predictions based on corrected values
        y_pred_binary = (y_pred_corrected >= threshold).astype(int)
        
        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary).ravel()
        
        # Calculate metrics
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        f1_score = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
        
        # Likelihood ratios
        if specificity < 1.0 and specificity > 0:
            lr_positive = sensitivity / (1 - specificity)
        elif specificity == 1.0 and sensitivity > 0:
            lr_positive = np.inf
        else:
            lr_positive = np.nan
        
        if specificity > 0 and sensitivity < 1.0:
            lr_negative = (1 - sensitivity) / specificity
        elif sensitivity == 1.0:
            lr_negative = 0.0
        else:
            lr_negative = np.nan
        
        # Youden's J statistic
        youden_j = sensitivity + specificity - 1
        
        # Convert threshold back to original scale if inverse
        if is_inverse:
            threshold_original = -threshold
        else:
            threshold_original = threshold
        
        return {
            'Feature': feature,
            'Comparison': comparison,
            'N_Normal': n_normal,
            'N_Disease': n_disease,
            'AUC': float(auc_corrected),
            'CI_Lower': float(ci_lower) if not np.isnan(ci_lower) else np.nan,
            'CI_Upper': float(ci_upper) if not np.isnan(ci_upper) else np.nan,
            'Optimal_Threshold': float(threshold_original),
            'Sensitivity': float(sensitivity),
            'Specificity': float(specificity),
            'PPV': float(ppv),
            'NPV': float(npv),
            'Accuracy': float(accuracy),
            'F1_Score': float(f1_score),
            'Youden_J': float(youden_j),
            'LR_Positive': float(lr_positive) if np.isfinite(lr_positive) else np.nan,
            'LR_Negative': float(lr_negative) if np.isfinite(lr_negative) else np.nan,
            'TP': int(tp),
            'TN': int(tn),
            'FP': int(fp),
            'FN': int(fn),
            'Is_Inverse': is_inverse  # Track if relationship was inverted
        }
    
    except Exception as e:
        import traceback
        error_msg = f"Feature {feature} for {comparison}: {str(e)}\n{traceback.format_exc()}"
        log_message(error_msg, error=True)
        return None

# ========================================
# PULSE COUNT SUMMARY (for manuscript reporting)
# ========================================

def summarize_pulse_counts(df, output_dir):
    """
    Report actual number of valid cardiac cycles per state.
    This is the base N before any feature-level NaN removal.
    Used for manuscript Methods section reporting.
    """
    log_message("\n" + "="*80)
    log_message("VALID CARDIAC CYCLE COUNTS (Base N per State)")
    log_message("="*80)
    
    # Total per state
    summary = (
        df.groupby('State')
        .size()
        .reset_index(name='N_Pulses')
        .sort_values('State')
    )
    
    total = summary['N_Pulses'].sum()
    
    log_message(f"\n{'State':<15} {'N Pulses':>10}")
    log_message("-" * 26)
    for _, row in summary.iterrows():
        log_message(f"  {row['State']:<13} {row['N_Pulses']:>10}")
    log_message("-" * 26)
    log_message(f"  {'Total':<13} {total:>10}")
    
    # Save CSV
    summary_file = os.path.join(output_dir, 'pulse_count_summary.csv')
    summary.to_csv(summary_file, index=False)
    log_message(f"\n✓ Saved pulse count summary: {summary_file}")
    
    # Also report per-feature valid N range (min/max after NaN removal)
    # Useful to confirm data quality across features
    metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
    feature_cols = [col for col in df.columns if col not in metadata_cols]
    
    valid_n_per_feature = (
        df[feature_cols + ['State']]
        .replace([np.inf, -np.inf], np.nan)
        .groupby('State')
        .apply(lambda g: g.drop(columns='State').notna().sum())
    )
    
    log_message("\nPer-feature valid N range (after NaN/Inf removal):")
    for state in valid_n_per_feature.index:
        row = valid_n_per_feature.loc[state]
        log_message(
            f"  {state:<13}: "
            f"min={row.min()}, max={row.max()}, "
            f"mean={row.mean():.1f}"
        )
    
    return summary

# ========================================
# MAIN ANALYSIS
# ========================================

def main():
    log_message("="*80)
    log_message("DIAGNOSTIC PERFORMANCE ANALYSIS: 9mm IR Channel (FULLY CORRECTED)")
    log_message("="*80)
    
    # Load master table
    if not os.path.exists(INPUT_FILE):
        log_message(f"ERROR: Input file not found: {INPUT_FILE}")
        return
    
    df = pd.read_csv(INPUT_FILE)
    log_message(f"✓ Loaded: {INPUT_FILE}")
    log_message(f"  Total rows: {len(df)}")
    log_message(f"  Total columns: {len(df.columns)}")
    pulse_counts = summarize_pulse_counts(df, OUTPUT_DIR)
    
    # Data distribution
    log_message("\nData Distribution:")
    for state in sorted(df['State'].unique()):
        count = len(df[df['State'] == state])
        log_message(f"  {state:12s}: {count} pulses")
    
    # Get feature columns (exclude metadata)
    metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
    feature_cols = [col for col in df.columns if col not in metadata_cols]
    
    log_message(f"\n✓ Total features to analyze: {len(feature_cols)}")
    
    # Analysis configurations
    comparisons = ['Ischaemia', 'Congestion']
    
    all_results = []
    success_count = 0
    error_count = 0
    inverse_count = 0
    
    for comparison in comparisons:
        log_message(f"\n{'='*80}")
        log_message(f"Analyzing: Normal vs {comparison}")
        log_message(f"{'='*80}")
        
        for i, feature in enumerate(feature_cols, 1):
            if i % 10 == 0:
                log_message(f"  Progress: {i}/{len(feature_cols)}")
            
            result = calculate_diagnostic_performance(df, feature, comparison)
            
            if result:
                all_results.append(result)
                success_count += 1
                if result['Is_Inverse']:
                    inverse_count += 1
            else:
                error_count += 1
    
    # Save results
    if all_results:
        df_results = pd.DataFrame(all_results)
        
        # Sort by AUC descending within each comparison
        df_results = df_results.sort_values(['Comparison', 'AUC'], 
                                            ascending=[True, False])
        
        # Save comprehensive results
        output_csv = os.path.join(OUTPUT_DIR, 'diagnostic_performance_9mm_IR_all.csv')
        df_results.to_csv(output_csv, index=False, float_format='%.6f')
        
        log_message("\n" + "="*80)
        log_message("RESULTS SUMMARY")
        log_message("="*80)
        log_message(f"✓ Output file: {output_csv}")
        log_message(f"✓ Total successful analyses: {success_count}")
        log_message(f"✓ Total failed analyses: {error_count}")
        log_message(f"✓ Inverse relationships corrected: {inverse_count}")
        log_message(f"✓ Success rate: {100*success_count/(success_count+error_count):.1f}%")
        
        # Top 10 features by comparison
        log_message("\n" + "="*80)
        log_message("TOP 10 FEATURES BY AUC")
        log_message("="*80)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            top10 = df_comp.nlargest(10, 'AUC')
            
            log_message(f"\n{comparison} (Normal vs {comparison}):")
            log_message("-"*80)
            
            for idx, row in top10.iterrows():
                inverse_marker = " [INV]" if row['Is_Inverse'] else ""
                log_message(
                    f"  {row['Feature']:30s}{inverse_marker:6s} | "
                    f"AUC: {row['AUC']:.4f} ({row['CI_Lower']:.4f}-{row['CI_Upper']:.4f}) | "
                    f"Sens: {row['Sensitivity']:.3f} | Spec: {row['Specificity']:.3f} | "
                    f"Acc: {row['Accuracy']:.3f}"
                )
        
        # Create separate files for each comparison
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            comp_file = os.path.join(OUTPUT_DIR, f'diagnostic_performance_9mm_IR_{comparison}.csv')
            df_comp.to_csv(comp_file, index=False, float_format='%.6f')
            log_message(f"\n✓ Saved {comparison} results: {comp_file} ({len(df_comp)} features)")
        
        # Create ranking tables (top 20)
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            df_rank = df_comp.nlargest(20, 'AUC')
            
            rank_file = os.path.join(OUTPUT_DIR, f'ranking_table_9mm_IR_{comparison}.csv')
            df_rank.to_csv(rank_file, index=False, float_format='%.6f')
            log_message(f"✓ Saved ranking: {rank_file}")
        
        # Statistical summary by feature category
        log_message("\n" + "="*80)
        log_message("FEATURE CATEGORY ANALYSIS")
        log_message("="*80)
        
        def categorize_feature(name):
            """Categorize features by type"""
            name_lower = name.lower()
            if any(x in name_lower for x in ['pi', 'amplitude', 'ac_', 'dc_']):
                return 'Intensity'
            elif any(x in name_lower for x in ['auc', 'area', 'datum']):
                return 'Area'
            elif any(x in name_lower for x in ['time', 'width', 'pw50', 'delta_t']):
                return 'Time'
            elif any(x in name_lower for x in ['slope', 'upslope', 'downslope', 'length']):
                return 'Slope'
            elif any(x in name_lower for x in ['sdppg', '_a', '_b', '_c', '_d', '_e', 'dnp']):
                return 'SDPPG'
            elif any(x in name_lower for x in ['skewness', 'kurtosis', 'variance', 'rwi', 'ipar']):
                return 'Shape & Stats'
            else:
                return 'Other'
        
        df_results['Category'] = df_results['Feature'].apply(categorize_feature)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            
            log_message(f"\n{comparison}:")
            log_message("-"*80)
            
            for category in sorted(df_comp['Category'].unique()):
                df_cat = df_comp[df_comp['Category'] == category]
                mean_auc = df_cat['AUC'].mean()
                median_auc = df_cat['AUC'].median()
                max_auc = df_cat['AUC'].max()
                n_inverse = df_cat['Is_Inverse'].sum()
                
                log_message(
                    f"  {category:15s}: "
                    f"N={len(df_cat):2d} | "
                    f"Mean AUC={mean_auc:.4f} | "
                    f"Median AUC={median_auc:.4f} | "
                    f"Max AUC={max_auc:.4f} | "
                    f"Inverse={n_inverse}"
                )
        
        # Inverse relationship summary
        log_message("\n" + "="*80)
        log_message("INVERSE RELATIONSHIP SUMMARY")
        log_message("="*80)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            df_inv = df_comp[df_comp['Is_Inverse'] == True]
            
            if len(df_inv) > 0:
                log_message(f"\n{comparison} - Features with inverse relationships ({len(df_inv)}):")
                for idx, row in df_inv.iterrows():
                    log_message(f"  {row['Feature']:30s} | AUC: {row['AUC']:.4f}")
        
        log_message("\n" + "="*80)
        log_message("✓ ANALYSIS COMPLETE")
        log_message(f"✓ Check {ERROR_LOG} for detailed error messages (if any)")
        log_message("="*80)
    
    else:
        log_message("\n⚠️  ERROR: No valid results generated. Check error log for details.")

if __name__ == "__main__":
    main()


In [ ]:
"""
Master Feature Extraction: 9mm IR Channel Only
===============================================
Extracts comprehensive pulse-level features from 9mm depth IR channel
Following the same pipeline as 9mm IR for consistency
Output format designed for easy integration with other wavelengths/depths
"""

import pandas as pd
import numpy as np
from scipy import signal, stats
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ========================================
# CONFIGURATION
# ========================================
FS = 2000
TARGET_DEPTH = 9
TARGET_CHANNEL = 'IR'
RAW_COLUMN = 'ppgA_IR_raw'
PROCESSED_COLUMN = 'IR_processed'

STATE_MAP = {1: 'Normal', 2: 'Ischaemia', 3: 'Congestion'}

OUTPUT_DIR = 'master_features_9mm_ir'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, 'processing_log.txt')

def log_message(message):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f"[{timestamp}] {message}"
    print(entry)
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(entry + '\n')

# ========================================
# HELPER FUNCTIONS
# ========================================

def calculate_delta_t_robust(vpg, t, fs):
    """
    Robust Delta_T calculation using multiple methods
    
    Delta_T = Time interval between:
    1. Maximum acceleration point (VPG peak)
    2. Maximum deceleration point (first significant VPG trough after peak)
    """
    # Method 1: Peak-to-trough approach
    vpg_max_idx = np.argmax(vpg)
    
    search_start = vpg_max_idx + int(0.02 * fs)
    search_end = min(vpg_max_idx + int(0.4 * fs), len(vpg))
    
    if search_end > search_start and search_start < len(vpg):
        vpg_search_region = vpg[search_start:search_end]
        
        troughs, properties = signal.find_peaks(
            -vpg_search_region, 
            prominence=0.03 * np.max(np.abs(vpg)),
            distance=int(0.02 * fs)
        )
        
        if len(troughs) > 0:
            first_trough_idx = search_start + troughs[0]
            delta_t = (first_trough_idx - vpg_max_idx) / fs
            
            if 0.05 <= delta_t <= 0.4:
                return delta_t
    
    # Method 2: Zero-crossing approach
    zero_crossings = np.where(np.diff(np.sign(vpg)))[0]
    
    if len(zero_crossings) >= 2:
        zc_before_peak = zero_crossings[zero_crossings < vpg_max_idx]
        zc_after_peak = zero_crossings[zero_crossings > vpg_max_idx]
        
        if len(zc_before_peak) > 0 and len(zc_after_peak) > 0:
            zc1 = zc_before_peak[-1]
            zc2 = zc_after_peak[0]
            delta_t = (zc2 - zc1) / fs
            
            if 0.05 <= delta_t <= 0.4:
                return delta_t
    
    # Method 3: Threshold-based approach
    threshold = 0.1 * vpg[vpg_max_idx]
    after_peak = vpg[vpg_max_idx:]
    
    if len(after_peak) > 10:
        below_threshold = np.where(after_peak < threshold)[0]
        if len(below_threshold) > 0:
            delta_t = below_threshold[0] / fs
            
            if 0.05 <= delta_t <= 0.4:
                return delta_t
    
    # Method 4: Default estimate
    pulse_duration = len(vpg) / fs
    delta_t = 0.2 * pulse_duration
    delta_t = np.clip(delta_t, 0.05, 0.4)
    
    return delta_t

# ========================================
# FEATURE EXTRACTION
# ========================================

def extract_comprehensive_features(seg_raw, seg_processed, fs=2000):
    """
    Extract all pulse-level features from single pulse
    """
    f = {}
    
    # ===== PREPROCESSING COMPONENTS =====
    dc_component = np.mean(seg_raw)
    f['DC_Component'] = dc_component
    
    ac_component = np.max(seg_processed) - np.min(seg_processed)
    f['AC_Component'] = ac_component
    
    f['PI'] = (ac_component / (dc_component + 1e-10)) * 100
    
    # ===== NORMALIZATION =====
    seg_min = np.min(seg_processed)
    seg_max = np.max(seg_processed)
    seg_norm = (seg_processed - seg_min) / (seg_max - seg_min + 1e-10)
    
    t = np.arange(len(seg_norm)) / fs
    total_len = len(seg_norm)
    s_peak_idx = np.argmax(seg_norm)
    
    # ===== INTENSITY / AMPLITUDE =====
    f['Amplitude'] = ac_component
    f['Max_Start_Datum_Diff'] = np.max(seg_processed[:total_len//4]) if total_len >= 4 else 0
    
    # ===== AREA FEATURES =====
    baseline = np.linspace(seg_processed[0], seg_processed[-1], total_len)
    sig_bc = seg_processed - baseline
    
    f['AUC'] = np.trapz(sig_bc)
    f['S_AUC'] = np.trapz(sig_bc[:s_peak_idx+1])
    f['D_AUC'] = np.trapz(sig_bc[s_peak_idx:])
    f['AUC_Ratio'] = f['S_AUC'] / (f['D_AUC'] + 1e-10)
    
    q = total_len // 4
    if q > 0:
        f['Start_Datum_Area'] = np.trapz(sig_bc[:q])
        f['End_Datum_Area'] = np.trapz(sig_bc[-q:])
        f['Datum_Area_Ratio'] = f['Start_Datum_Area'] / (f['End_Datum_Area'] + 1e-10)
    else:
        f['Start_Datum_Area'] = 0
        f['End_Datum_Area'] = 0
        f['Datum_Area_Ratio'] = 0
    
    # ===== TIME / WIDTH =====
    f['Rise_Time'] = s_peak_idx / fs
    f['Fall_Time'] = (total_len - s_peak_idx) / fs
    f['Pulse_Width'] = total_len / fs
    f['Rise_Decay_Time_Ratio'] = f['Rise_Time'] / (f['Fall_Time'] + 1e-10)
    
    above_50 = np.where(seg_norm >= 0.5)[0]
    f['PW50'] = (above_50[-1] - above_50[0]) / fs if len(above_50) > 0 else 0
    
    notch_idx = int(s_peak_idx + 0.3 * (total_len - s_peak_idx))
    f['Systolic_Width'] = notch_idx / fs
    f['Diastolic_Width'] = (total_len - notch_idx) / fs
    f['Width_Ratio'] = f['Systolic_Width'] / (f['Diastolic_Width'] + 1e-10)
    
    # ===== SLOPE =====
    vpg = np.gradient(seg_norm, t)
    
    f['Upslope'] = np.max(vpg)
    f['Downslope'] = np.min(vpg)
    f['Onset_End_Slope'] = (seg_norm[-1] - seg_norm[0]) / (t[-1] - t[0] + 1e-10)
    f['Slope_Ratio'] = f['Upslope'] / (abs(f['Downslope']) + 1e-10)
    
    f['Delta_T'] = calculate_delta_t_robust(vpg, t, fs)
    
    dx = np.diff(t)
    dy = np.diff(seg_norm)
    arc_increments = np.sqrt(dx**2 + dy**2)
    total_arc = np.sum(arc_increments)
    upslope_arc = np.sum(arc_increments[:s_peak_idx]) if s_peak_idx > 0 else 0
    downslope_arc = np.sum(arc_increments[s_peak_idx:])
    
    f['Upslope_Length'] = upslope_arc
    f['Downslope_Length'] = downslope_arc
    f['Slope_Length_Ratio'] = upslope_arc / (downslope_arc + 1e-10)
    f['Upslope_Length_Ratio'] = upslope_arc / (total_arc + 1e-10)
    f['Downslope_Length_Ratio'] = downslope_arc / (total_arc + 1e-10)
    f['Length_Height_Ratio'] = total_arc / (seg_norm[s_peak_idx] + 1e-10)
    
    # ===== SDPPG =====
    apg = np.gradient(vpg, t)
    
    a_wave_idx_max = min(int(0.2 * fs), len(apg))
    a_wave = np.max(apg[:a_wave_idx_max]) if a_wave_idx_max > 0 else 1e-10
    
    def safe_apg_ratio(start_frac, end_frac, mode='max'):
        start_idx = int(start_frac * fs)
        end_idx = int(end_frac * fs)
        if start_idx >= len(apg) or end_idx > len(apg) or start_idx >= end_idx:
            return 0
        segment = apg[start_idx:end_idx]
        if len(segment) == 0:
            return 0
        val = np.max(segment) if mode == 'max' else np.min(segment)
        return val / (a_wave + 1e-10)
    
    f['SDPPG_b_a'] = safe_apg_ratio(0.05, 0.15, 'min')
    f['SDPPG_c_a'] = safe_apg_ratio(0.15, 0.25, 'max')
    f['SDPPG_d_a'] = safe_apg_ratio(0.25, 0.35, 'min')
    f['SDPPG_e_a'] = safe_apg_ratio(0.35, 0.50, 'max')
    
    search_start = s_peak_idx + int(0.1 * fs)
    if search_start < len(seg_norm) - 10:
        f['DNP'] = seg_norm[notch_idx] if notch_idx < len(seg_norm) else 0
    else:
        f['DNP'] = 0
    
    # ===== SHAPE & STATS =====
    f['Skewness'] = stats.skew(seg_norm)
    f['Kurtosis'] = stats.kurtosis(seg_norm)
    f['Variance'] = np.var(seg_norm)
    
    if search_start < len(seg_norm) - 10:
        diastolic_seg = seg_norm[search_start:]
        d_peak = np.max(diastolic_seg)
        f['RWI'] = d_peak / (seg_norm[s_peak_idx] + 1e-10)
    else:
        f['RWI'] = 0
    
    inflection_idx = int(0.3 * total_len)
    if inflection_idx > 0 and inflection_idx < len(sig_bc):
        area_before = np.trapz(sig_bc[:inflection_idx])
        area_after = np.trapz(sig_bc[inflection_idx:])
        f['IPAR'] = area_before / (area_after + 1e-10)
    else:
        f['IPAR'] = 0
    
    return f

# ========================================
# MAIN PROCESSING
# ========================================

def main():
    log_message("="*80)
    log_message(f"MASTER FEATURE EXTRACTION: {TARGET_DEPTH}mm {TARGET_CHANNEL} Channel")
    log_message("="*80)
    
    all_features = []
    delta_t_stats = {'zero_count': 0, 'valid_count': 0, 'values': []}
    
    for state_code, state_name in STATE_MAP.items():
        filename = f'{TARGET_DEPTH}mm_processed_data{state_code}.csv'
        
        if not os.path.exists(filename):
            log_message(f"⚠️  File not found: {filename}")
            continue
        
        try:
            df = pd.read_csv(filename)
            log_message(f"📂 Processing: {filename} ({state_name})")
            
            if RAW_COLUMN not in df.columns:
                log_message(f"  ⚠️  Raw column '{RAW_COLUMN}' not found")
                continue
            if PROCESSED_COLUMN not in df.columns:
                log_message(f"  ⚠️  Processed column '{PROCESSED_COLUMN}' not found")
                continue
            
            max_samples = int(30 * FS)
            raw_signal = df[RAW_COLUMN].values[:max_samples]
            processed_signal = df[PROCESSED_COLUMN].values[:max_samples]
            
            troughs, _ = signal.find_peaks(-processed_signal, distance=int(FS * 0.6))
            
            pulse_count = 0
            for i in range(len(troughs) - 1):
                idx1, idx2 = troughs[i], troughs[i+1]
                
                pulse_duration = (idx2 - idx1) / FS
                if pulse_duration < 0.4 or pulse_duration > 1.5:
                    continue
                
                seg_raw = raw_signal[idx1:idx2]
                seg_processed = processed_signal[idx1:idx2]
                
                features = extract_comprehensive_features(seg_raw, seg_processed, FS)
                
                if features['Delta_T'] == 0:
                    delta_t_stats['zero_count'] += 1
                else:
                    delta_t_stats['valid_count'] += 1
                    delta_t_stats['values'].append(features['Delta_T'])
                
                features['Depth'] = TARGET_DEPTH
                features['Channel'] = TARGET_CHANNEL
                features['State'] = state_name
                features['Pulse_Index'] = i
                
                all_features.append(features)
                pulse_count += 1
            
            log_message(f"  ✓ Extracted {pulse_count} pulses from {state_name}")
        
        except Exception as e:
            log_message(f"  ✗ Error processing {filename}: {str(e)}")
            import traceback
            log_message(traceback.format_exc())
            continue
    
    if all_features:
        df_master = pd.DataFrame(all_features)
        
        metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
        feature_cols = [col for col in df_master.columns if col not in metadata_cols]
        df_master = df_master[metadata_cols + sorted(feature_cols)]
        
        output_csv = os.path.join(OUTPUT_DIR, 'master_features_9mm_IR.csv')
        df_master.to_csv(output_csv, index=False)
        
        log_message("\n" + "="*80)
        log_message("SUMMARY")
        log_message("="*80)
        log_message(f"✓ Output file: {output_csv}")
        log_message(f"✓ Total pulses: {len(df_master)}")
        log_message(f"✓ Total features: {len(feature_cols)}")
        
        log_message("\nData Distribution:")
        for state in STATE_MAP.values():
            count = len(df_master[df_master['State'] == state])
            log_message(f"  {state:12s}: {count} pulses")
        
        log_message("\nDelta_T Quality Check:")
        log_message(f"  Valid values: {delta_t_stats['valid_count']}")
        log_message(f"  Zero values: {delta_t_stats['zero_count']}")
        if delta_t_stats['values']:
            log_message(f"  Mean: {np.mean(delta_t_stats['values']):.4f} s")
            log_message(f"  Std: {np.std(delta_t_stats['values']):.4f} s")
            log_message(f"  Min: {np.min(delta_t_stats['values']):.4f} s")
            log_message(f"  Max: {np.max(delta_t_stats['values']):.4f} s")
            log_message(f"  Median: {np.median(delta_t_stats['values']):.4f} s")
        
        log_message("\n" + "="*80)
        log_message("✓ FEATURE EXTRACTION COMPLETE")
        log_message("="*80)
    else:
        log_message("\n⚠️  ERROR: No pulses extracted.")

if __name__ == "__main__":
    main()


In [ ]:
"""
Diagnostic Performance Analysis: 9mm IR Channel Only (FULLY CORRECTED)
========================================================================
Calculate comprehensive diagnostic performance metrics for all features
FIXED: Proper handling of inverse relationships (AUC < 0.5)
Comparisons: Normal vs Ischaemia, Normal vs Congestion
Output: Detailed performance tables with AUC, sensitivity, specificity, etc.
"""

import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from scipy import stats
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ========================================
# CONFIGURATION
# ========================================
INPUT_FILE = 'master_features_9mm_ir/master_features_9mm_IR.csv'
OUTPUT_DIR = 'diagnostic_performance_9mm_ir'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, 'performance_analysis_log.txt')
ERROR_LOG = os.path.join(OUTPUT_DIR, 'error_details.txt')

def log_message(message, error=False):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f"[{timestamp}] {message}"
    print(entry)
    
    log_file = ERROR_LOG if error else LOG_FILE
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write(entry + '\n')

# ========================================
# PERFORMANCE METRICS CALCULATION
# ========================================

def calculate_auc_ci(y_true, y_pred, confidence=0.95, n_bootstraps=1000):
    """
    Calculate AUC confidence interval using bootstrap method
    
    FIXED: Convert pandas Series to numpy array to avoid index issues
    """
    if len(np.unique(y_true)) < 2:
        return np.nan, np.nan
    
    # ===== FIX: Convert to numpy arrays =====
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    try:
        auc_base = roc_auc_score(y_true, y_pred)
    except:
        return np.nan, np.nan
    
    # Bootstrap
    np.random.seed(42)
    auc_scores = []
    n_samples = len(y_true)
    
    for i in range(n_bootstraps):
        indices = np.random.choice(n_samples, n_samples, replace=True)
        y_true_boot = y_true[indices]  # ← Now works with numpy array
        y_pred_boot = y_pred[indices]
        
        if len(np.unique(y_true_boot)) < 2:
            continue
        
        try:
            auc_boot = roc_auc_score(y_true_boot, y_pred_boot)
            auc_scores.append(auc_boot)
        except:
            continue
    
    if len(auc_scores) < 10:
        return np.nan, np.nan
    
    alpha = (1 - confidence) / 2
    ci_lower = np.percentile(auc_scores, alpha * 100)
    ci_upper = np.percentile(auc_scores, (1 - alpha) * 100)
    
    return ci_lower, ci_upper

def calculate_optimal_threshold(y_true, y_pred):
    """
    Calculate optimal threshold using Youden's J statistic
    J = Sensitivity + Specificity - 1
    
    Parameters:
    -----------
    y_true : array
        True binary labels
    y_pred : array
        Predicted values (already corrected for direction)
    
    Returns:
    --------
    float : Optimal threshold value
    """
    try:
        fpr, tpr, thresholds = roc_curve(y_true, y_pred)
        j_scores = tpr - fpr
        optimal_idx = np.argmax(j_scores)
        return thresholds[optimal_idx]
    except:
        return np.median(y_pred)

def calculate_diagnostic_performance(df, feature, comparison='Ischaemia'):
    """
    Calculate comprehensive diagnostic performance metrics
    FULLY CORRECTED: Handles inverse relationships properly
    
    Parameters:
    -----------
    df : DataFrame
        Master features dataframe
    feature : str
        Feature name to analyze
    comparison : str
        'Ischaemia' or 'Congestion'
    
    Returns:
    --------
    dict : Dictionary with performance metrics, or None if calculation fails
    """
    try:
        # Filter data: Normal vs comparison state
        df_filtered = df[df['State'].isin(['Normal', comparison])].copy()
        
        # Check if feature exists and has valid data
        if feature not in df_filtered.columns:
            log_message(f"Feature {feature} not found", error=True)
            return None
        
        # Remove NaN and infinite values
        df_clean = df_filtered[[feature, 'State']].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(df_clean) < 10:
            log_message(f"Feature {feature}: insufficient data (N={len(df_clean)})", error=True)
            return None
        
        if len(df_clean['State'].unique()) < 2:
            log_message(f"Feature {feature}: only one class present", error=True)
            return None
        
        # Binary labels
        y_true = (df_clean['State'] == comparison).astype(int)
        y_pred_original = df_clean[feature].values
        
        # Check for variance
        if np.std(y_pred_original) < 1e-10:
            log_message(f"Feature {feature}: zero variance", error=True)
            return None
        
        # Sample sizes
        n_normal = int((y_true == 0).sum())
        n_disease = int((y_true == 1).sum())
        
        # ===== CRITICAL FIX: Handle inverse relationships =====
        # Calculate raw AUC
        auc_raw = roc_auc_score(y_true, y_pred_original)
        
        # Determine if relationship is inverse (AUC < 0.5)
        if auc_raw < 0.5:
            # Inverse relationship: lower values indicate disease
            auc_corrected = 1 - auc_raw
            y_pred_corrected = -y_pred_original  # Invert predictions
            is_inverse = True
        else:
            # Normal relationship: higher values indicate disease
            auc_corrected = auc_raw
            y_pred_corrected = y_pred_original
            is_inverse = False
        
        # Use corrected predictions for all downstream calculations
        ci_lower, ci_upper = calculate_auc_ci(y_true, y_pred_corrected)
        threshold = calculate_optimal_threshold(y_true, y_pred_corrected)
        
        # Binary predictions based on corrected values
        y_pred_binary = (y_pred_corrected >= threshold).astype(int)
        
        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary).ravel()
        
        # Calculate metrics
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        f1_score = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
        
        # Likelihood ratios
        if specificity < 1.0 and specificity > 0:
            lr_positive = sensitivity / (1 - specificity)
        elif specificity == 1.0 and sensitivity > 0:
            lr_positive = np.inf
        else:
            lr_positive = np.nan
        
        if specificity > 0 and sensitivity < 1.0:
            lr_negative = (1 - sensitivity) / specificity
        elif sensitivity == 1.0:
            lr_negative = 0.0
        else:
            lr_negative = np.nan
        
        # Youden's J statistic
        youden_j = sensitivity + specificity - 1
        
        # Convert threshold back to original scale if inverse
        if is_inverse:
            threshold_original = -threshold
        else:
            threshold_original = threshold
        
        return {
            'Feature': feature,
            'Comparison': comparison,
            'N_Normal': n_normal,
            'N_Disease': n_disease,
            'AUC': float(auc_corrected),
            'CI_Lower': float(ci_lower) if not np.isnan(ci_lower) else np.nan,
            'CI_Upper': float(ci_upper) if not np.isnan(ci_upper) else np.nan,
            'Optimal_Threshold': float(threshold_original),
            'Sensitivity': float(sensitivity),
            'Specificity': float(specificity),
            'PPV': float(ppv),
            'NPV': float(npv),
            'Accuracy': float(accuracy),
            'F1_Score': float(f1_score),
            'Youden_J': float(youden_j),
            'LR_Positive': float(lr_positive) if np.isfinite(lr_positive) else np.nan,
            'LR_Negative': float(lr_negative) if np.isfinite(lr_negative) else np.nan,
            'TP': int(tp),
            'TN': int(tn),
            'FP': int(fp),
            'FN': int(fn),
            'Is_Inverse': is_inverse  # Track if relationship was inverted
        }
    
    except Exception as e:
        import traceback
        error_msg = f"Feature {feature} for {comparison}: {str(e)}\n{traceback.format_exc()}"
        log_message(error_msg, error=True)
        return None

# ========================================
# PULSE COUNT SUMMARY (for manuscript reporting)
# ========================================

def summarize_pulse_counts(df, output_dir):
    """
    Report actual number of valid cardiac cycles per state.
    This is the base N before any feature-level NaN removal.
    Used for manuscript Methods section reporting.
    """
    log_message("\n" + "="*80)
    log_message("VALID CARDIAC CYCLE COUNTS (Base N per State)")
    log_message("="*80)
    
    # Total per state
    summary = (
        df.groupby('State')
        .size()
        .reset_index(name='N_Pulses')
        .sort_values('State')
    )
    
    total = summary['N_Pulses'].sum()
    
    log_message(f"\n{'State':<15} {'N Pulses':>10}")
    log_message("-" * 26)
    for _, row in summary.iterrows():
        log_message(f"  {row['State']:<13} {row['N_Pulses']:>10}")
    log_message("-" * 26)
    log_message(f"  {'Total':<13} {total:>10}")
    
    # Save CSV
    summary_file = os.path.join(output_dir, 'pulse_count_summary.csv')
    summary.to_csv(summary_file, index=False)
    log_message(f"\n✓ Saved pulse count summary: {summary_file}")
    
    # Also report per-feature valid N range (min/max after NaN removal)
    # Useful to confirm data quality across features
    metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
    feature_cols = [col for col in df.columns if col not in metadata_cols]
    
    valid_n_per_feature = (
        df[feature_cols + ['State']]
        .replace([np.inf, -np.inf], np.nan)
        .groupby('State')
        .apply(lambda g: g.drop(columns='State').notna().sum())
    )
    
    log_message("\nPer-feature valid N range (after NaN/Inf removal):")
    for state in valid_n_per_feature.index:
        row = valid_n_per_feature.loc[state]
        log_message(
            f"  {state:<13}: "
            f"min={row.min()}, max={row.max()}, "
            f"mean={row.mean():.1f}"
        )
    
    return summary

# ========================================
# MAIN ANALYSIS
# ========================================

def main():
    log_message("="*80)
    log_message("DIAGNOSTIC PERFORMANCE ANALYSIS: 9mm IR Channel (FULLY CORRECTED)")
    log_message("="*80)
    
    # Load master table
    if not os.path.exists(INPUT_FILE):
        log_message(f"ERROR: Input file not found: {INPUT_FILE}")
        return
    
    df = pd.read_csv(INPUT_FILE)
    log_message(f"✓ Loaded: {INPUT_FILE}")
    log_message(f"  Total rows: {len(df)}")
    log_message(f"  Total columns: {len(df.columns)}")
    pulse_counts = summarize_pulse_counts(df, OUTPUT_DIR)
    
    # Data distribution
    log_message("\nData Distribution:")
    for state in sorted(df['State'].unique()):
        count = len(df[df['State'] == state])
        log_message(f"  {state:12s}: {count} pulses")
    
    # Get feature columns (exclude metadata)
    metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
    feature_cols = [col for col in df.columns if col not in metadata_cols]
    
    log_message(f"\n✓ Total features to analyze: {len(feature_cols)}")
    
    # Analysis configurations
    comparisons = ['Ischaemia', 'Congestion']
    
    all_results = []
    success_count = 0
    error_count = 0
    inverse_count = 0
    
    for comparison in comparisons:
        log_message(f"\n{'='*80}")
        log_message(f"Analyzing: Normal vs {comparison}")
        log_message(f"{'='*80}")
        
        for i, feature in enumerate(feature_cols, 1):
            if i % 10 == 0:
                log_message(f"  Progress: {i}/{len(feature_cols)}")
            
            result = calculate_diagnostic_performance(df, feature, comparison)
            
            if result:
                all_results.append(result)
                success_count += 1
                if result['Is_Inverse']:
                    inverse_count += 1
            else:
                error_count += 1
    
    # Save results
    if all_results:
        df_results = pd.DataFrame(all_results)
        
        # Sort by AUC descending within each comparison
        df_results = df_results.sort_values(['Comparison', 'AUC'], 
                                            ascending=[True, False])
        
        # Save comprehensive results
        output_csv = os.path.join(OUTPUT_DIR, 'diagnostic_performance_9mm_IR_all.csv')
        df_results.to_csv(output_csv, index=False, float_format='%.6f')
        
        log_message("\n" + "="*80)
        log_message("RESULTS SUMMARY")
        log_message("="*80)
        log_message(f"✓ Output file: {output_csv}")
        log_message(f"✓ Total successful analyses: {success_count}")
        log_message(f"✓ Total failed analyses: {error_count}")
        log_message(f"✓ Inverse relationships corrected: {inverse_count}")
        log_message(f"✓ Success rate: {100*success_count/(success_count+error_count):.1f}%")
        
        # Top 10 features by comparison
        log_message("\n" + "="*80)
        log_message("TOP 10 FEATURES BY AUC")
        log_message("="*80)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            top10 = df_comp.nlargest(10, 'AUC')
            
            log_message(f"\n{comparison} (Normal vs {comparison}):")
            log_message("-"*80)
            
            for idx, row in top10.iterrows():
                inverse_marker = " [INV]" if row['Is_Inverse'] else ""
                log_message(
                    f"  {row['Feature']:30s}{inverse_marker:6s} | "
                    f"AUC: {row['AUC']:.4f} ({row['CI_Lower']:.4f}-{row['CI_Upper']:.4f}) | "
                    f"Sens: {row['Sensitivity']:.3f} | Spec: {row['Specificity']:.3f} | "
                    f"Acc: {row['Accuracy']:.3f}"
                )
        
        # Create separate files for each comparison
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            comp_file = os.path.join(OUTPUT_DIR, f'diagnostic_performance_9mm_IR_{comparison}.csv')
            df_comp.to_csv(comp_file, index=False, float_format='%.6f')
            log_message(f"\n✓ Saved {comparison} results: {comp_file} ({len(df_comp)} features)")
        
        # Create ranking tables (top 20)
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            df_rank = df_comp.nlargest(20, 'AUC')
            
            rank_file = os.path.join(OUTPUT_DIR, f'ranking_table_9mm_IR_{comparison}.csv')
            df_rank.to_csv(rank_file, index=False, float_format='%.6f')
            log_message(f"✓ Saved ranking: {rank_file}")
        
        # Statistical summary by feature category
        log_message("\n" + "="*80)
        log_message("FEATURE CATEGORY ANALYSIS")
        log_message("="*80)
        
        def categorize_feature(name):
            """Categorize features by type"""
            name_lower = name.lower()
            if any(x in name_lower for x in ['pi', 'amplitude', 'ac_', 'dc_']):
                return 'Intensity'
            elif any(x in name_lower for x in ['auc', 'area', 'datum']):
                return 'Area'
            elif any(x in name_lower for x in ['time', 'width', 'pw50', 'delta_t']):
                return 'Time'
            elif any(x in name_lower for x in ['slope', 'upslope', 'downslope', 'length']):
                return 'Slope'
            elif any(x in name_lower for x in ['sdppg', '_a', '_b', '_c', '_d', '_e', 'dnp']):
                return 'SDPPG'
            elif any(x in name_lower for x in ['skewness', 'kurtosis', 'variance', 'rwi', 'ipar']):
                return 'Shape & Stats'
            else:
                return 'Other'
        
        df_results['Category'] = df_results['Feature'].apply(categorize_feature)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            
            log_message(f"\n{comparison}:")
            log_message("-"*80)
            
            for category in sorted(df_comp['Category'].unique()):
                df_cat = df_comp[df_comp['Category'] == category]
                mean_auc = df_cat['AUC'].mean()
                median_auc = df_cat['AUC'].median()
                max_auc = df_cat['AUC'].max()
                n_inverse = df_cat['Is_Inverse'].sum()
                
                log_message(
                    f"  {category:15s}: "
                    f"N={len(df_cat):2d} | "
                    f"Mean AUC={mean_auc:.4f} | "
                    f"Median AUC={median_auc:.4f} | "
                    f"Max AUC={max_auc:.4f} | "
                    f"Inverse={n_inverse}"
                )
        
        # Inverse relationship summary
        log_message("\n" + "="*80)
        log_message("INVERSE RELATIONSHIP SUMMARY")
        log_message("="*80)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            df_inv = df_comp[df_comp['Is_Inverse'] == True]
            
            if len(df_inv) > 0:
                log_message(f"\n{comparison} - Features with inverse relationships ({len(df_inv)}):")
                for idx, row in df_inv.iterrows():
                    log_message(f"  {row['Feature']:30s} | AUC: {row['AUC']:.4f}")
        
        log_message("\n" + "="*80)
        log_message("✓ ANALYSIS COMPLETE")
        log_message(f"✓ Check {ERROR_LOG} for detailed error messages (if any)")
        log_message("="*80)
    
    else:
        log_message("\n⚠️  ERROR: No valid results generated. Check error log for details.")

if __name__ == "__main__":
    main()


In [ ]:
"""
Master Feature Extraction: 15mm IR Channel Only
===============================================
Extracts comprehensive pulse-level features from 15mm depth IR channel
Following the same pipeline as 15mm IR for consistency
Output format designed for easy integration with other wavelengths/depths
"""

import pandas as pd
import numpy as np
from scipy import signal, stats
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ========================================
# CONFIGURATION
# ========================================
FS = 2000
TARGET_DEPTH = 15
TARGET_CHANNEL = 'IR'
RAW_COLUMN = 'ppgA_IR_raw'
PROCESSED_COLUMN = 'IR_processed'

STATE_MAP = {1: 'Normal', 2: 'Ischaemia', 3: 'Congestion'}

OUTPUT_DIR = 'master_features_15mm_ir'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, 'processing_log.txt')

def log_message(message):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f"[{timestamp}] {message}"
    print(entry)
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(entry + '\n')

# ========================================
# HELPER FUNCTIONS
# ========================================

def calculate_delta_t_robust(vpg, t, fs):
    """
    Robust Delta_T calculation using multiple methods
    
    Delta_T = Time interval between:
    1. Maximum acceleration point (VPG peak)
    2. Maximum deceleration point (first significant VPG trough after peak)
    """
    # Method 1: Peak-to-trough approach
    vpg_max_idx = np.argmax(vpg)
    
    search_start = vpg_max_idx + int(0.02 * fs)
    search_end = min(vpg_max_idx + int(0.4 * fs), len(vpg))
    
    if search_end > search_start and search_start < len(vpg):
        vpg_search_region = vpg[search_start:search_end]
        
        troughs, properties = signal.find_peaks(
            -vpg_search_region, 
            prominence=0.03 * np.max(np.abs(vpg)),
            distance=int(0.02 * fs)
        )
        
        if len(troughs) > 0:
            first_trough_idx = search_start + troughs[0]
            delta_t = (first_trough_idx - vpg_max_idx) / fs
            
            if 0.05 <= delta_t <= 0.4:
                return delta_t
    
    # Method 2: Zero-crossing approach
    zero_crossings = np.where(np.diff(np.sign(vpg)))[0]
    
    if len(zero_crossings) >= 2:
        zc_before_peak = zero_crossings[zero_crossings < vpg_max_idx]
        zc_after_peak = zero_crossings[zero_crossings > vpg_max_idx]
        
        if len(zc_before_peak) > 0 and len(zc_after_peak) > 0:
            zc1 = zc_before_peak[-1]
            zc2 = zc_after_peak[0]
            delta_t = (zc2 - zc1) / fs
            
            if 0.05 <= delta_t <= 0.4:
                return delta_t
    
    # Method 3: Threshold-based approach
    threshold = 0.1 * vpg[vpg_max_idx]
    after_peak = vpg[vpg_max_idx:]
    
    if len(after_peak) > 10:
        below_threshold = np.where(after_peak < threshold)[0]
        if len(below_threshold) > 0:
            delta_t = below_threshold[0] / fs
            
            if 0.05 <= delta_t <= 0.4:
                return delta_t
    
    # Method 4: Default estimate
    pulse_duration = len(vpg) / fs
    delta_t = 0.2 * pulse_duration
    delta_t = np.clip(delta_t, 0.05, 0.4)
    
    return delta_t

# ========================================
# FEATURE EXTRACTION
# ========================================

def extract_comprehensive_features(seg_raw, seg_processed, fs=2000):
    """
    Extract all pulse-level features from single pulse
    """
    f = {}
    
    # ===== PREPROCESSING COMPONENTS =====
    dc_component = np.mean(seg_raw)
    f['DC_Component'] = dc_component
    
    ac_component = np.max(seg_processed) - np.min(seg_processed)
    f['AC_Component'] = ac_component
    
    f['PI'] = (ac_component / (dc_component + 1e-10)) * 100
    
    # ===== NORMALIZATION =====
    seg_min = np.min(seg_processed)
    seg_max = np.max(seg_processed)
    seg_norm = (seg_processed - seg_min) / (seg_max - seg_min + 1e-10)
    
    t = np.arange(len(seg_norm)) / fs
    total_len = len(seg_norm)
    s_peak_idx = np.argmax(seg_norm)
    
    # ===== INTENSITY / AMPLITUDE =====
    f['Amplitude'] = ac_component
    f['Max_Start_Datum_Diff'] = np.max(seg_processed[:total_len//4]) if total_len >= 4 else 0
    
    # ===== AREA FEATURES =====
    baseline = np.linspace(seg_processed[0], seg_processed[-1], total_len)
    sig_bc = seg_processed - baseline
    
    f['AUC'] = np.trapz(sig_bc)
    f['S_AUC'] = np.trapz(sig_bc[:s_peak_idx+1])
    f['D_AUC'] = np.trapz(sig_bc[s_peak_idx:])
    f['AUC_Ratio'] = f['S_AUC'] / (f['D_AUC'] + 1e-10)
    
    q = total_len // 4
    if q > 0:
        f['Start_Datum_Area'] = np.trapz(sig_bc[:q])
        f['End_Datum_Area'] = np.trapz(sig_bc[-q:])
        f['Datum_Area_Ratio'] = f['Start_Datum_Area'] / (f['End_Datum_Area'] + 1e-10)
    else:
        f['Start_Datum_Area'] = 0
        f['End_Datum_Area'] = 0
        f['Datum_Area_Ratio'] = 0
    
    # ===== TIME / WIDTH =====
    f['Rise_Time'] = s_peak_idx / fs
    f['Fall_Time'] = (total_len - s_peak_idx) / fs
    f['Pulse_Width'] = total_len / fs
    f['Rise_Decay_Time_Ratio'] = f['Rise_Time'] / (f['Fall_Time'] + 1e-10)
    
    above_50 = np.where(seg_norm >= 0.5)[0]
    f['PW50'] = (above_50[-1] - above_50[0]) / fs if len(above_50) > 0 else 0
    
    notch_idx = int(s_peak_idx + 0.3 * (total_len - s_peak_idx))
    f['Systolic_Width'] = notch_idx / fs
    f['Diastolic_Width'] = (total_len - notch_idx) / fs
    f['Width_Ratio'] = f['Systolic_Width'] / (f['Diastolic_Width'] + 1e-10)
    
    # ===== SLOPE =====
    vpg = np.gradient(seg_norm, t)
    
    f['Upslope'] = np.max(vpg)
    f['Downslope'] = np.min(vpg)
    f['Onset_End_Slope'] = (seg_norm[-1] - seg_norm[0]) / (t[-1] - t[0] + 1e-10)
    f['Slope_Ratio'] = f['Upslope'] / (abs(f['Downslope']) + 1e-10)
    
    f['Delta_T'] = calculate_delta_t_robust(vpg, t, fs)
    
    dx = np.diff(t)
    dy = np.diff(seg_norm)
    arc_increments = np.sqrt(dx**2 + dy**2)
    total_arc = np.sum(arc_increments)
    upslope_arc = np.sum(arc_increments[:s_peak_idx]) if s_peak_idx > 0 else 0
    downslope_arc = np.sum(arc_increments[s_peak_idx:])
    
    f['Upslope_Length'] = upslope_arc
    f['Downslope_Length'] = downslope_arc
    f['Slope_Length_Ratio'] = upslope_arc / (downslope_arc + 1e-10)
    f['Upslope_Length_Ratio'] = upslope_arc / (total_arc + 1e-10)
    f['Downslope_Length_Ratio'] = downslope_arc / (total_arc + 1e-10)
    f['Length_Height_Ratio'] = total_arc / (seg_norm[s_peak_idx] + 1e-10)
    
    # ===== SDPPG =====
    apg = np.gradient(vpg, t)
    
    a_wave_idx_max = min(int(0.2 * fs), len(apg))
    a_wave = np.max(apg[:a_wave_idx_max]) if a_wave_idx_max > 0 else 1e-10
    
    def safe_apg_ratio(start_frac, end_frac, mode='max'):
        start_idx = int(start_frac * fs)
        end_idx = int(end_frac * fs)
        if start_idx >= len(apg) or end_idx > len(apg) or start_idx >= end_idx:
            return 0
        segment = apg[start_idx:end_idx]
        if len(segment) == 0:
            return 0
        val = np.max(segment) if mode == 'max' else np.min(segment)
        return val / (a_wave + 1e-10)
    
    f['SDPPG_b_a'] = safe_apg_ratio(0.05, 0.15, 'min')
    f['SDPPG_c_a'] = safe_apg_ratio(0.15, 0.25, 'max')
    f['SDPPG_d_a'] = safe_apg_ratio(0.25, 0.35, 'min')
    f['SDPPG_e_a'] = safe_apg_ratio(0.35, 0.50, 'max')
    
    search_start = s_peak_idx + int(0.1 * fs)
    if search_start < len(seg_norm) - 10:
        f['DNP'] = seg_norm[notch_idx] if notch_idx < len(seg_norm) else 0
    else:
        f['DNP'] = 0
    
    # ===== SHAPE & STATS =====
    f['Skewness'] = stats.skew(seg_norm)
    f['Kurtosis'] = stats.kurtosis(seg_norm)
    f['Variance'] = np.var(seg_norm)
    
    if search_start < len(seg_norm) - 10:
        diastolic_seg = seg_norm[search_start:]
        d_peak = np.max(diastolic_seg)
        f['RWI'] = d_peak / (seg_norm[s_peak_idx] + 1e-10)
    else:
        f['RWI'] = 0
    
    inflection_idx = int(0.3 * total_len)
    if inflection_idx > 0 and inflection_idx < len(sig_bc):
        area_before = np.trapz(sig_bc[:inflection_idx])
        area_after = np.trapz(sig_bc[inflection_idx:])
        f['IPAR'] = area_before / (area_after + 1e-10)
    else:
        f['IPAR'] = 0
    
    return f

# ========================================
# MAIN PROCESSING
# ========================================

def main():
    log_message("="*80)
    log_message(f"MASTER FEATURE EXTRACTION: {TARGET_DEPTH}mm {TARGET_CHANNEL} Channel")
    log_message("="*80)
    
    all_features = []
    delta_t_stats = {'zero_count': 0, 'valid_count': 0, 'values': []}
    
    for state_code, state_name in STATE_MAP.items():
        filename = f'{TARGET_DEPTH}mm_processed_data{state_code}.csv'
        
        if not os.path.exists(filename):
            log_message(f"⚠️  File not found: {filename}")
            continue
        
        try:
            df = pd.read_csv(filename)
            log_message(f"📂 Processing: {filename} ({state_name})")
            
            if RAW_COLUMN not in df.columns:
                log_message(f"  ⚠️  Raw column '{RAW_COLUMN}' not found")
                continue
            if PROCESSED_COLUMN not in df.columns:
                log_message(f"  ⚠️  Processed column '{PROCESSED_COLUMN}' not found")
                continue
            
            max_samples = int(30 * FS)
            raw_signal = df[RAW_COLUMN].values[:max_samples]
            processed_signal = df[PROCESSED_COLUMN].values[:max_samples]
            
            troughs, _ = signal.find_peaks(-processed_signal, distance=int(FS * 0.6))
            
            pulse_count = 0
            for i in range(len(troughs) - 1):
                idx1, idx2 = troughs[i], troughs[i+1]
                
                pulse_duration = (idx2 - idx1) / FS
                if pulse_duration < 0.4 or pulse_duration > 1.5:
                    continue
                
                seg_raw = raw_signal[idx1:idx2]
                seg_processed = processed_signal[idx1:idx2]
                
                features = extract_comprehensive_features(seg_raw, seg_processed, FS)
                
                if features['Delta_T'] == 0:
                    delta_t_stats['zero_count'] += 1
                else:
                    delta_t_stats['valid_count'] += 1
                    delta_t_stats['values'].append(features['Delta_T'])
                
                features['Depth'] = TARGET_DEPTH
                features['Channel'] = TARGET_CHANNEL
                features['State'] = state_name
                features['Pulse_Index'] = i
                
                all_features.append(features)
                pulse_count += 1
            
            log_message(f"  ✓ Extracted {pulse_count} pulses from {state_name}")
        
        except Exception as e:
            log_message(f"  ✗ Error processing {filename}: {str(e)}")
            import traceback
            log_message(traceback.format_exc())
            continue
    
    if all_features:
        df_master = pd.DataFrame(all_features)
        
        metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
        feature_cols = [col for col in df_master.columns if col not in metadata_cols]
        df_master = df_master[metadata_cols + sorted(feature_cols)]
        
        output_csv = os.path.join(OUTPUT_DIR, 'master_features_15mm_IR.csv')
        df_master.to_csv(output_csv, index=False)
        
        log_message("\n" + "="*80)
        log_message("SUMMARY")
        log_message("="*80)
        log_message(f"✓ Output file: {output_csv}")
        log_message(f"✓ Total pulses: {len(df_master)}")
        log_message(f"✓ Total features: {len(feature_cols)}")
        
        log_message("\nData Distribution:")
        for state in STATE_MAP.values():
            count = len(df_master[df_master['State'] == state])
            log_message(f"  {state:12s}: {count} pulses")
        
        log_message("\nDelta_T Quality Check:")
        log_message(f"  Valid values: {delta_t_stats['valid_count']}")
        log_message(f"  Zero values: {delta_t_stats['zero_count']}")
        if delta_t_stats['values']:
            log_message(f"  Mean: {np.mean(delta_t_stats['values']):.4f} s")
            log_message(f"  Std: {np.std(delta_t_stats['values']):.4f} s")
            log_message(f"  Min: {np.min(delta_t_stats['values']):.4f} s")
            log_message(f"  Max: {np.max(delta_t_stats['values']):.4f} s")
            log_message(f"  Median: {np.median(delta_t_stats['values']):.4f} s")
        
        log_message("\n" + "="*80)
        log_message("✓ FEATURE EXTRACTION COMPLETE")
        log_message("="*80)
    else:
        log_message("\n⚠️  ERROR: No pulses extracted.")

if __name__ == "__main__":
    main()

In [ ]:
"""
Diagnostic Performance Analysis: 15mm IR Channel Only (FULLY CORRECTED)
========================================================================
Calculate comprehensive diagnostic performance metrics for all features
FIXED: Proper handling of inverse relationships (AUC < 0.5)
Comparisons: Normal vs Ischaemia, Normal vs Congestion
Output: Detailed performance tables with AUC, sensitivity, specificity, etc.
"""

import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from scipy import stats
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ========================================
# CONFIGURATION
# ========================================
INPUT_FILE = 'master_features_15mm_ir/master_features_15mm_IR.csv'
OUTPUT_DIR = 'diagnostic_performance_15mm_ir'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, 'performance_analysis_log.txt')
ERROR_LOG = os.path.join(OUTPUT_DIR, 'error_details.txt')

def log_message(message, error=False):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f"[{timestamp}] {message}"
    print(entry)
    
    log_file = ERROR_LOG if error else LOG_FILE
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write(entry + '\n')

# ========================================
# PERFORMANCE METRICS CALCULATION
# ========================================

def calculate_auc_ci(y_true, y_pred, confidence=0.95, n_bootstraps=1000):
    """
    Calculate AUC confidence interval using bootstrap method
    
    FIXED: Convert pandas Series to numpy array to avoid index issues
    """
    if len(np.unique(y_true)) < 2:
        return np.nan, np.nan
    
    # ===== FIX: Convert to numpy arrays =====
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    try:
        auc_base = roc_auc_score(y_true, y_pred)
    except:
        return np.nan, np.nan
    
    # Bootstrap
    np.random.seed(42)
    auc_scores = []
    n_samples = len(y_true)
    
    for i in range(n_bootstraps):
        indices = np.random.choice(n_samples, n_samples, replace=True)
        y_true_boot = y_true[indices]  # ← Now works with numpy array
        y_pred_boot = y_pred[indices]
        
        if len(np.unique(y_true_boot)) < 2:
            continue
        
        try:
            auc_boot = roc_auc_score(y_true_boot, y_pred_boot)
            auc_scores.append(auc_boot)
        except:
            continue
    
    if len(auc_scores) < 10:
        return np.nan, np.nan
    
    alpha = (1 - confidence) / 2
    ci_lower = np.percentile(auc_scores, alpha * 100)
    ci_upper = np.percentile(auc_scores, (1 - alpha) * 100)
    
    return ci_lower, ci_upper

def calculate_optimal_threshold(y_true, y_pred):
    """
    Calculate optimal threshold using Youden's J statistic
    J = Sensitivity + Specificity - 1
    
    Parameters:
    -----------
    y_true : array
        True binary labels
    y_pred : array
        Predicted values (already corrected for direction)
    
    Returns:
    --------
    float : Optimal threshold value
    """
    try:
        fpr, tpr, thresholds = roc_curve(y_true, y_pred)
        j_scores = tpr - fpr
        optimal_idx = np.argmax(j_scores)
        return thresholds[optimal_idx]
    except:
        return np.median(y_pred)

def calculate_diagnostic_performance(df, feature, comparison='Ischaemia'):
    """
    Calculate comprehensive diagnostic performance metrics
    FULLY CORRECTED: Handles inverse relationships properly
    
    Parameters:
    -----------
    df : DataFrame
        Master features dataframe
    feature : str
        Feature name to analyze
    comparison : str
        'Ischaemia' or 'Congestion'
    
    Returns:
    --------
    dict : Dictionary with performance metrics, or None if calculation fails
    """
    try:
        # Filter data: Normal vs comparison state
        df_filtered = df[df['State'].isin(['Normal', comparison])].copy()
        
        # Check if feature exists and has valid data
        if feature not in df_filtered.columns:
            log_message(f"Feature {feature} not found", error=True)
            return None
        
        # Remove NaN and infinite values
        df_clean = df_filtered[[feature, 'State']].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(df_clean) < 10:
            log_message(f"Feature {feature}: insufficient data (N={len(df_clean)})", error=True)
            return None
        
        if len(df_clean['State'].unique()) < 2:
            log_message(f"Feature {feature}: only one class present", error=True)
            return None
        
        # Binary labels
        y_true = (df_clean['State'] == comparison).astype(int)
        y_pred_original = df_clean[feature].values
        
        # Check for variance
        if np.std(y_pred_original) < 1e-10:
            log_message(f"Feature {feature}: zero variance", error=True)
            return None
        
        # Sample sizes
        n_normal = int((y_true == 0).sum())
        n_disease = int((y_true == 1).sum())
        
        # ===== CRITICAL FIX: Handle inverse relationships =====
        # Calculate raw AUC
        auc_raw = roc_auc_score(y_true, y_pred_original)
        
        # Determine if relationship is inverse (AUC < 0.5)
        if auc_raw < 0.5:
            # Inverse relationship: lower values indicate disease
            auc_corrected = 1 - auc_raw
            y_pred_corrected = -y_pred_original  # Invert predictions
            is_inverse = True
        else:
            # Normal relationship: higher values indicate disease
            auc_corrected = auc_raw
            y_pred_corrected = y_pred_original
            is_inverse = False
        
        # Use corrected predictions for all downstream calculations
        ci_lower, ci_upper = calculate_auc_ci(y_true, y_pred_corrected)
        threshold = calculate_optimal_threshold(y_true, y_pred_corrected)
        
        # Binary predictions based on corrected values
        y_pred_binary = (y_pred_corrected >= threshold).astype(int)
        
        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary).ravel()
        
        # Calculate metrics
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        f1_score = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
        
        # Likelihood ratios
        if specificity < 1.0 and specificity > 0:
            lr_positive = sensitivity / (1 - specificity)
        elif specificity == 1.0 and sensitivity > 0:
            lr_positive = np.inf
        else:
            lr_positive = np.nan
        
        if specificity > 0 and sensitivity < 1.0:
            lr_negative = (1 - sensitivity) / specificity
        elif sensitivity == 1.0:
            lr_negative = 0.0
        else:
            lr_negative = np.nan
        
        # Youden's J statistic
        youden_j = sensitivity + specificity - 1
        
        # Convert threshold back to original scale if inverse
        if is_inverse:
            threshold_original = -threshold
        else:
            threshold_original = threshold
        
        return {
            'Feature': feature,
            'Comparison': comparison,
            'N_Normal': n_normal,
            'N_Disease': n_disease,
            'AUC': float(auc_corrected),
            'CI_Lower': float(ci_lower) if not np.isnan(ci_lower) else np.nan,
            'CI_Upper': float(ci_upper) if not np.isnan(ci_upper) else np.nan,
            'Optimal_Threshold': float(threshold_original),
            'Sensitivity': float(sensitivity),
            'Specificity': float(specificity),
            'PPV': float(ppv),
            'NPV': float(npv),
            'Accuracy': float(accuracy),
            'F1_Score': float(f1_score),
            'Youden_J': float(youden_j),
            'LR_Positive': float(lr_positive) if np.isfinite(lr_positive) else np.nan,
            'LR_Negative': float(lr_negative) if np.isfinite(lr_negative) else np.nan,
            'TP': int(tp),
            'TN': int(tn),
            'FP': int(fp),
            'FN': int(fn),
            'Is_Inverse': is_inverse  # Track if relationship was inverted
        }
    
    except Exception as e:
        import traceback
        error_msg = f"Feature {feature} for {comparison}: {str(e)}\n{traceback.format_exc()}"
        log_message(error_msg, error=True)
        return None

# ========================================
# PULSE COUNT SUMMARY (for manuscript reporting)
# ========================================

def summarize_pulse_counts(df, output_dir):
    """
    Report actual number of valid cardiac cycles per state.
    This is the base N before any feature-level NaN removal.
    Used for manuscript Methods section reporting.
    """
    log_message("\n" + "="*80)
    log_message("VALID CARDIAC CYCLE COUNTS (Base N per State)")
    log_message("="*80)
    
    # Total per state
    summary = (
        df.groupby('State')
        .size()
        .reset_index(name='N_Pulses')
        .sort_values('State')
    )
    
    total = summary['N_Pulses'].sum()
    
    log_message(f"\n{'State':<15} {'N Pulses':>10}")
    log_message("-" * 26)
    for _, row in summary.iterrows():
        log_message(f"  {row['State']:<13} {row['N_Pulses']:>10}")
    log_message("-" * 26)
    log_message(f"  {'Total':<13} {total:>10}")
    
    # Save CSV
    summary_file = os.path.join(output_dir, 'pulse_count_summary.csv')
    summary.to_csv(summary_file, index=False)
    log_message(f"\n✓ Saved pulse count summary: {summary_file}")
    
    # Also report per-feature valid N range (min/max after NaN removal)
    # Useful to confirm data quality across features
    metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
    feature_cols = [col for col in df.columns if col not in metadata_cols]
    
    valid_n_per_feature = (
        df[feature_cols + ['State']]
        .replace([np.inf, -np.inf], np.nan)
        .groupby('State')
        .apply(lambda g: g.drop(columns='State').notna().sum())
    )
    
    log_message("\nPer-feature valid N range (after NaN/Inf removal):")
    for state in valid_n_per_feature.index:
        row = valid_n_per_feature.loc[state]
        log_message(
            f"  {state:<13}: "
            f"min={row.min()}, max={row.max()}, "
            f"mean={row.mean():.1f}"
        )
    
    return summary

# ========================================
# MAIN ANALYSIS
# ========================================

def main():
    log_message("="*80)
    log_message("DIAGNOSTIC PERFORMANCE ANALYSIS: 15mm IR Channel (FULLY CORRECTED)")
    log_message("="*80)
    
    # Load master table
    if not os.path.exists(INPUT_FILE):
        log_message(f"ERROR: Input file not found: {INPUT_FILE}")
        return
    
    df = pd.read_csv(INPUT_FILE)
    log_message(f"✓ Loaded: {INPUT_FILE}")
    log_message(f"  Total rows: {len(df)}")
    log_message(f"  Total columns: {len(df.columns)}")
    pulse_counts = summarize_pulse_counts(df, OUTPUT_DIR)
    
    # Data distribution
    log_message("\nData Distribution:")
    for state in sorted(df['State'].unique()):
        count = len(df[df['State'] == state])
        log_message(f"  {state:12s}: {count} pulses")
    
    # Get feature columns (exclude metadata)
    metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
    feature_cols = [col for col in df.columns if col not in metadata_cols]
    
    log_message(f"\n✓ Total features to analyze: {len(feature_cols)}")
    
    # Analysis configurations
    comparisons = ['Ischaemia', 'Congestion']
    
    all_results = []
    success_count = 0
    error_count = 0
    inverse_count = 0
    
    for comparison in comparisons:
        log_message(f"\n{'='*80}")
        log_message(f"Analyzing: Normal vs {comparison}")
        log_message(f"{'='*80}")
        
        for i, feature in enumerate(feature_cols, 1):
            if i % 10 == 0:
                log_message(f"  Progress: {i}/{len(feature_cols)}")
            
            result = calculate_diagnostic_performance(df, feature, comparison)
            
            if result:
                all_results.append(result)
                success_count += 1
                if result['Is_Inverse']:
                    inverse_count += 1
            else:
                error_count += 1
    
    # Save results
    if all_results:
        df_results = pd.DataFrame(all_results)
        
        # Sort by AUC descending within each comparison
        df_results = df_results.sort_values(['Comparison', 'AUC'], 
                                            ascending=[True, False])
        
        # Save comprehensive results
        output_csv = os.path.join(OUTPUT_DIR, 'diagnostic_performance_15mm_IR_all.csv')
        df_results.to_csv(output_csv, index=False, float_format='%.6f')
        
        log_message("\n" + "="*80)
        log_message("RESULTS SUMMARY")
        log_message("="*80)
        log_message(f"✓ Output file: {output_csv}")
        log_message(f"✓ Total successful analyses: {success_count}")
        log_message(f"✓ Total failed analyses: {error_count}")
        log_message(f"✓ Inverse relationships corrected: {inverse_count}")
        log_message(f"✓ Success rate: {100*success_count/(success_count+error_count):.1f}%")
        
        # Top 10 features by comparison
        log_message("\n" + "="*80)
        log_message("TOP 10 FEATURES BY AUC")
        log_message("="*80)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            top10 = df_comp.nlargest(10, 'AUC')
            
            log_message(f"\n{comparison} (Normal vs {comparison}):")
            log_message("-"*80)
            
            for idx, row in top10.iterrows():
                inverse_marker = " [INV]" if row['Is_Inverse'] else ""
                log_message(
                    f"  {row['Feature']:30s}{inverse_marker:6s} | "
                    f"AUC: {row['AUC']:.4f} ({row['CI_Lower']:.4f}-{row['CI_Upper']:.4f}) | "
                    f"Sens: {row['Sensitivity']:.3f} | Spec: {row['Specificity']:.3f} | "
                    f"Acc: {row['Accuracy']:.3f}"
                )
        
        # Create separate files for each comparison
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            comp_file = os.path.join(OUTPUT_DIR, f'diagnostic_performance_15mm_IR_{comparison}.csv')
            df_comp.to_csv(comp_file, index=False, float_format='%.6f')
            log_message(f"\n✓ Saved {comparison} results: {comp_file} ({len(df_comp)} features)")
        
        # Create ranking tables (top 20)
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            df_rank = df_comp.nlargest(20, 'AUC')
            
            rank_file = os.path.join(OUTPUT_DIR, f'ranking_table_15mm_IR_{comparison}.csv')
            df_rank.to_csv(rank_file, index=False, float_format='%.6f')
            log_message(f"✓ Saved ranking: {rank_file}")
        
        # Statistical summary by feature category
        log_message("\n" + "="*80)
        log_message("FEATURE CATEGORY ANALYSIS")
        log_message("="*80)
        
        def categorize_feature(name):
            """Categorize features by type"""
            name_lower = name.lower()
            if any(x in name_lower for x in ['pi', 'amplitude', 'ac_', 'dc_']):
                return 'Intensity'
            elif any(x in name_lower for x in ['auc', 'area', 'datum']):
                return 'Area'
            elif any(x in name_lower for x in ['time', 'width', 'pw50', 'delta_t']):
                return 'Time'
            elif any(x in name_lower for x in ['slope', 'upslope', 'downslope', 'length']):
                return 'Slope'
            elif any(x in name_lower for x in ['sdppg', '_a', '_b', '_c', '_d', '_e', 'dnp']):
                return 'SDPPG'
            elif any(x in name_lower for x in ['skewness', 'kurtosis', 'variance', 'rwi', 'ipar']):
                return 'Shape & Stats'
            else:
                return 'Other'
        
        df_results['Category'] = df_results['Feature'].apply(categorize_feature)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            
            log_message(f"\n{comparison}:")
            log_message("-"*80)
            
            for category in sorted(df_comp['Category'].unique()):
                df_cat = df_comp[df_comp['Category'] == category]
                mean_auc = df_cat['AUC'].mean()
                median_auc = df_cat['AUC'].median()
                max_auc = df_cat['AUC'].max()
                n_inverse = df_cat['Is_Inverse'].sum()
                
                log_message(
                    f"  {category:15s}: "
                    f"N={len(df_cat):2d} | "
                    f"Mean AUC={mean_auc:.4f} | "
                    f"Median AUC={median_auc:.4f} | "
                    f"Max AUC={max_auc:.4f} | "
                    f"Inverse={n_inverse}"
                )
        
        # Inverse relationship summary
        log_message("\n" + "="*80)
        log_message("INVERSE RELATIONSHIP SUMMARY: 15mm IR Channel")
        log_message("="*80)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            df_inv = df_comp[df_comp['Is_Inverse'] == True]
            
            if len(df_inv) > 0:
                log_message(f"\n{comparison} - Features with inverse relationships ({len(df_inv)}):")
                for idx, row in df_inv.iterrows():
                    log_message(f"  {row['Feature']:30s} | AUC: {row['AUC']:.4f}")
        
        log_message("\n" + "="*80)
        log_message("✓ ANALYSIS COMPLETE")
        log_message(f"✓ Check {ERROR_LOG} for detailed error messages (if any)")
        log_message("="*80)
    
    else:
        log_message("\n⚠️  ERROR: No valid results generated. Check error log for details.")

if __name__ == "__main__":
    main()

In [ ]:
"""
Master Feature Extraction: 3mm Red&IR Combined Features
========================================================
Extracts 2-wavelength ratio features from synchronized 3mm Red and IR pulses
Combined features: PI_Ratio, Amplitude_Ratio, Delta_A_Ratio
Output format designed for integration with single-wavelength results
"""

import pandas as pd
import numpy as np
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ========================================
# CONFIGURATION
# ========================================
FS = 2000
TARGET_DEPTH = 3

STATE_MAP = {1: 'Normal', 2: 'Ischaemia', 3: 'Congestion'}

# Input files (must be generated first)
RED_INPUT = 'master_features_3mm_red/master_features_3mm_Red.csv'
IR_INPUT = 'master_features_3mm_ir/master_features_3mm_IR.csv'

OUTPUT_DIR = 'master_features_3mm_combined'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, 'processing_log.txt')

def log_message(message):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f"[{timestamp}] {message}"
    print(entry)
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(entry + '\n')

# ========================================
# COMBINED FEATURE CALCULATION
# ========================================

def calculate_combined_features(red_row, ir_row):
    """
    Calculate 2-wavelength combined features from synchronized Red and IR pulses
    
    Parameters:
    -----------
    red_row : pandas Series
        Feature values from Red channel (one pulse)
    ir_row : pandas Series
        Feature values from IR channel (one pulse)
    
    Returns:
    --------
    dict : Combined features
    """
    combined = {}
    
    # ===== 1. PI_Ratio (Perfusion Index Ratio) =====
    # Most commonly used 2-wavelength feature
    pi_red = red_row['PI']
    pi_ir = ir_row['PI']
    combined['PI_Ratio'] = pi_red / (pi_ir + 1e-10)
    
    # ===== 2. Amplitude_Ratio =====
    # Ratio of AC components
    amp_red = red_row['Amplitude']
    amp_ir = ir_row['Amplitude']
    combined['Amplitude_Ratio'] = amp_red / (amp_ir + 1e-10)
    
    # ===== 3. Delta_A_Ratio (Absorbance Change Ratio) =====
    # Based on Beer-Lambert law: ΔA = -log(I/I0)
    # ΔA_red / ΔA_ir ratio is theoretically related to SpO2
    
    # Calculate absorbance change for each wavelength
    # ΔA ≈ -log((DC - AC) / DC) = -log(1 - AC/DC)
    ac_red = red_row['AC_Component']
    dc_red = red_row['DC_Component']
    ac_ir = ir_row['AC_Component']
    dc_ir = ir_row['DC_Component']
    
    # Avoid log of negative or zero
    delta_a_red = -np.log((dc_red - ac_red) / (dc_red + 1e-10) + 1e-10) if dc_red > ac_red else 0
    delta_a_ir = -np.log((dc_ir - ac_ir) / (dc_ir + 1e-10) + 1e-10) if dc_ir > ac_ir else 0
    
    combined['Delta_A_Ratio'] = delta_a_red / (delta_a_ir + 1e-10)
    
    # ===== 4. AC_Component_Ratio =====
    combined['AC_Component_Ratio'] = ac_red / (ac_ir + 1e-10)
    
    # ===== 5. DC_Component_Ratio =====
    combined['DC_Component_Ratio'] = dc_red / (dc_ir + 1e-10)
    
    # ===== 6. AUC_Ratio (Area Under Curve Ratio) =====
    auc_red = red_row['AUC']
    auc_ir = ir_row['AUC']
    combined['AUC_Ratio'] = auc_red / (auc_ir + 1e-10)
    
    # ===== 7. Upslope_Ratio =====
    upslope_red = red_row['Upslope']
    upslope_ir = ir_row['Upslope']
    combined['Upslope_Ratio'] = upslope_red / (upslope_ir + 1e-10)
    
    # ===== 8. Downslope_Ratio =====
    downslope_red = red_row['Downslope']
    downslope_ir = ir_row['Downslope']
    combined['Downslope_Ratio'] = downslope_red / (downslope_ir + 1e-10)
    
    # ===== 9. Delta_T_Difference =====
    # Difference in timing between Red and IR
    delta_t_red = red_row['Delta_T']
    delta_t_ir = ir_row['Delta_T']
    combined['Delta_T_Difference'] = delta_t_red - delta_t_ir
    
    # ===== 10. Rise_Time_Ratio =====
    rise_red = red_row['Rise_Time']
    rise_ir = ir_row['Rise_Time']
    combined['Rise_Time_Ratio'] = rise_red / (rise_ir + 1e-10)
    
    # ===== 11. Width_Ratio_Difference =====
    width_ratio_red = red_row['Width_Ratio']
    width_ratio_ir = ir_row['Width_Ratio']
    combined['Width_Ratio_Difference'] = width_ratio_red - width_ratio_ir
    
    return combined

# ========================================
# MAIN PROCESSING
# ========================================

def main():
    log_message("="*80)
    log_message(f"COMBINED FEATURE EXTRACTION: {TARGET_DEPTH}mm Red&IR")
    log_message("="*80)
    
    # Load Red and IR master tables
    if not os.path.exists(RED_INPUT):
        log_message(f"ERROR: Red input file not found: {RED_INPUT}")
        log_message("Please run 3mm Red feature extraction first.")
        return
    
    if not os.path.exists(IR_INPUT):
        log_message(f"ERROR: IR input file not found: {IR_INPUT}")
        log_message("Please run 3mm IR feature extraction first.")
        return
    
    df_red = pd.read_csv(RED_INPUT)
    df_ir = pd.read_csv(IR_INPUT)
    
    log_message(f"✓ Loaded Red features: {len(df_red)} pulses")
    log_message(f"✓ Loaded IR features: {len(df_ir)} pulses")
    
    # Verify data structure
    log_message("\nRed Data Distribution:")
    for state in STATE_MAP.values():
        count = len(df_red[df_red['State'] == state])
        log_message(f"  {state:12s}: {count} pulses")
    
    log_message("\nIR Data Distribution:")
    for state in STATE_MAP.values():
        count = len(df_ir[df_ir['State'] == state])
        log_message(f"  {state:12s}: {count} pulses")
    
    # Synchronize Red and IR pulses
    log_message("\n" + "-"*80)
    log_message("Synchronizing Red and IR pulses...")
    log_message("-"*80)
    
    all_combined_features = []
    
    for state_code, state_name in STATE_MAP.items():
        # Filter by state
        df_red_state = df_red[df_red['State'] == state_name].reset_index(drop=True)
        df_ir_state = df_ir[df_ir['State'] == state_name].reset_index(drop=True)
        
        # Synchronize by pulse index (take minimum number of pulses)
        n_pulses = min(len(df_red_state), len(df_ir_state))
        
        if n_pulses == 0:
            log_message(f"⚠️  No synchronized pulses for {state_name}")
            continue
        
        log_message(f"  {state_name}: Synchronizing {n_pulses} pulses")
        
        for i in range(n_pulses):
            red_row = df_red_state.iloc[i]
            ir_row = df_ir_state.iloc[i]
            
            # Calculate combined features
            combined = calculate_combined_features(red_row, ir_row)
            
            # Add metadata
            combined['Depth'] = TARGET_DEPTH
            combined['Channel'] = 'Combined'
            combined['State'] = state_name
            combined['Pulse_Index'] = i
            
            all_combined_features.append(combined)
    
    # Save combined features
    if all_combined_features:
        df_combined = pd.DataFrame(all_combined_features)
        
        # Reorder columns: metadata first, then features
        metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
        feature_cols = [col for col in df_combined.columns if col not in metadata_cols]
        df_combined = df_combined[metadata_cols + sorted(feature_cols)]
        
        output_csv = os.path.join(OUTPUT_DIR, 'master_features_3mm_Combined.csv')
        df_combined.to_csv(output_csv, index=False)
        
        log_message("\n" + "="*80)
        log_message("SUMMARY")
        log_message("="*80)
        log_message(f"✓ Output file: {output_csv}")
        log_message(f"✓ Total synchronized pulses: {len(df_combined)}")
        log_message(f"✓ Total combined features: {len(feature_cols)}")
        
        log_message("\nData Distribution:")
        for state in STATE_MAP.values():
            count = len(df_combined[df_combined['State'] == state])
            log_message(f"  {state:12s}: {count} pulses")
        
        log_message(f"\nExtracted Combined Features ({len(feature_cols)}):")
        for feat in sorted(feature_cols):
            log_message(f"  - {feat}")
        
        # Feature statistics
        log_message("\n" + "-"*80)
        log_message("Feature Statistics (Normal state sample):")
        log_message("-"*80)
        
        df_normal = df_combined[df_combined['State'] == 'Normal']
        if len(df_normal) > 0:
            for feat in sorted(feature_cols):
                mean_val = df_normal[feat].mean()
                std_val = df_normal[feat].std()
                log_message(f"  {feat:30s}: Mean={mean_val:8.4f}, Std={std_val:8.4f}")
        
        log_message("\n" + "="*80)
        log_message("✓ COMBINED FEATURE EXTRACTION COMPLETE")
        log_message("="*80)
    else:
        log_message("\n⚠️  ERROR: No synchronized pulses found.")

if __name__ == "__main__":
    main()


In [ ]:
"""
Diagnostic Performance Analysis: 3mm Combined (Red&IR) Features
================================================================
Calculate comprehensive diagnostic performance metrics for 2-wavelength combined features
Comparisons: Normal vs Ischaemia, Normal vs Congestion
Output: Detailed performance tables with AUC, sensitivity, specificity, etc.
"""

import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from scipy import stats
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ========================================
# CONFIGURATION
# ========================================
INPUT_FILE = 'master_features_3mm_combined/master_features_3mm_Combined.csv'
OUTPUT_DIR = 'diagnostic_performance_3mm_combined'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, 'performance_analysis_log.txt')
ERROR_LOG = os.path.join(OUTPUT_DIR, 'error_details.txt')

def log_message(message, error=False):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f"[{timestamp}] {message}"
    print(entry)
    
    log_file = ERROR_LOG if error else LOG_FILE
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write(entry + '\n')

# ========================================
# PERFORMANCE METRICS CALCULATION
# ========================================

def calculate_auc_ci(y_true, y_pred, confidence=0.95, n_bootstraps=1000):
    """
    Calculate AUC confidence interval using bootstrap method
    FIXED: Convert to numpy arrays to avoid pandas index issues
    """
    if len(np.unique(y_true)) < 2:
        return np.nan, np.nan
    
    # Convert to numpy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    try:
        auc_base = roc_auc_score(y_true, y_pred)
    except:
        return np.nan, np.nan
    
    # Bootstrap
    np.random.seed(42)
    auc_scores = []
    n_samples = len(y_true)
    
    for i in range(n_bootstraps):
        indices = np.random.choice(n_samples, n_samples, replace=True)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]
        
        if len(np.unique(y_true_boot)) < 2:
            continue
        
        try:
            auc_boot = roc_auc_score(y_true_boot, y_pred_boot)
            auc_scores.append(auc_boot)
        except:
            continue
    
    if len(auc_scores) < 10:
        return np.nan, np.nan
    
    alpha = (1 - confidence) / 2
    ci_lower = np.percentile(auc_scores, alpha * 100)
    ci_upper = np.percentile(auc_scores, (1 - alpha) * 100)
    
    return ci_lower, ci_upper

def calculate_optimal_threshold(y_true, y_pred):
    """
    Calculate optimal threshold using Youden's J statistic
    """
    try:
        fpr, tpr, thresholds = roc_curve(y_true, y_pred)
        j_scores = tpr - fpr
        optimal_idx = np.argmax(j_scores)
        return thresholds[optimal_idx]
    except:
        return np.median(y_pred)

def calculate_diagnostic_performance(df, feature, comparison='Ischaemia'):
    """
    Calculate comprehensive diagnostic performance metrics
    FULLY CORRECTED: Handles inverse relationships properly
    """
    try:
        # Filter data: Normal vs comparison state
        df_filtered = df[df['State'].isin(['Normal', comparison])].copy()
        
        # Check if feature exists and has valid data
        if feature not in df_filtered.columns:
            log_message(f"Feature {feature} not found", error=True)
            return None
        
        # Remove NaN and infinite values
        df_clean = df_filtered[[feature, 'State']].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(df_clean) < 10:
            log_message(f"Feature {feature}: insufficient data (N={len(df_clean)})", error=True)
            return None
        
        if len(df_clean['State'].unique()) < 2:
            log_message(f"Feature {feature}: only one class present", error=True)
            return None
        
        # Binary labels
        y_true = (df_clean['State'] == comparison).astype(int)
        y_pred_original = df_clean[feature].values
        
        # Check for variance
        if np.std(y_pred_original) < 1e-10:
            log_message(f"Feature {feature}: zero variance", error=True)
            return None
        
        # Sample sizes
        n_normal = int((y_true == 0).sum())
        n_disease = int((y_true == 1).sum())
        
        # ===== Handle inverse relationships =====
        auc_raw = roc_auc_score(y_true, y_pred_original)
        
        if auc_raw < 0.5:
            auc_corrected = 1 - auc_raw
            y_pred_corrected = -y_pred_original
            is_inverse = True
        else:
            auc_corrected = auc_raw
            y_pred_corrected = y_pred_original
            is_inverse = False
        
        # Use corrected predictions for all downstream calculations
        ci_lower, ci_upper = calculate_auc_ci(y_true, y_pred_corrected)
        threshold = calculate_optimal_threshold(y_true, y_pred_corrected)
        
        # Binary predictions
        y_pred_binary = (y_pred_corrected >= threshold).astype(int)
        
        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary).ravel()
        
        # Calculate metrics
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        f1_score = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
        
        # Likelihood ratios
        if specificity < 1.0 and specificity > 0:
            lr_positive = sensitivity / (1 - specificity)
        elif specificity == 1.0 and sensitivity > 0:
            lr_positive = np.inf
        else:
            lr_positive = np.nan
        
        if specificity > 0 and sensitivity < 1.0:
            lr_negative = (1 - sensitivity) / specificity
        elif sensitivity == 1.0:
            lr_negative = 0.0
        else:
            lr_negative = np.nan
        
        # Youden's J statistic
        youden_j = sensitivity + specificity - 1
        
        # Convert threshold back to original scale if inverse
        if is_inverse:
            threshold_original = -threshold
        else:
            threshold_original = threshold
        
        return {
            'Feature': feature,
            'Comparison': comparison,
            'N_Normal': n_normal,
            'N_Disease': n_disease,
            'AUC': float(auc_corrected),
            'CI_Lower': float(ci_lower) if not np.isnan(ci_lower) else np.nan,
            'CI_Upper': float(ci_upper) if not np.isnan(ci_upper) else np.nan,
            'Optimal_Threshold': float(threshold_original),
            'Sensitivity': float(sensitivity),
            'Specificity': float(specificity),
            'PPV': float(ppv),
            'NPV': float(npv),
            'Accuracy': float(accuracy),
            'F1_Score': float(f1_score),
            'Youden_J': float(youden_j),
            'LR_Positive': float(lr_positive) if np.isfinite(lr_positive) else np.nan,
            'LR_Negative': float(lr_negative) if np.isfinite(lr_negative) else np.nan,
            'TP': int(tp),
            'TN': int(tn),
            'FP': int(fp),
            'FN': int(fn),
            'Is_Inverse': is_inverse
        }
    
    except Exception as e:
        import traceback
        error_msg = f"Feature {feature} for {comparison}: {str(e)}\n{traceback.format_exc()}"
        log_message(error_msg, error=True)
        return None

# ========================================
# PULSE COUNT SUMMARY (for manuscript reporting)
# ========================================

def summarize_pulse_counts(df, output_dir):
    """
    Report actual number of valid cardiac cycles per state.
    This is the base N before any feature-level NaN removal.
    Used for manuscript Methods section reporting.
    """
    log_message("\n" + "="*80)
    log_message("VALID CARDIAC CYCLE COUNTS (Base N per State)")
    log_message("="*80)
    
    # Total per state
    summary = (
        df.groupby('State')
        .size()
        .reset_index(name='N_Pulses')
        .sort_values('State')
    )
    
    total = summary['N_Pulses'].sum()
    
    log_message(f"\n{'State':<15} {'N Pulses':>10}")
    log_message("-" * 26)
    for _, row in summary.iterrows():
        log_message(f"  {row['State']:<13} {row['N_Pulses']:>10}")
    log_message("-" * 26)
    log_message(f"  {'Total':<13} {total:>10}")
    
    # Save CSV
    summary_file = os.path.join(output_dir, 'pulse_count_summary.csv')
    summary.to_csv(summary_file, index=False)
    log_message(f"\n✓ Saved pulse count summary: {summary_file}")
    
    # Also report per-feature valid N range (min/max after NaN removal)
    # Useful to confirm data quality across features
    metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
    feature_cols = [col for col in df.columns if col not in metadata_cols]
    
    valid_n_per_feature = (
        df[feature_cols + ['State']]
        .replace([np.inf, -np.inf], np.nan)
        .groupby('State')
        .apply(lambda g: g.drop(columns='State').notna().sum())
    )
    
    log_message("\nPer-feature valid N range (after NaN/Inf removal):")
    for state in valid_n_per_feature.index:
        row = valid_n_per_feature.loc[state]
        log_message(
            f"  {state:<13}: "
            f"min={row.min()}, max={row.max()}, "
            f"mean={row.mean():.1f}"
        )
    
    return summary

# ========================================
# MAIN ANALYSIS
# ========================================

def main():
    log_message("="*80)
    log_message("DIAGNOSTIC PERFORMANCE ANALYSIS: 3mm Combined (Red&IR)")
    log_message("="*80)
    
    # Load master table
    if not os.path.exists(INPUT_FILE):
        log_message(f"ERROR: Input file not found: {INPUT_FILE}")
        log_message("Please run 3mm Combined feature extraction first.")
        return
    
    df = pd.read_csv(INPUT_FILE)
    log_message(f"✓ Loaded: {INPUT_FILE}")
    log_message(f"  Total rows: {len(df)}")
    log_message(f"  Total columns: {len(df.columns)}")
    pulse_counts = summarize_pulse_counts(df, OUTPUT_DIR)
    
    # Data distribution
    log_message("\nData Distribution:")
    for state in sorted(df['State'].unique()):
        count = len(df[df['State'] == state])
        log_message(f"  {state:12s}: {count} pulses")
    
    # Get feature columns (exclude metadata)
    metadata_cols = ['Depth', 'Channel', 'State', 'Pulse_Index']
    feature_cols = [col for col in df.columns if col not in metadata_cols]
    
    log_message(f"\n✓ Total combined features to analyze: {len(feature_cols)}")
    
    # Analysis configurations
    comparisons = ['Ischaemia', 'Congestion']
    
    all_results = []
    success_count = 0
    error_count = 0
    inverse_count = 0
    
    for comparison in comparisons:
        log_message(f"\n{'='*80}")
        log_message(f"Analyzing: Normal vs {comparison}")
        log_message(f"{'='*80}")
        
        for feature in feature_cols:
            result = calculate_diagnostic_performance(df, feature, comparison)
            
            if result:
                all_results.append(result)
                success_count += 1
                if result['Is_Inverse']:
                    inverse_count += 1
            else:
                error_count += 1
    
    # Save results
    if all_results:
        df_results = pd.DataFrame(all_results)
        
        # Sort by AUC descending within each comparison
        df_results = df_results.sort_values(['Comparison', 'AUC'], 
                                            ascending=[True, False])
        
        # Save comprehensive results
        output_csv = os.path.join(OUTPUT_DIR, 'diagnostic_performance_3mm_Combined_all.csv')
        df_results.to_csv(output_csv, index=False, float_format='%.6f')
        
        log_message("\n" + "="*80)
        log_message("RESULTS SUMMARY")
        log_message("="*80)
        log_message(f"✓ Output file: {output_csv}")
        log_message(f"✓ Total successful analyses: {success_count}")
        log_message(f"✓ Total failed analyses: {error_count}")
        log_message(f"✓ Inverse relationships corrected: {inverse_count}")
        log_message(f"✓ Success rate: {100*success_count/(success_count+error_count):.1f}%")
        
        # Top features by comparison
        log_message("\n" + "="*80)
        log_message("COMBINED FEATURES RANKED BY AUC")
        log_message("="*80)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            
            log_message(f"\n{comparison} (Normal vs {comparison}):")
            log_message("-"*80)
            
            for idx, row in df_comp.iterrows():
                inverse_marker = " [INV]" if row['Is_Inverse'] else ""
                log_message(
                    f"  {row['Feature']:30s}{inverse_marker:6s} | "
                    f"AUC: {row['AUC']:.4f} ({row['CI_Lower']:.4f}-{row['CI_Upper']:.4f}) | "
                    f"Sens: {row['Sensitivity']:.3f} | Spec: {row['Specificity']:.3f} | "
                    f"Acc: {row['Accuracy']:.3f}"
                )
        
        # Create separate files for each comparison
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            comp_file = os.path.join(OUTPUT_DIR, f'diagnostic_performance_3mm_Combined_{comparison}.csv')
            df_comp.to_csv(comp_file, index=False, float_format='%.6f')
            log_message(f"\n✓ Saved {comparison} results: {comp_file} ({len(df_comp)} features)")
        
        # Create ranking tables (all features, sorted by AUC)
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            
            rank_file = os.path.join(OUTPUT_DIR, f'ranking_table_3mm_Combined_{comparison}.csv')
            df_comp.to_csv(rank_file, index=False, float_format='%.6f')
            log_message(f"✓ Saved ranking: {rank_file}")
        
        # Feature performance summary
        log_message("\n" + "="*80)
        log_message("FEATURE PERFORMANCE SUMMARY")
        log_message("="*80)
        
        for comparison in comparisons:
            df_comp = df_results[df_results['Comparison'] == comparison]
            
            # Count features by AUC range
            excellent = len(df_comp[df_comp['AUC'] >= 0.9])
            good = len(df_comp[(df_comp['AUC'] >= 0.8) & (df_comp['AUC'] < 0.9)])
            fair = len(df_comp[(df_comp['AUC'] >= 0.7) & (df_comp['AUC'] < 0.8)])
            poor = len(df_comp[df_comp['AUC'] < 0.7])
            
            log_message(f"\n{comparison}:")
            log_message(f"  Excellent (AUC ≥ 0.9): {excellent} features")
            log_message(f"  Good (0.8 ≤ AUC < 0.9): {good} features")
            log_message(f"  Fair (0.7 ≤ AUC < 0.8): {fair} features")
            log_message(f"  Poor (AUC < 0.7): {poor} features")
            
            # Best feature
            best = df_comp.iloc[0]
            log_message(f"\n  Best feature: {best['Feature']}")
            log_message(f"    AUC: {best['AUC']:.4f}")
            log_message(f"    Sensitivity: {best['Sensitivity']:.3f}")
            log_message(f"    Specificity: {best['Specificity']:.3f}")
            log_message(f"    Accuracy: {best['Accuracy']:.3f}")
        
        # Comparison with single-wavelength results
        log_message("\n" + "="*80)
        log_message("NOTES")
        log_message("="*80)
        log_message("Combined features represent 2-wavelength (Red&IR) interactions.")
        log_message("Compare these results with single-wavelength (Red-only, IR-only) to assess")
        log_message("whether combined features provide superior diagnostic performance.")
        log_message("\nKey combined features:")
        log_message("  - PI_Ratio: Standard 2-wavelength perfusion index ratio")
        log_message("  - Delta_A_Ratio: Absorbance change ratio (SpO2-related)")
        log_message("  - Amplitude_Ratio: AC component ratio")
        
        log_message("\n" + "="*80)
        log_message("✓ ANALYSIS COMPLETE")
        log_message(f"✓ Check {ERROR_LOG} for detailed error messages (if any)")
        log_message("="*80)
    
    else:
        log_message("\n⚠️  ERROR: No valid results generated. Check error log for details.")

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ===========================
# Configuration
# ===========================
INPUT_FILES = {
    '3mm_Red': 'diagnostic_performance_3mm_Red_all.csv',
    '3mm_IR': 'diagnostic_performance_3mm_IR_all.csv',
    '9mm_IR': 'diagnostic_performance_9mm_IR_all.csv',
    '15mm_IR': 'diagnostic_performance_15mm_IR_all.csv',
    '3mm_Combined': 'diagnostic_performance_3mm_Combined_all.csv'
}

OUTPUT_DIR = Path('integrated_depth_comparison')
OUTPUT_DIR.mkdir(exist_ok=True)

STATE_MAP = {1: 'Normal', 2: 'Ischaemia', 3: 'Congestion'}
COMPARISONS = ['Ischaemia', 'Congestion']

# Category colors
CATEGORY_COLORS = {
    'Intensity': '#FFB74D',      # Light orange
    'Area': '#FFF176',           # Light yellow
    'Time': '#4FC3F7',           # Light cyan
    'Slope': '#BA68C8',          # Light purple
    'SDPPG': '#81C784',          # Light green
    'Shape & Stats': '#BDBDBD',  # Light gray
    'Other': "#fcf6dc"           # White
}

# Wavelength colors for markers
WAVELENGTH_COLORS = {
    'Red': '#D32F2F',
    'IR': '#1976D2',
    'Red&IR': '#7B1FA2'  # Combined (purple)
}

# ===========================
# Helper Functions
# ===========================
def get_category(feature_name):
    """Assign category based on feature name."""
    intensity_keywords = ['PI', 'AC_Component', 'DC_Component', 'Amplitude']
    area_keywords = ['AUC', 'Area', 'Datum']
    time_keywords = ['Time', 'Width', 'PW50']
    slope_keywords = ['slope', 'Slope']
    sdppg_keywords = ['SDPPG', 'DNP']
    shape_keywords = ['Skewness', 'Kurtosis', 'Variance', 'RWI', 'IPAR']
    
    fname = feature_name
    if any(k in fname for k in intensity_keywords):
        return 'Intensity'
    elif any(k in fname for k in area_keywords):
        return 'Area'
    elif any(k in fname for k in time_keywords):
        return 'Time'
    elif any(k in fname for k in slope_keywords):
        return 'Slope'
    elif any(k in fname for k in sdppg_keywords):
        return 'SDPPG'
    elif any(k in fname for k in shape_keywords):
        return 'Shape & Stats'
    else:
        return 'Other'

def get_source(feature_name):
    """Get wavelength source from feature name."""
    if '_Ratio' in feature_name or '_Difference' in feature_name:
        return 'Red&IR'
    # Otherwise infer from loaded file context (handled in load)
    return 'Unknown'

def clean_feature_name(feature_name):
    """Remove suffix like _Ischaemia or _Congestion."""
    for comp in COMPARISONS:
        if feature_name.endswith(f'_{comp}'):
            return feature_name[:-len(f'_{comp}')]
    return feature_name

def load_and_filter_data(file_path, depth, source, comparison):
    """
    Load CSV and filter for given comparison (Ischaemia or Congestion).
    Add Depth, Source, Category, Display_Name columns.
    """
    df = pd.read_csv(file_path)
    
    # Filter by comparison
    df = df[df['Comparison'] == comparison].copy()
    
    # Add metadata
    df['Depth'] = depth
    df['Source'] = source
    df['Category'] = df['Feature'].apply(get_category)
    df['Clean_Feature'] = df['Feature'].apply(clean_feature_name)
    df['Display_Name'] = df['Clean_Feature']
    
    # Exclude DC_Component  ← ここに挿入
    df = df[~df['Clean_Feature'].str.contains('DC_Component', na=False)]

    return df

# ===========================
# Load and Integrate Data
# ===========================
print("Loading diagnostic performance files...")

all_data = []

# 3mm Red
if Path(INPUT_FILES['3mm_Red']).exists():
    for comp in COMPARISONS:
        df_red = load_and_filter_data(INPUT_FILES['3mm_Red'], depth=3, source='Red', comparison=comp)
        all_data.append(df_red)
        print(f"  Loaded 3mm Red {comp}: {len(df_red)} features")

# 3mm IR
if Path(INPUT_FILES['3mm_IR']).exists():
    for comp in COMPARISONS:
        df_ir = load_and_filter_data(INPUT_FILES['3mm_IR'], depth=3, source='IR', comparison=comp)
        all_data.append(df_ir)
        print(f"  Loaded 3mm IR {comp}: {len(df_ir)} features")

# 3mm Combined
if Path(INPUT_FILES['3mm_Combined']).exists():
    for comp in COMPARISONS:
        df_comb = load_and_filter_data(INPUT_FILES['3mm_Combined'], depth=3, source='Red&IR', comparison=comp)
        all_data.append(df_comb)
        print(f"  Loaded 3mm Combined {comp}: {len(df_comb)} features")

# 9mm IR
if Path(INPUT_FILES['9mm_IR']).exists():
    for comp in COMPARISONS:
        df_9ir = load_and_filter_data(INPUT_FILES['9mm_IR'], depth=9, source='IR', comparison=comp)
        all_data.append(df_9ir)
        print(f"  Loaded 9mm IR {comp}: {len(df_9ir)} features")

# 15mm IR
if Path(INPUT_FILES['15mm_IR']).exists():
    for comp in COMPARISONS:
        df_15ir = load_and_filter_data(INPUT_FILES['15mm_IR'], depth=15, source='IR', comparison=comp)
        all_data.append(df_15ir)
        print(f"  Loaded 15mm IR {comp}: {len(df_15ir)} features")

# Concatenate all
df_all = pd.concat(all_data, ignore_index=True)

# Save integrated data
integrated_csv = OUTPUT_DIR / 'diagnostic_performance_integrated_all.csv'
df_all.to_csv(integrated_csv, index=False)
print(f"\nIntegrated data saved to: {integrated_csv}")
print(f"Total rows: {len(df_all)}")
print(f"Columns: {list(df_all.columns)}")

# ===========================
# Top 10 Ranking per Depth/Comparison
# ===========================
print("\n" + "="*60)
print("Generating Top 10 Rankings")
print("="*60)

for comp in COMPARISONS:
    for depth in [3, 9, 15]:
        # Filter by depth and comparison
        df_subset = df_all[(df_all['Depth'] == depth) & (df_all['Comparison'] == comp)].copy()
        
        if len(df_subset) == 0:
            continue
        
        # Sort by AUC descending
        df_sorted = df_subset.sort_values('AUC', ascending=False).reset_index(drop=True)
        
        # Top 10
        df_top10 = df_sorted.head(10).copy()
        
        # Save ranking table
        ranking_file = OUTPUT_DIR / f'ranking_table_{comp}_{depth}mm_Top10.csv'
        df_top10.to_csv(ranking_file, index=False)
        
        print(f"\n{comp} - {depth}mm - Top 10 Features (by AUC):")
        print(f"  File: {ranking_file}")
        for idx, row in df_top10.iterrows():
            print(f"    {idx+1}. {row['Display_Name']:30s} | Source: {row['Source']:8s} | AUC: {row['AUC']:.4f} | Category: {row['Category']}")

print("\n" + "="*60)
print("Integration Complete!")
print("="*60)

# ===========================
# Visualization: Depth Comparison Bar Chart
# ===========================
print("\nGenerating Depth Comparison Visualization...")

def plot_combined_depth_comparison():
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Depth Comparison Visualization: Top 10 Features by AUC', 
                 fontsize=18, weight='bold', fontname='Arial')
    
    depths = [3, 9, 15]
    comparisons_plot = ['Ischaemia', 'Congestion']
    
    all_handles = []
    all_labels = []
    
    for row_idx, comp in enumerate(comparisons_plot):
        for col_idx, depth in enumerate(depths):
            ax = axes[row_idx, col_idx]
            
            # Filter data
            df_plot = df_all[(df_all['Depth'] == depth) & (df_all['Comparison'] == comp)].copy()
            
            if len(df_plot) == 0:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', fontsize=14, fontname='Arial')
                ax.set_xlim(0.5, 1.0)
                ax.set_ylim(0, 1)
                ax.set_title(f'{depth}mm - {comp}', fontsize=14, weight='bold', fontname='Arial')
                continue
            
            # ===== All depths: Group by AUC + Category + Source, Top 10 =====
            # Round AUC to avoid floating point precision issues
            df_plot['AUC_rounded'] = df_plot['AUC'].round(4)
            
            # Group by AUC + Category + Source
            df_cat = df_plot.groupby(['AUC_rounded', 'Category', 'Source']).agg({
                'AUC': 'mean',
                'CI_Lower': 'mean',
                'CI_Upper': 'mean',
                'Clean_Feature': lambda x: ', '.join(sorted(x)),
                'Feature': 'count'
            }).reset_index()
            df_cat.rename(columns={'Feature': 'N_Features', 'Clean_Feature': 'Feature_List'}, inplace=True)
            
            # Sort by AUC descending and take top 10
            df_cat = df_cat.sort_values('AUC', ascending=False).head(10).reset_index(drop=True)
            
            # Create display names and annotations
            display_names = []
            annotations = []
            
            for idx, row in df_cat.iterrows():
                n = int(row['N_Features'])
                category = row['Category']
                source = row['Source']
                feature_list = row['Feature_List']
                
                if n == 1:
                    # Single feature: show feature name
                    display_name = feature_list
                    annotation = None
                else:
                    # Multiple features: show "Category (n=X)"
                    display_name = f"{category} (n={n})"
                    annotation = f"{category} {source}: {feature_list}"
                
                display_names.append(display_name)
                annotations.append(annotation)
            
            df_cat['Display_Name'] = display_names
            df_cat['Annotation'] = annotations
            
            # Determine x-axis range - ensure CI_Lower is included
            ci_lower_min = df_cat['CI_Lower'].min()
            x_min = max(0.5, ci_lower_min - 0.05)
            # Round down to nearest 0.1 for cleaner axis
            x_min = np.floor(x_min * 10) / 10
            x_max = 1.0
            
            # Plot bars
            y_positions = np.arange(len(df_cat))
            
            for i, row in df_cat.iterrows():
                category = row['Category']
                source = row['Source']
                auc = row['AUC']
                ci_lower = max(row['CI_Lower'], x_min)  # Clamp to x_min
                ci_upper = min(row['CI_Upper'], x_max)  # Clamp to x_max
                
                # Horizontal bar with category color - start from x_min
                bar_color = CATEGORY_COLORS.get(category, CATEGORY_COLORS['Other'])
                ax.barh(i, auc - x_min, left=x_min, height=0.6, 
                       color=bar_color, alpha=0.7, edgecolor='black', linewidth=0.5)
                
                # Error bar with source color marker - clamp to visible range
                xerr = [[auc - ci_lower], [ci_upper - auc]]
                marker_color = WAVELENGTH_COLORS.get(source, '#000000')
                ax.errorbar(auc, i, xerr=xerr, fmt='o', color=marker_color, 
                           markersize=6, capsize=4, capthick=1.5, elinewidth=1.5)
            
            # Y-axis labels
            ax.set_yticks(y_positions)
            ax.set_yticklabels(df_cat['Display_Name'], fontsize=9, fontname='Arial')
            ax.invert_yaxis()
            
            # X-axis - use dynamic x_min
            ax.set_xlim(x_min, x_max)
            # Generate ticks from x_min to x_max in 0.1 increments
            x_ticks = np.arange(x_min, x_max + 0.01, 0.1)
            ax.set_xticks(x_ticks)
            ax.set_xticklabels([f'{x:.1f}' for x in x_ticks], fontsize=10, fontname='Arial')
            ax.set_xlabel('AUC', fontsize=11, weight='bold', fontname='Arial')
            
            # Title
            ax.set_title(f'{depth}mm - {comp}', fontsize=14, weight='bold', fontname='Arial')
            ax.grid(axis='x', linestyle='--', alpha=0.3)
            
            # Add annotations below the plot
            annotation_text = []
            for annot in df_cat['Annotation']:
                if annot is not None:
                    annotation_text.append(annot)
            
            if annotation_text:
                annot_str = '\n'.join(annotation_text)
                ax.text(0.02, -0.15, annot_str, transform=ax.transAxes,
                       fontsize=7, fontname='Arial', verticalalignment='top',
                       horizontalalignment='left', wrap=True)
            
            # Collect legend handles (only once)
            if row_idx == 0 and col_idx == 0:
                # Category colors
                for cat, color in CATEGORY_COLORS.items():
                    handle = plt.Line2D([0], [0], marker='s', color='w', 
                                       markerfacecolor=color, markersize=10, label=cat)
                    all_handles.append(handle)
                    all_labels.append(cat)
                
                # Wavelength markers
                for source, color in WAVELENGTH_COLORS.items():
                    handle = plt.Line2D([0], [0], marker='o', color='w', 
                                       markerfacecolor=color, markersize=8, 
                                       markeredgecolor='black', markeredgewidth=0.5, label=source)
                    all_handles.append(handle)
                    all_labels.append(source)
    
    # Unified legend on the right
    fig.legend(all_handles, all_labels, loc='center left', bbox_to_anchor=(0.88, 0.5), 
              frameon=True, fontsize=11, title='Legend', title_fontsize=12)
    
    plt.tight_layout(rect=[0, 0, 0.88, 0.96])
    
    # Save figure
    output_file = OUTPUT_DIR / 'depth_comparison_combined_all_bar.png'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"Visualization saved to: {output_file}")
    plt.close()

plot_combined_depth_comparison()







print("\n" + "="*60)
print("All tasks completed successfully!")
print(f"Output directory: {OUTPUT_DIR}")
print("="*60)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')


# ===========================
# Sci Rep準拠の設定
# ===========================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['legend.title_fontsize'] = 9


# 線幅設定
plt.rcParams['lines.linewidth'] = 1.0
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['ytick.major.width'] = 1.0
plt.rcParams['patch.linewidth'] = 1.0


# PDF出力設定
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


# ===========================
# Configuration
# ===========================
INPUT_FILES = {
    '3mm_Red': 'diagnostic_performance_3mm_Red_all.csv',
    '3mm_IR': 'diagnostic_performance_3mm_IR_all.csv',
    '9mm_IR': 'diagnostic_performance_9mm_IR_all.csv',
    '15mm_IR': 'diagnostic_performance_15mm_IR_all.csv',
    '3mm_Combined': 'diagnostic_performance_3mm_Combined_all.csv'
}


OUTPUT_DIR = Path('integrated_depth_comparison')
OUTPUT_DIR.mkdir(exist_ok=True)


STATE_MAP = {1: 'Normal', 2: 'Ischaemia', 3: 'Congestion'}
COMPARISONS = ['Ischaemia', 'Congestion']


# Category colors
CATEGORY_COLORS = {
    'Intensity': '#FFB74D',
    'Area': '#FFF176',
    'Time': '#4FC3F7',
    'Slope': '#BA68C8',
    'SDPPG': '#81C784',
    'Shape & Stats': '#BDBDBD',
    'Combined': "#fcf6dc"
}


# Wavelength colors
WAVELENGTH_COLORS = {
    'Red': '#D32F2F',
    'IR': '#1976D2',
    'Red&IR': '#7B1FA2'
}


# ===========================
# Helper Functions
# ===========================
def get_category(feature_name):
    intensity_keywords = ['PI', 'AC_Component', 'DC_Component', 'Amplitude']
    area_keywords = ['AUC', 'Area', 'Datum']
    time_keywords = ['Time', 'Width', 'PW50']
    slope_keywords = ['slope', 'Slope']
    sdppg_keywords = ['SDPPG', 'DNP']
    shape_keywords = ['Skewness', 'Kurtosis', 'Variance', 'RWI', 'IPAR']
    
    fname = feature_name
    if any(k in fname for k in intensity_keywords):
        return 'Intensity'
    elif any(k in fname for k in area_keywords):
        return 'Area'
    elif any(k in fname for k in time_keywords):
        return 'Time'
    elif any(k in fname for k in slope_keywords):
        return 'Slope'
    elif any(k in fname for k in sdppg_keywords):
        return 'SDPPG'
    elif any(k in fname for k in shape_keywords):
        return 'Shape & Stats'
    else:
        return 'Combined'


def clean_feature_name(feature_name):
    for comp in COMPARISONS:
        if feature_name.endswith(f'_{comp}'):
            return feature_name[:-len(f'_{comp}')]
    return feature_name


def load_and_filter_data(file_path, depth, source, comparison):
    df = pd.read_csv(file_path)
    df = df[df['Comparison'] == comparison].copy()
    
    df['Depth'] = depth
    df['Source'] = source
    df['Category'] = df['Feature'].apply(get_category)
    df['Clean_Feature'] = df['Feature'].apply(clean_feature_name)
    df['Display_Name'] = df['Clean_Feature']
    
    # Exclude DC_Component
    df = df[~df['Clean_Feature'].str.contains('DC_Component', na=False)]
    
    return df


# ===========================
# Load Data
# ===========================
print("Loading diagnostic performance files...")


all_data = []


if Path(INPUT_FILES['3mm_Red']).exists():
    for comp in COMPARISONS:
        df_red = load_and_filter_data(INPUT_FILES['3mm_Red'], depth=3, source='Red', comparison=comp)
        all_data.append(df_red)


if Path(INPUT_FILES['3mm_IR']).exists():
    for comp in COMPARISONS:
        df_ir = load_and_filter_data(INPUT_FILES['3mm_IR'], depth=3, source='IR', comparison=comp)
        all_data.append(df_ir)


if Path(INPUT_FILES['3mm_Combined']).exists():
    for comp in COMPARISONS:
        df_comb = load_and_filter_data(INPUT_FILES['3mm_Combined'], depth=3, source='Red&IR', comparison=comp)
        all_data.append(df_comb)


if Path(INPUT_FILES['9mm_IR']).exists():
    for comp in COMPARISONS:
        df_9ir = load_and_filter_data(INPUT_FILES['9mm_IR'], depth=9, source='IR', comparison=comp)
        all_data.append(df_9ir)


if Path(INPUT_FILES['15mm_IR']).exists():
    for comp in COMPARISONS:
        df_15ir = load_and_filter_data(INPUT_FILES['15mm_IR'], depth=15, source='IR', comparison=comp)
        all_data.append(df_15ir)


df_all = pd.concat(all_data, ignore_index=True)


# ===========================
# Visualization
# ===========================
print("\nGenerating Depth Comparison Visualization...")


def plot_combined_depth_comparison():
    # ダブルカラム幅、より広く調整
    fig_width = 7.2
    fig_height = 6.5
    
    fig, axes = plt.subplots(2, 3, figsize=(fig_width, fig_height))
    
    depths = [3, 9, 15]
    comparisons_plot = ['Ischaemia', 'Congestion']
    
    all_handles = []
    all_labels = []
    
    # 注釈を格納する辞書（グラフ外出力用）
    annotations_dict = {}
    
    for row_idx, comp in enumerate(comparisons_plot):
        for col_idx, depth in enumerate(depths):
            ax = axes[row_idx, col_idx]
            
            df_plot = df_all[(df_all['Depth'] == depth) & (df_all['Comparison'] == comp)].copy()
            
            if len(df_plot) == 0:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', fontsize=8)
                ax.set_xlim(0.5, 1.0)
                ax.set_ylim(0, 1)
                ax.set_title(f'{depth} mm - {comp}', fontsize=10)
                continue
            
            # Group by AUC + Category + Source
            df_plot['AUC_rounded'] = df_plot['AUC'].round(4)
            
            df_cat = df_plot.groupby(['AUC_rounded', 'Category', 'Source']).agg({
                'AUC': 'mean',
                'CI_Lower': 'mean',
                'CI_Upper': 'mean',
                'Clean_Feature': lambda x: ', '.join(sorted(x)),
                'Feature': 'count'
            }).reset_index()
            df_cat.rename(columns={'Feature': 'N_Features', 'Clean_Feature': 'Feature_List'}, inplace=True)
            
            df_cat = df_cat.sort_values('AUC', ascending=False).head(10).reset_index(drop=True)
            
            # Display names (n>1の場合は注釈を別ファイルに保存)
            display_names = []
            key = f"{depth}mm_{comp}"
            annotations_dict[key] = []
            
            for idx, row in df_cat.iterrows():
                n = int(row['N_Features'])
                category = row['Category']
                source = row['Source']
                feature_list = row['Feature_List']
                
                if n == 1:
                    display_name = feature_list
                else:
                    display_name = f"{category} (n={n})"
                    # 注釈を辞書に保存
                    annotations_dict[key].append(f"{category} {source}: {feature_list}")
                
                display_names.append(display_name)
            
            df_cat['Display_Name'] = display_names
            
            # X-axis range
            ci_lower_min = df_cat['CI_Lower'].min()
            x_min = max(0.5, ci_lower_min - 0.05)
            x_min = np.floor(x_min * 10) / 10
            x_max = 1.0
            
            # Plot bars
            y_positions = np.arange(len(df_cat))
            
            for i, row in df_cat.iterrows():
                category = row['Category']
                source = row['Source']
                auc = row['AUC']
                ci_lower = max(row['CI_Lower'], x_min)
                ci_upper = min(row['CI_Upper'], x_max)
                
                bar_color = CATEGORY_COLORS.get(category, CATEGORY_COLORS['Combined'])
                ax.barh(i, auc - x_min, left=x_min, height=0.6, 
                       color=bar_color, alpha=0.7, edgecolor='black', linewidth=1.0)
                
                xerr = [[auc - ci_lower], [ci_upper - auc]]
                marker_color = WAVELENGTH_COLORS.get(source, '#000000')
                ax.errorbar(auc, i, xerr=xerr, fmt='o', color=marker_color, 
                           markersize=4, capsize=3, capthick=1.0, elinewidth=1.0)
            
            # Y-axis labels
            ax.set_yticks(y_positions)
            ax.set_yticklabels(df_cat['Display_Name'], fontsize=7)
            ax.invert_yaxis()
            
            # X-axis - 15mm Congestion のみ 0.25 刻み、それ以外は 0.1 刻み
            ax.set_xlim(x_min, x_max)
            
            if depth == 15 and comp == 'Congestion':
                # 0.25 刻み（15mm Congestion のみ）
                # x_min を 0.25 の倍数に切り下げ
                x_min_adjusted = np.floor(x_min * 4) / 4
                x_ticks = np.arange(x_min_adjusted, x_max + 0.01, 0.25)
                # 小数点第2位を表示しないフォーマット（0.5, 0.75, 1.0）
                ax.set_xticklabels([f'{x:.2f}'.rstrip('0').rstrip('.') if x != int(x) else f'{int(x)}' 
                                   for x in x_ticks], fontsize=8)
            else:
                # 0.1 刻み（その他全て）
                x_ticks = np.arange(x_min, x_max + 0.01, 0.1)
                # 小数点第1位のみ表示（0.5, 0.6, ..., 1.0）
                ax.set_xticklabels([f'{x:.1f}' for x in x_ticks], fontsize=8)
            
            ax.set_xticks(x_ticks)
            ax.set_xlabel('AUC', fontsize=9)
            
            # Title
            ax.set_title(f'{depth} mm - {comp}', fontsize=10, loc='right')
            ax.grid(axis='x', linestyle='--', alpha=0.3, linewidth=0.5)
            
            # Collect legend handles
            if row_idx == 0 and col_idx == 0:
                for cat, color in CATEGORY_COLORS.items():
                    handle = plt.Line2D([0], [0], marker='s', color='w', 
                                       markerfacecolor=color, markersize=8, label=cat,
                                       markeredgewidth=1.0, markeredgecolor='black')
                    all_handles.append(handle)
                    all_labels.append(cat)
                
                for source, color in WAVELENGTH_COLORS.items():
                    handle = plt.Line2D([0], [0], marker='o', color='w', 
                                       markerfacecolor=color, markersize=6, 
                                       markeredgecolor='black', markeredgewidth=1.0, label=source)
                    all_handles.append(handle)
                    all_labels.append(source)
    
    # Legend
    fig.legend(all_handles, all_labels, loc='center left', bbox_to_anchor=(0.90, 0.5), 
              frameon=True, fontsize=8, title='Legend', title_fontsize=9)
    
    # より広いスペーシングで重なりを防止
    plt.tight_layout(rect=[0, 0, 0.89, 1.0])
    
    # 注釈をテキストファイルに出力
    annotation_file = OUTPUT_DIR / 'feature_annotations.txt'
    with open(annotation_file, 'w', encoding='utf-8') as f:
        f.write("Feature Annotations for Grouped Categories (n>1)\n")
        f.write("=" * 80 + "\n\n")
        for key, annots in annotations_dict.items():
            if annots:
                f.write(f"{key}:\n")
                for annot in annots:
                    f.write(f"  - {annot}\n")
                f.write("\n")
    
    print(f"Feature annotations saved to: {annotation_file}")
    
    # Save in multiple formats
    output_tiff = OUTPUT_DIR / 'depth_comparison_combined_all_bar.tiff'
    plt.savefig(output_tiff, dpi=600, format='tiff', bbox_inches='tight',
                pil_kwargs={'compression': 'tiff_lzw'})
    print(f"Saved: {output_tiff}")
    
    output_pdf = OUTPUT_DIR / 'depth_comparison_combined_all_bar.pdf'
    plt.savefig(output_pdf, dpi=600, format='pdf', bbox_inches='tight')
    print(f"Saved: {output_pdf}")
    
    output_eps = OUTPUT_DIR / 'depth_comparison_combined_all_bar.eps'
    plt.savefig(output_eps, dpi=600, format='eps', bbox_inches='tight')
    print(f"Saved: {output_eps}")
    
    plt.close()


plot_combined_depth_comparison()


print("\n" + "="*60)
print("All tasks completed successfully!")
print(f"Output directory: {OUTPUT_DIR}")
print("Files generated:")
print("  - depth_comparison_combined_all_bar.tiff/pdf/eps")
print("  - feature_annotations.txt (grouped feature details)")
print("="*60)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from matplotlib.ticker import MaxNLocator
import warnings

warnings.filterwarnings('ignore')

# ===========================
# Sci Rep 準拠：視認性強化設定
# ===========================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 8           # 基本サイズを8ptに（規定の最小限）
plt.rcParams['axes.labelsize'] = 8
plt.rcParams['axes.titlesize'] = 8
plt.rcParams['xtick.labelsize'] = 8     # 7pt以下にはしない
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8

plt.rcParams['lines.linewidth'] = 1.2    # 1pt以上を確保
plt.rcParams['axes.linewidth'] = 1.0    # 枠線を太くして視認性向上
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['ytick.major.width'] = 1.0

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# ===========================
# 対象特徴量リスト
# ===========================
TARGET_FEATURES = [
    'Amplitude', 'PI', 'Max_Start_Datum_Diff', 'AUC',
    'S_AUC', 'D_AUC', 'Start_Datum_Area', 'End_Datum_Area',
    'Rise_Time', 'Fall_Time', 'Pulse_Width', 'Diastolic_Width',
    'Rise_Decay_Time_Ratio', 'Upslope', 'Downslope', 'Downslope_Length',
    'Slope_Ratio', 'SDPPG_d_a', 'SDPPG_e_a', 'DNP'
]

OUTPUT_DIR = Path('integrated_depth_comparison')
OUTPUT_DIR.mkdir(exist_ok=True)

# ===========================
# データ計算（関数の定義）
# ===========================
def calculate_state_means(df, features):
    state_map = {'Normal': 1, 'Ischaemia': 2, 'Congestion': 3}
    results = {}
    for feature in features:
        if feature not in df.columns: continue
        means = {}
        for state_name in state_map.keys():
            subset = df[df['State'] == state_name]
            means[state_name] = subset[feature].mean() if len(subset) > 0 else np.nan
        results[feature] = means
    return results

# ダミーデータ読み込み（パスは既存環境に合わせてください）
# df_3mm_red = pd.read_csv('...') 
# 等が実行されている前提です。

# ===========================
# 可視化レイアウト設定
# ===========================
n_features = len(TARGET_FEATURES)
n_features_per_row = 2 
n_cols = 7 
n_rows = int(np.ceil(n_features / n_features_per_row))

# Figureサイズ：ダブルカラム幅(183mm = 7.2inch) 
# 高さを1.3 -> 1.8に拡大してy軸の潰れを解消
fig_width = 7.2
fig_height = n_rows * 1.0 

fig = plt.figure(figsize=(fig_width, fig_height))

# wspaceを0.25 -> 0.45に拡大（y軸ラベルのスペース確保）
gs = gridspec.GridSpec(n_rows, n_cols, figure=fig, 
                       width_ratios=[1, 1, 1, 0.2, 1, 1, 1], 
                       wspace=0.5, hspace=0.4,
                       left=0.08, right=0.97, top=0.97, bottom=0.05)

state_colors = {'Normal': '#90EE90', 'Ischaemia': '#FFB6C6', 'Congestion': '#ADD8E6'}
color_3mm_red = '#D32F2F'
color_3mm_ir = '#1976D2'

# ===========================
# 描画ループ
# ===========================
for feat_idx, feature in enumerate(TARGET_FEATURES):
    row = feat_idx // n_features_per_row
    col_offset = 4 if (feat_idx % n_features_per_row) == 1 else 0
    
    # 各深度のグラフ用リスト
    depth_axes = []
    
    # 3mm, 9mm, 15mm のサブプロット作成
    for i, title in enumerate(["3 mm", "9 mm", "15 mm"]):
        ax = fig.add_subplot(gs[row, col_offset + i])
        ax.set_title(title, fontsize=8, pad=4)
        depth_axes.append(ax)
        
    ax_3mm, ax_9mm, ax_15mm = depth_axes
    all_values = []

    # --- 3mm 描画 ---
    x_pos = np.array([0, 0.4, 0.8])
    bar_w = 0.35 # 少し太くして視認性アップ
    
    # Red & IR for 3mm
    for color, m_data, lbl in zip([color_3mm_red, color_3mm_ir], [means_3mm_red, means_3mm_ir], ['Red', 'IR']):
        if feature in m_data:
            ref = m_data[feature].get('Normal', np.nan)
            if not np.isnan(ref) and abs(ref) > 1e-10:
                vals = [1.0, m_data[feature].get('Ischaemia', 0)/ref, m_data[feature].get('Congestion', 0)/ref]
                ax_3mm.bar(x_pos, vals, color=color, alpha=0.5, edgecolor='black', linewidth=0.8, width=bar_w, label=lbl)
                all_values.extend(vals)

    # --- 9mm / 15mm 描画 ---
    for ax, m_data in zip([ax_9mm, ax_15mm], [means_9mm_ir, means_15mm_ir]):
        if feature in m_data:
            ref = m_data[feature].get('Normal', np.nan)
            if not np.isnan(ref) and abs(ref) > 1e-10:
                vals = [1.0, m_data[feature].get('Ischaemia', 0)/ref, m_data[feature].get('Congestion', 0)/ref]
                colors = [state_colors['Normal'], state_colors['Ischaemia'], state_colors['Congestion']]
                ax.bar(x_pos, vals, color=colors, edgecolor='black', linewidth=0.8, width=bar_w)
                all_values.extend(vals)

    # --- 軸の微調整（全グラフ共通） ---
    for ax in depth_axes:
        ax.axhline(1.0, color='black', linestyle='--', linewidth=0.7, alpha=0.5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(['N', 'I', 'C'], fontsize=8)
        ax.grid(axis='y', alpha=0.2, linewidth=0.5)
        ax.set_xlim(-0.3, 1.1)
        # y軸の目盛り数を制限して重なりを防止
        from matplotlib.ticker import FormatStrFormatter # (ファイルの冒頭に追記)
        ax.tick_params(axis='y', pad=1)
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4, prune='both'))

    # --- 動的Y軸範囲 ---
    if all_values:
        v_min, v_max = min(all_values), max(all_values)
        margin = (v_max - v_min) * 0.15
        ax_3mm.set_ylim(max(0, v_min - margin), v_max + margin)
        # 3mmの範囲を他の深度にも適用（比較のため）
        ax_9mm.set_ylim(ax_3mm.get_ylim())
        ax_15mm.set_ylim(ax_3mm.get_ylim())
        
    for ax in [ax_3mm, ax_9mm, ax_15mm]:
        pos = ax.get_position()
        ax.set_position([pos.x0, pos.y0, pos.width, pos.height * 0.8])

    # --- 特徴量ラベル（中央のグラフの下） ---
    ax_9mm.text(0.5, -0.30, feature, transform=ax_9mm.transAxes,
               fontsize=8, ha='center', va='top', weight='bold')

    if row == 0 and col_offset == 0:
        ax_3mm.legend(loc='upper right', frameon=True, borderpad=0.3, handletextpad=0.5)

# 保存設定
save_params = {'dpi': 600, 'bbox_inches': 'tight'}
plt.savefig(OUTPUT_DIR / 'feature_comparison_final.tiff', format='tiff', pil_kwargs={'compression': 'tiff_lzw'}, **save_params)
plt.savefig(OUTPUT_DIR / 'feature_comparison_final.pdf', format='pdf', **save_params)

plt.show()